In [26]:
# filepath: ./tennis_upset_two_head_pipeline.py
"""
Two-head tennis match predictor focused on upsets.

What you get:
- Pre-match Elo (overall + surface) built from raw matches (no leakage).
- Fatigue/schedule features (rest days, matches in last 7/14/28 days).
- Accuracy-head (Random Forest) for stable overall accuracy.
- Upset-head (Random Forest) trained to spot underdog edges in strong-favorite spots.
- GroupKFold CV that keeps mirror rows (symmetric data) together.
- Threshold table for underdog picks (count vs win-rate), to guide staking policy.

Why specific choices:
- Elo pre-match: reliable baseline signal that generalizes, crucial for defining “favorite/underdog”.
- Upset-head excludes Elo diff by design to learn residual/situational patterns rather than re-learn Elo.
- GroupKFold on a match identity prevents optimistic leakage from mirrored rows.

Add-on (easy): if you have closing odds, inject them and compute EV with calibration; this turns the threshold table into a profit table.
"""

from __future__ import annotations

import math
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


# -----------------------------
# Data loading
# -----------------------------
PATH_MATCHES = "atp_matches_2021_2024.csv"
PATH_SYM     = "tennis_model_data.csv"


# -----------------------------
# Elo construction (pre-match)
# -----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_prematch_elo(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    """
    Return oriented pre-match Elo for each (player_1, player_2, date, surface) pair.
    Why: Using pre-match Elo avoids leaking the match result.
    """
    m = matches.copy()
    m["tourney_date"] = pd.to_datetime(m["tourney_date"].astype(str), format="%Y%m%d", errors="coerce")
    m["surface"] = m["surface"].fillna("Unknown")
    m = m.sort_values(["tourney_date", "tourney_name", "match_num"], kind="mergesort")

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str, str], float] = {}
    rows: List[Dict] = []

    BASE = params.base
    K = params.k_overall
    Ks = params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]
        s = r["surface"]
        w = r["winner_name"]
        l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        # Oriented records for both orders so we can merge cleanly later
        rows.append({"tourney_date": d, "surface": s, "player_1": w, "player_2": l,
                     "elo1": ew_w, "elo2": ew_l, "selo1": es_w, "selo2": es_l})
        rows.append({"tourney_date": d, "surface": s, "player_1": l, "player_2": w,
                     "elo1": ew_l, "elo2": ew_w, "selo1": es_l, "selo2": es_w})

        # Update after outcome
        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        elo_overall[w] = ew_w + K * (1 - exp_w)
        elo_overall[l] = ew_l - K * (1 - exp_w)

        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))
        elo_surface[(w, s)] = es_w + Ks * (1 - exp_w_s)
        elo_surface[(l, s)] = es_l - Ks * (1 - exp_w_s)

    return pd.DataFrame(rows)


# -----------------------------
# Fatigue / schedule features
# -----------------------------
def build_fatigue_features(matches: pd.DataFrame) -> pd.DataFrame:
    """
    For each (player, date, opponent), compute rest_days and matches played in last 7/14/28 days.
    Why: Upsets often arise from fatigue and scheduling constraints rather than skill alone.
    """
    m = matches.copy()
    m["tourney_date"] = pd.to_datetime(m["tourney_date"].astype(str), format="%Y%m%d", errors="coerce")

    rows = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; w = r["winner_name"]; l = r["loser_name"]
        rows.append({"player": w, "opp": l, "date": d})
        rows.append({"player": l, "opp": w, "date": d})

    long = pd.DataFrame(rows).sort_values(["player", "date"], kind="mergesort").reset_index(drop=True)

    # rest_days (days since prior appearance)
    long["prev_date"] = long.groupby("player")["date"].shift(1)
    long["rest_days"] = (long["date"] - long["prev_date"]).dt.days.fillna(30).clip(0, 60)

    # rolling matches BEFORE the current match
    def rolling_counts(dates: List[pd.Timestamp], window: int) -> List[int]:
        dq = deque()
        out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window:
                dq.popleft()
            out.append(len(dq))
            dq.append(dt)
        return out

    for w in (7, 14, 28):
        long[f"matches_{w}d"] = long.groupby("player")["date"].transform(lambda s: rolling_counts(list(s), w))

    return long[["player", "opp", "date", "rest_days", "matches_7d", "matches_14d", "matches_28d"]].copy()


# -----------------------------
# Modeling helpers
# -----------------------------
def make_group_key(df: pd.DataFrame) -> pd.Series:
    """
    Group key keeps both symmetric rows for the same real-world match in the same fold.
    """
    # Using (sorted players | date | round) is robust for symmetric rows
    return (
        (df["player_1"] + " vs " + df["player_2"]).apply(lambda s: "||".join(sorted(s.split(" vs "))))
        + "|" + df["tourney_date"].astype(str)
        + "|" + df["round_encoded"].astype(str)
    )

def evaluate_upset_thresholds(underdog_prob: np.ndarray, true_upset: np.ndarray, thresholds: np.ndarray) -> pd.DataFrame:
    rows = []
    for t in thresholds:
        sel = underdog_prob >= t
        n = int(sel.sum())
        win_rate = float((true_upset[sel] == 1).mean()) if n > 0 else float("nan")
        rows.append({"threshold": round(float(t), 3), "n_bets": n, "underdog_win_rate": round(win_rate, 4) if n else None})
    return pd.DataFrame(rows)


# -----------------------------
# Main pipeline
# -----------------------------
def main():
    # Load data
    sym = pd.read_csv(PATH_SYM)
    sym["tourney_date"] = pd.to_datetime(sym["tourney_date"])
    sym["surface"] = sym["surface"].fillna("Unknown")
    sym["outcome"] = sym["outcome"].astype(int)

    matches = pd.read_csv(PATH_MATCHES)

    # Build pre-match Elo
    elo_map = build_prematch_elo(matches, EloParams())

    # Merge Elo into symmetric data (oriented join)
    data = sym.merge(elo_map, on=["tourney_date", "surface", "player_1", "player_2"], how="left")
    assert data[["elo1", "elo2", "selo1", "selo2"]].isna().mean().max() < 0.01, "Elo merge seems off."

    # Fatigue features (merge on exact match (player, opp, date))
    fatigue = build_fatigue_features(matches)
    data = data.merge(
        fatigue.rename(columns={"player": "player_1", "opp": "player_2", "date": "tourney_date"}),
        on=["player_1", "player_2", "tourney_date"], how="left",
    ).merge(
        fatigue.rename(columns={"player": "player_2", "opp": "player_1", "date": "tourney_date"}),
        on=["player_1", "player_2", "tourney_date"], how="left", suffixes=("_p1", "_p2"),
    )

    # Core diffs
    data["elo_diff"]  = data["elo1"] - data["elo2"]
    data["selo_diff"] = data["selo1"] - data["selo2"]
    data["rank_diff"] = (data["p1_rank"] - data["p2_rank"]).fillna(0)
    data["age_diff"]  = (data["p1_age"] - data["p2_age"]).fillna(0.0)
    data["ht_diff"]   = (data["p1_ht"] - data["p2_ht"]).fillna(0.0)
    data["seed_diff"] = (data["p1_seed"] - data["p2_seed"]).fillna(0.0)

    data["rest_diff"] = data["rest_days_p1"] - data["rest_days_p2"]
    data["ml7_diff"]  = data["matches_7d_p1"] - data["matches_7d_p2"]
    data["ml14_diff"] = data["matches_14d_p1"] - data["matches_14d_p2"]
    data["ml28_diff"] = data["matches_28d_p1"] - data["matches_28d_p2"]

    # Targets and groups
    y = data["outcome"].values
    groups = make_group_key(data)

    # -------------------------
    # Accuracy head (Random Forest)
    # -------------------------
    acc_num = ["elo_diff","selo_diff","rank_diff","age_diff","ht_diff","seed_diff",
               "rest_diff","ml7_diff","ml14_diff","ml28_diff","round_encoded"]
    acc_cat = ["surface","tourney_level","best_of"]

    acc_prep = ColumnTransformer([
        ("num", "passthrough", acc_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), acc_cat),
    ])

    acc_model = RandomForestClassifier(
        n_estimators=300, min_samples_split=10, min_samples_leaf=5, random_state=42, n_jobs=-1
    )

    acc_pipe = Pipeline([("prep", acc_prep), ("clf", acc_model)])

    gkf = GroupKFold(n_splits=3)
    acc_proba = cross_val_predict(
        acc_pipe, data[acc_num + acc_cat], y,
        cv=gkf.split(data, y, groups=groups),
        method="predict_proba"
    )[:, 1]

    # Basic metrics
    acc_pred = (acc_proba >= 0.5).astype(int)
    acc_acc   = accuracy_score(y, acc_pred)
    acc_auc   = roc_auc_score(y, acc_proba)
    acc_brier = brier_score_loss(y, acc_proba)

    # -------------------------
    # Upset head (learn residual/situational edges)
    # - Label = 1 if lower Elo wins
    # - Train without Elo features to avoid trivially re-learning Elo
    # -------------------------
    p1_is_lower_elo = (data["elo1"] < data["elo2"]).astype(int).values
    upset_label = np.where(p1_is_lower_elo == 1, y == 1, y == 0).astype(int)

    upset_num = ["rank_diff","age_diff","ht_diff","seed_diff","rest_diff","ml7_diff","ml14_diff","ml28_diff","round_encoded"]
    upset_cat = ["surface","tourney_level","best_of"]

    upset_prep = ColumnTransformer([
        ("num", "passthrough", upset_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), upset_cat),
    ])

    upset_model = RandomForestClassifier(
        n_estimators=200, min_samples_split=10, min_samples_leaf=5, random_state=43, n_jobs=-1
    )
    upset_pipe = Pipeline([("prep", upset_prep), ("clf", upset_model)])

    # Focus training on strong-favorite situations to teach the head where upsets live
    strong_fav_mask = (data["elo_diff"].abs() >= 75).values  # tuneable margin
    sample_weight = np.where(strong_fav_mask & (upset_label == 1), 3.0, 1.0)

    upset_proba = cross_val_predict(
        upset_pipe, data[upset_num + upset_cat], upset_label,
        cv=gkf.split(data, upset_label, groups=groups),
        method="predict_proba",
        fit_params={"clf__sample_weight": sample_weight}
    )[:, 1]

    # -------------------------
    # Fuse heads to get a better underdog probability
    # Base underdog prob from accuracy head, then nudge in strong-favorite zones with upset head
    # -------------------------
    # Underdog is p1 if p1 has lower Elo; else p2
    # From the accuracy-head p1-win prob, derive underdog prob:
    base_underdog_prob = np.where(data["elo_diff"].values > 0, 1.0 - acc_proba, acc_proba)

    # Blend only when there's a strong Elo favorite (avoid hurting base accuracy elsewhere)
    blend_alpha = 0.35  # how much upset head can move the base prob in strong-favorite regions
    fused_underdog_prob = base_underdog_prob.copy()
    fused_underdog_prob[strong_fav_mask] = (
        (1 - blend_alpha) * base_underdog_prob[strong_fav_mask] + blend_alpha * upset_proba[strong_fav_mask]
    )

    # True underdog-upset label we care about:
    true_upset = upset_label.astype(int)

    # Threshold sweep: where should we bet the underdog?
    thresholds = np.linspace(0.50, 0.80, 7)
    table_base = evaluate_upset_thresholds(base_underdog_prob, true_upset, thresholds)
    table_fused = evaluate_upset_thresholds(fused_underdog_prob, true_upset, thresholds)

    # Report
    summary = pd.DataFrame({
        "metric": ["accuracy", "auc", "brier"],
        "accuracy_head": [acc_acc, acc_auc, acc_brier],
    })

    print("=== Accuracy-head metrics (CV) ===")
    print(summary.to_string(index=False))

    print("\n=== Underdog selection (Accuracy-head only) ===")
    print(table_base.to_string(index=False))

    print("\n=== Underdog selection (FUSED: accuracy-head + upset-head) ===")
    print(table_fused.to_string(index=False))

    # Hints if you add market odds later:
    # - Calibrate acc_proba via CalibratedClassifierCV (isotonic), then for each match compute EV:
    #     EV_underdog = p_underdog * odds_underdog - (1 - p_underdog)
    #   Select bets where EV_underdog > 0 and p_underdog exceeds your calibrated threshold.
    # - Keep conformal risk control for selective betting to avoid over-confident false signals.


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future 

=== Accuracy-head metrics (CV) ===
  metric  accuracy_head
accuracy       0.832019
     auc       0.922765
   brier       0.113398

=== Underdog selection (Accuracy-head only) ===
 threshold  n_bets  underdog_win_rate
      0.50    7740             0.8056
      0.55    7008             0.8388
      0.60    6430             0.8667
      0.65    5989             0.8895
      0.70    5544             0.9100
      0.75    4973             0.9282
      0.80    4199             0.9443

=== Underdog selection (FUSED: accuracy-head + upset-head) ===
 threshold  n_bets  underdog_win_rate
      0.50    7759             0.8053
      0.55    6982             0.8410
      0.60    6401             0.8678
      0.65    5926             0.8893
      0.70    5372             0.9103
      0.75    4564             0.9268
      0.80    3561             0.9441


In [27]:
# filepath: ./player_profiles_dashboard.py
"""
Player-centric tennis analytics & visuals (ATP 2021-2024).

Run:
    python player_profiles_dashboard.py

Outputs (./reports):
- player_profiles.csv            : one row per player with key metrics
- leaderboards_*.png             : global visuals
- players/<PLAYER>/...png        : per-player visual report
- logs/player_match_log.parquet  : per-player match log (pre-match Elo, fatigue, form)

Notes:
- Uses only matplotlib (no seaborn). One chart per figure. No explicit colors.
- Safely handles missing serve/return columns.
"""

from __future__ import annotations

import os
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ----------------------------
# Config
# ----------------------------
PATH_MATCHES = "atp_matches_2021_2024.csv"
PATH_SYM = "tennis_model_data.csv"  # optional, not required but loaded to align surface/levels if needed

OUT_DIR = "./reports"
N_TOP_PLAYERS_FOR_REPORT = 12  # generates detailed reports for the most active players


# ----------------------------
# Elo construction (no leakage)
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def _safe_int_date(x) -> pd.Timestamp:
    """Ensures integer yyyymmdd -> Timestamp; guards strings/ints."""
    if pd.isna(x):
        return pd.NaT
    s = str(int(x))
    try:
        return pd.to_datetime(s, format="%Y%m%d")
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def build_player_match_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    """
    Create one row per player per match containing pre-match Elo, opponent Elo,
    fatigue, rolling form, and optional serve/return stats.
    """
    m = matches.copy()

    # Required columns; the script is defensive if some stats are missing.
    required = [
        "tourney_date", "tourney_name", "surface", "tourney_level", "match_num",
        "winner_name", "loser_name", "round", "best_of", "score"
    ]
    for c in required:
        if c not in m.columns:
            raise ValueError(f"Missing required column '{c}' in matches CSV.")

    m["tourney_date"] = m["tourney_date"].apply(_safe_int_date)
    m["surface"] = m["surface"].fillna("Unknown")
    m = m.sort_values(["tourney_date", "tourney_name", "match_num"], kind="mergesort").reset_index(drop=True)

    # Optional stat columns
    w_cols = [c for c in m.columns if c.startswith("w_")]
    l_cols = [c for c in m.columns if c.startswith("l_")]
    has_stats = bool(w_cols and l_cols)

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str, str], float] = {}

    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]
        s = r["surface"]
        w = r["winner_name"]
        l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        # Winner row (player-centric)
        row_w = {
            "player": w, "opp": l, "is_win": 1,
            "date": d, "surface": s, "tourney_name": r["tourney_name"],
            "tourney_level": r.get("tourney_level", np.nan),
            "round": r.get("round", np.nan),
            "best_of": r.get("best_of", np.nan),
            "score": r.get("score", np.nan),
            "pre_elo": ew_w, "pre_selo": es_w,
            "opp_pre_elo": ew_l, "opp_pre_selo": es_l,
        }
        # Loser row
        row_l = {
            "player": l, "opp": w, "is_win": 0,
            "date": d, "surface": s, "tourney_name": r["tourney_name"],
            "tourney_level": r.get("tourney_level", np.nan),
            "round": r.get("round", np.nan),
            "best_of": r.get("best_of", np.nan),
            "score": r.get("score", np.nan),
            "pre_elo": ew_l, "pre_selo": es_l,
            "opp_pre_elo": ew_w, "opp_pre_selo": es_w,
        }

        if has_stats:
            # Why: serve/return diagnostics are informative if present; guarded merge.
            for wc in w_cols:
                row_w[wc.replace("w_", "sv_")] = r[wc]
            for lc in l_cols:
                row_l[lc.replace("l_", "sv_")] = r[lc]

        rows.append(row_w)
        rows.append(row_l)

        # Elo updates after outcome
        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        elo_overall[w] = ew_w + K * (1 - exp_w)
        elo_overall[l] = ew_l - K * (1 - exp_w)

        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))
        elo_surface[(w, s)] = es_w + KS * (1 - exp_w_s)
        elo_surface[(l, s)] = es_l - KS * (1 - exp_w_s)

    log = pd.DataFrame(rows).sort_values(["player", "date"], kind="mergesort").reset_index(drop=True)

    # Rest days
    log["prev_date"] = log.groupby("player")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days.clip(lower=0)
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    # Rolling workload counts (before current match)
    def _rolling_counts(dates: List[pd.Timestamp], window: int) -> List[int]:
        dq = deque()
        out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window:
                dq.popleft()
            out.append(len(dq))
            dq.append(dt)
        return out

    for w in (7, 14, 28):
        log[f"matches_{w}d"] = log.groupby("player")["date"].transform(lambda s: _rolling_counts(list(s), w))

    # Rolling form
    log["rolling_10_winrate"] = (
        log.groupby("player")["is_win"]
           .transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean())
           .fillna(method="bfill")
           .clip(0, 1)
    )
    log["rolling_5_winrate"] = (
        log.groupby("player")["is_win"]
           .transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean())
           .fillna(method="bfill")
           .clip(0, 1)
    )

    # Streak length (+ for win streak, - for loss streak)
    def _streak(series: pd.Series) -> pd.Series:
        streak = []
        cur = 0
        for v in series:
            cur = cur + 1 if v == 1 else (cur - 1 if cur > 0 else -1 if v == 0 else 0)
            if v == 1 and cur < 0:
                cur = 1
            if v == 0 and cur > 0:
                cur = -1
            streak.append(cur)
        return pd.Series(streak, index=series.index)

    log["streak"] = log.groupby("player")["is_win"].transform(_streak)

    return log


# ----------------------------
# Profiles & aggregates
# ----------------------------
def compute_player_profiles(log: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate per-player snapshots: overall/surface records, current/peak Elo, fatigue & form.
    """
    last_rows = log.sort_values("date").groupby("player").tail(1)
    peak_elo = log.groupby("player")["pre_elo"].max().rename("peak_elo")
    cur_elo = last_rows.set_index("player")["pre_elo"].rename("current_elo")

    # Outcomes
    agg = log.groupby("player").agg(
        matches=("is_win", "size"),
        wins=("is_win", "sum"),
        losses=("is_win", lambda s: (1 - s).sum()),
        avg_rest_days=("rest_days", "mean"),
        matches_28d_latest=("matches_28d", "last"),
        rolling_10_winrate_latest=("rolling_10_winrate", "last"),
        rolling_5_winrate_latest=("rolling_5_winrate", "last"),
        streak_latest=("streak", "last"),
    ).reset_index()

    agg["win_rate"] = (agg["wins"] / agg["matches"]).round(4)

    # Surface splits
    surf = (
        log.groupby(["player", "surface"])["is_win"]
           .agg(matches="size", wins="sum")
           .reset_index()
    )
    surf["losses"] = surf["matches"] - surf["wins"]
    surf["win_rate"] = (surf["wins"] / surf["matches"]).round(4)
    surf_pivot = surf.pivot(index="player", columns="surface", values="win_rate").add_prefix("wr_").fillna(np.nan)

    prof = (
        agg.merge(peak_elo, on="player")
           .merge(cur_elo, on="player")
           .merge(surf_pivot, on="player", how="left")
           .sort_values(["current_elo", "matches"], ascending=[False, False])
           .reset_index(drop=True)
    )
    return prof


# ----------------------------
# Head-to-Head helpers
# ----------------------------
def compute_h2h(log: pd.DataFrame, player: str) -> pd.DataFrame:
    """Return H2H tallies vs all opponents for a specific player."""
    sub = log[log["player"] == player]
    h2h = sub.groupby("opp")["is_win"].agg(matches="size", wins="sum").reset_index()
    h2h["losses"] = h2h["matches"] - h2h["wins"]
    h2h["win_rate"] = (h2h["wins"] / h2h["matches"]).round(3)
    return h2h.sort_values(["matches", "win_rate"], ascending=[False, False])


# ----------------------------
# Plotting (matplotlib only)
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def plot_leaderboards(prof: pd.DataFrame, out_dir: str) -> None:
    """Global visuals."""
    _ensure_dir(out_dir)

    # Elo leaderboard
    top = prof.head(25)

    plt.figure()
    plt.barh(top["player"][::-1], top["current_elo"][::-1])
    plt.title("Top 25 by Current Elo")
    plt.xlabel("Elo")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "leaderboard_current_elo.png"))
    plt.close()

    # Elo distribution
    plt.figure()
    plt.hist(prof["current_elo"].dropna(), bins=30)
    plt.title("Distribution of Current Elo")
    plt.xlabel("Elo")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "elo_distribution.png"))
    plt.close()

    # Elo vs win rate
    valid = prof.dropna(subset=["current_elo", "win_rate", "matches"])
    plt.figure()
    sizes = (valid["matches"].clip(1, 150) ** 0.5) * 5.0
    plt.scatter(valid["current_elo"], valid["win_rate"], s=sizes)
    plt.title("Current Elo vs Career Win Rate (size ~ matches)")
    plt.xlabel("Current Elo")
    plt.ylabel("Win Rate")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "elo_vs_win_rate.png"))
    plt.close()

def plot_player_report(log: pd.DataFrame, prof: pd.DataFrame, player: str, out_dir: str) -> None:
    """Per-player visuals."""
    sub = log[log["player"] == player].copy()
    if sub.empty:
        return
    pdir = os.path.join(out_dir, "players", player.replace(" ", "_"))
    _ensure_dir(pdir)

    # Elo timeline
    plt.figure()
    plt.plot(sub["date"], sub["pre_elo"])
    plt.title(f"{player} — Pre-match Elo Over Time")
    plt.xlabel("Date")
    plt.ylabel("Elo")
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "elo_timeline.png"))
    plt.close()

    # Rolling 10-match win rate
    plt.figure()
    plt.plot(sub["date"], sub["rolling_10_winrate"])
    plt.title(f"{player} — Rolling 10-match Win Rate")
    plt.xlabel("Date")
    plt.ylabel("Win Rate")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "rolling10_winrate.png"))
    plt.close()

    # Surface win rates (bars)
    surf = (
        sub.groupby("surface")["is_win"]
           .agg(matches="size", wins="sum")
           .reset_index()
           .sort_values("matches", ascending=False)
    )
    if not surf.empty:
        surf["wr"] = surf["wins"] / surf["matches"]
        plt.figure()
        plt.bar(surf["surface"], surf["wr"])
        plt.title(f"{player} — Surface Win Rates")
        plt.xlabel("Surface")
        plt.ylabel("Win Rate")
        plt.ylim(0, 1)
        plt.tight_layout()
        plt.savefig(os.path.join(pdir, "surface_win_rates.png"))
        plt.close()

    # Rest-days histogram
    plt.figure()
    plt.hist(sub["rest_days"].clip(0, 60), bins=20)
    plt.title(f"{player} — Rest Days Distribution")
    plt.xlabel("Rest Days")
    plt.ylabel("Matches")
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "rest_days_hist.png"))
    plt.close()

    # Opponent quality scatter (opp Elo vs date)
    plt.figure()
    plt.scatter(sub["date"], sub["opp_pre_elo"], s=20)
    plt.title(f"{player} — Opponent Elo Over Time")
    plt.xlabel("Date")
    plt.ylabel("Opponent Pre-match Elo")
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "opponent_elo_over_time.png"))
    plt.close()

    # Top H2H opponents (by matches)
    h2h = compute_h2h(log, player).head(15)
    if not h2h.empty:
        plt.figure()
        plt.barh(h2h["opp"][::-1], h2h["win_rate"][::-1])
        plt.title(f"{player} — Top H2H Win Rates")
        plt.xlabel("Win Rate")
        plt.tight_layout()
        plt.savefig(os.path.join(pdir, "top_h2h.png"))
        plt.close()


# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_DIR)
    _ensure_dir(os.path.join(OUT_DIR, "logs"))

    matches = pd.read_csv(PATH_MATCHES)
    # Load symmetric dataset if you want to align metadata like encoded rounds; not required for visuals
    try:
        _ = pd.read_csv(PATH_SYM)
    except Exception:
        pass

    log = build_player_match_log(matches, EloParams())
    log.to_parquet(os.path.join(OUT_DIR, "logs", "player_match_log.parquet"), index=False)

    prof = compute_player_profiles(log)
    prof.to_csv(os.path.join(OUT_DIR, "player_profiles.csv"), index=False)

    # Global visuals
    plot_leaderboards(prof, OUT_DIR)

    # Pick players for reports: most active
    top_players = (
        prof.sort_values(["matches", "current_elo"], ascending=[False, False])
            .head(N_TOP_PLAYERS_FOR_REPORT)["player"].tolist()
    )
    for p in top_players:
        plot_player_report(log, prof, p, OUT_DIR)

    print(f"Saved player profiles: {os.path.join(OUT_DIR, 'player_profiles.csv')}")
    print(f"Saved per-player match log: {os.path.join(OUT_DIR, 'logs', 'player_match_log.parquet')}")
    print(f"Saved leaderboards and player reports under: {OUT_DIR}")

if __name__ == "__main__":
    main()


C:\Users\SAM\AppData\Local\Temp\ipykernel_39632\4169073853.py:167: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method="bfill")
C:\Users\SAM\AppData\Local\Temp\ipykernel_39632\4169073853.py:173: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method="bfill")


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [29]:
# filepath: ./player_profiles_dashboard.py
"""
Player-centric tennis analytics & visuals (ATP 2021-2024).

Run:
    python player_profiles_dashboard.py

Outputs (./reports):
- player_profiles.csv or .parquet (auto-fallback)
- leaderboards_*.png
- players/<PLAYER>/*.png
- logs/player_match_log.csv or .parquet (auto-fallback)

Notes:
- Falls back to CSV when pyarrow/fastparquet are missing.
- Matplotlib only; one chart per figure; no explicit colors.
"""

from __future__ import annotations

import os
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ----------------------------
# Config
# ----------------------------
PATH_MATCHES = "atp_matches_2021_2024.csv"
PATH_SYM = "tennis_model_data.csv"  # optional, not required

OUT_DIR = "./reports"
N_TOP_PLAYERS_FOR_REPORT = 12


# ----------------------------
# IO helpers (Parquet -> CSV fallback)
# ----------------------------
def safe_write_table(df: pd.DataFrame, parquet_path: str, index: bool = False) -> str:
    """
    Try Parquet; if engine missing, save CSV with same basename.
    Why: Avoid optional-dependency errors on machines without pyarrow/fastparquet.
    """
    base, _ = os.path.splitext(parquet_path)
    try:
        df.to_parquet(parquet_path, index=index)
        print(f"Saved: {parquet_path}")
        return parquet_path
    except ImportError:
        csv_path = base + ".csv"
        df.to_csv(csv_path, index=index)
        print(f"Parquet engine not available; saved CSV instead: {csv_path}")
        return csv_path


# ----------------------------
# Utilities
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def _safe_int_date(x) -> pd.Timestamp:
    """Guard yyyymmdd -> Timestamp; tolerates int/str."""
    if pd.isna(x):
        return pd.NaT
    s = str(int(x))
    try:
        return pd.to_datetime(s, format="%Y%m%d")
    except Exception:
        return pd.to_datetime(s, errors="coerce")


# ----------------------------
# Elo construction (no leakage)
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_match_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    """
    One row per player per match with pre-match Elo (overall/surface), opponent Elo,
    fatigue, rolling form, and optional serve/return stats.
    """
    m = matches.copy()

    required = [
        "tourney_date", "tourney_name", "surface", "tourney_level", "match_num",
        "winner_name", "loser_name", "round", "best_of", "score"
    ]
    for c in required:
        if c not in m.columns:
            raise ValueError(f"Missing required column '{c}' in matches CSV.")

    m["tourney_date"] = m["tourney_date"].apply(_safe_int_date)
    m["surface"] = m["surface"].fillna("Unknown")
    m = m.sort_values(["tourney_date", "tourney_name", "match_num"], kind="mergesort").reset_index(drop=True)

    w_cols = [c for c in m.columns if c.startswith("w_")]
    l_cols = [c for c in m.columns if c.startswith("l_")]
    has_stats = bool(w_cols and l_cols)

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str, str], float] = {}

    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]
        s = r["surface"]
        w = r["winner_name"]
        l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        row_w = {
            "player": w, "opp": l, "is_win": 1,
            "date": d, "surface": s, "tourney_name": r["tourney_name"],
            "tourney_level": r.get("tourney_level", np.nan),
            "round": r.get("round", np.nan),
            "best_of": r.get("best_of", np.nan),
            "score": r.get("score", np.nan),
            "pre_elo": ew_w, "pre_selo": es_w,
            "opp_pre_elo": ew_l, "opp_pre_selo": es_l,
        }
        row_l = {
            "player": l, "opp": w, "is_win": 0,
            "date": d, "surface": s, "tourney_name": r["tourney_name"],
            "tourney_level": r.get("tourney_level", np.nan),
            "round": r.get("round", np.nan),
            "best_of": r.get("best_of", np.nan),
            "score": r.get("score", np.nan),
            "pre_elo": ew_l, "pre_selo": es_l,
            "opp_pre_elo": ew_w, "opp_pre_selo": es_w,
        }

        if has_stats:
            # why: include serve/return diagnostics when present
            for wc in w_cols:
                row_w[wc.replace("w_", "sv_")] = r[wc]
            for lc in l_cols:
                row_l[lc.replace("l_", "sv_")] = r[lc]

        rows.append(row_w)
        rows.append(row_l)

        # Elo updates after the match (prevent leakage)
        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        elo_overall[w] = ew_w + K * (1 - exp_w)
        elo_overall[l] = ew_l - K * (1 - exp_w)

        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))
        elo_surface[(w, s)] = es_w + KS * (1 - exp_w_s)
        elo_surface[(l, s)] = es_l - KS * (1 - exp_w_s)

    log = pd.DataFrame(rows).sort_values(["player", "date"], kind="mergesort").reset_index(drop=True)

    # Rest days
    log["prev_date"] = log.groupby("player")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days.clip(lower=0)
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    # Rolling workload counts (before current match)
    def _rolling_counts(dates: List[pd.Timestamp], window: int) -> List[int]:
        dq = deque()
        out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window:
                dq.popleft()
            out.append(len(dq))
            dq.append(dt)
        return out

    for w in (7, 14, 28):
        log[f"matches_{w}d"] = log.groupby("player")["date"].transform(lambda s: _rolling_counts(list(s), w))

    # Rolling form
    log["rolling_10_winrate"] = (
        log.groupby("player")["is_win"]
           .transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean())
           .fillna(method="bfill")
           .clip(0, 1)
    )
    log["rolling_5_winrate"] = (
        log.groupby("player")["is_win"]
           .transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean())
           .fillna(method="bfill")
           .clip(0, 1)
    )

    # Streak length (+ for win streak, - for loss streak)
    def _streak(series: pd.Series) -> pd.Series:
        streak = []
        cur = 0
        for v in series:
            cur = cur + 1 if v == 1 else (cur - 1 if cur > 0 else -1 if v == 0 else 0)
            if v == 1 and cur < 0:
                cur = 1
            if v == 0 and cur > 0:
                cur = -1
            streak.append(cur)
        return pd.Series(streak, index=series.index)

    log["streak"] = log.groupby("player")["is_win"].transform(_streak)

    return log


# ----------------------------
# Profiles & aggregates
# ----------------------------
def compute_player_profiles(log: pd.DataFrame) -> pd.DataFrame:
    """Aggregate per-player snapshots (overall/surface records, Elo, fatigue, form)."""
    last_rows = log.sort_values("date").groupby("player").tail(1)
    peak_elo = log.groupby("player")["pre_elo"].max().rename("peak_elo")
    cur_elo = last_rows.set_index("player")["pre_elo"].rename("current_elo")

    agg = log.groupby("player").agg(
        matches=("is_win", "size"),
        wins=("is_win", "sum"),
        losses=("is_win", lambda s: (1 - s).sum()),
        avg_rest_days=("rest_days", "mean"),
        matches_28d_latest=("matches_28d", "last"),
        rolling_10_winrate_latest=("rolling_10_winrate", "last"),
        rolling_5_winrate_latest=("rolling_5_winrate", "last"),
        streak_latest=("streak", "last"),
    ).reset_index()
    agg["win_rate"] = (agg["wins"] / agg["matches"]).round(4)

    surf = (
        log.groupby(["player", "surface"])["is_win"]
           .agg(matches="size", wins="sum")
           .reset_index()
    )
    surf["losses"] = surf["matches"] - surf["wins"]
    surf["win_rate"] = (surf["wins"] / surf["matches"]).round(4)
    surf_pivot = surf.pivot(index="player", columns="surface", values="win_rate").add_prefix("wr_").fillna(np.nan)

    prof = (
        agg.merge(peak_elo, on="player")
           .merge(cur_elo, on="player")
           .merge(surf_pivot, on="player", how="left")
           .sort_values(["current_elo", "matches"], ascending=[False, False])
           .reset_index(drop=True)
    )
    return prof


# ----------------------------
# Head-to-Head helpers
# ----------------------------
def compute_h2h(log: pd.DataFrame, player: str) -> pd.DataFrame:
    sub = log[log["player"] == player]
    h2h = sub.groupby("opp")["is_win"].agg(matches="size", wins="sum").reset_index()
    h2h["losses"] = h2h["matches"] - h2h["wins"]
    h2h["win_rate"] = (h2h["wins"] / h2h["matches"]).round(3)
    return h2h.sort_values(["matches", "win_rate"], ascending=[False, False])


# ----------------------------
# Plotting (matplotlib only)
# ----------------------------
def plot_leaderboards(prof: pd.DataFrame, out_dir: str) -> None:
    """Global visuals."""
    _ensure_dir(out_dir)

    top = prof.head(25)

    plt.figure()
    plt.barh(top["player"][::-1], top["current_elo"][::-1])
    plt.title("Top 25 by Current Elo")
    plt.xlabel("Elo")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "leaderboard_current_elo.png"))
    plt.close()

    plt.figure()
    plt.hist(prof["current_elo"].dropna(), bins=30)
    plt.title("Distribution of Current Elo")
    plt.xlabel("Elo")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "elo_distribution.png"))
    plt.close()

    valid = prof.dropna(subset=["current_elo", "win_rate", "matches"])
    plt.figure()
    sizes = (valid["matches"].clip(1, 150) ** 0.5) * 5.0
    plt.scatter(valid["current_elo"], valid["win_rate"], s=sizes)
    plt.title("Current Elo vs Career Win Rate (size ~ matches)")
    plt.xlabel("Current Elo")
    plt.ylabel("Win Rate")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "elo_vs_win_rate.png"))
    plt.close()

def plot_player_report(log: pd.DataFrame, prof: pd.DataFrame, player: str, out_dir: str) -> None:
    """Per-player visuals."""
    sub = log[log["player"] == player].copy()
    if sub.empty:
        return
    pdir = os.path.join(out_dir, "players", player.replace(" ", "_"))
    _ensure_dir(pdir)

    # Elo timeline
    plt.figure()
    plt.plot(sub["date"], sub["pre_elo"])
    plt.title(f"{player} — Pre-match Elo Over Time")
    plt.xlabel("Date")
    plt.ylabel("Elo")
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "elo_timeline.png"))
    plt.close()

    # Rolling 10-match win rate
    plt.figure()
    plt.plot(sub["date"], sub["rolling_10_winrate"])
    plt.title(f"{player} — Rolling 10-match Win Rate")
    plt.xlabel("Date")
    plt.ylabel("Win Rate")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "rolling10_winrate.png"))
    plt.close()

    # Surface win rates (bars)
    surf = (
        sub.groupby("surface")["is_win"]
           .agg(matches="size", wins="sum")
           .reset_index()
           .sort_values("matches", ascending=False)
    )
    if not surf.empty:
        surf["wr"] = surf["wins"] / surf["matches"]
        plt.figure()
        plt.bar(surf["surface"], surf["wr"])
        plt.title(f"{player} — Surface Win Rates")
        plt.xlabel("Surface")
        plt.ylabel("Win Rate")
        plt.ylim(0, 1)
        plt.tight_layout()
        plt.savefig(os.path.join(pdir, "surface_win_rates.png"))
        plt.close()

    # Rest-days histogram
    plt.figure()
    plt.hist(sub["rest_days"].clip(0, 60), bins=20)
    plt.title(f"{player} — Rest Days Distribution")
    plt.xlabel("Rest Days")
    plt.ylabel("Matches")
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "rest_days_hist.png"))
    plt.close()

    # Opponent quality scatter (opp Elo vs date)
    plt.figure()
    plt.scatter(sub["date"], sub["opp_pre_elo"], s=20)
    plt.title(f"{player} — Opponent Elo Over Time")
    plt.xlabel("Date")
    plt.ylabel("Opponent Pre-match Elo")
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "opponent_elo_over_time.png"))
    plt.close()

    # Top H2H opponents (by matches)
    h2h = compute_h2h(log, player).head(15)
    if not h2h.empty:
        plt.figure()
        plt.barh(h2h["opp"][::-1], h2h["win_rate"][::-1])
        plt.title(f"{player} — Top H2H Win Rates")
        plt.xlabel("Win Rate")
        plt.tight_layout()
        plt.savefig(os.path.join(pdir, "top_h2h.png"))
        plt.close()


# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_DIR)
    _ensure_dir(os.path.join(OUT_DIR, "logs"))

    matches = pd.read_csv(PATH_MATCHES)
    try:
        _ = pd.read_csv(PATH_SYM)
    except Exception:
        pass

    log = build_player_match_log(matches, EloParams())
    log_path = safe_write_table(log, os.path.join(OUT_DIR, "logs", "player_match_log.parquet"), index=False)

    prof = compute_player_profiles(log)
    prof_path = safe_write_table(prof, os.path.join(OUT_DIR, "player_profiles.parquet"), index=False)

    # Global visuals
    plot_leaderboards(prof, OUT_DIR)

    # Per-player reports: most active players
    top_players = (
        prof.sort_values(["matches", "current_elo"], ascending=[False, False])
            .head(N_TOP_PLAYERS_FOR_REPORT)["player"].tolist()
    )
    for p in top_players:
        plot_player_report(log, prof, p, OUT_DIR)

    print(f"Saved player profiles table to: {prof_path}")
    print(f"Saved per-player match log to: {log_path}")
    print(f"Saved leaderboards and player reports under: {OUT_DIR}")

if __name__ == "__main__":
    main()


C:\Users\SAM\AppData\Local\Temp\ipykernel_39632\2000205831.py:190: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method="bfill")
C:\Users\SAM\AppData\Local\Temp\ipykernel_39632\2000205831.py:196: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method="bfill")


Parquet engine not available; saved CSV instead: ./reports\logs\player_match_log.csv
Parquet engine not available; saved CSV instead: ./reports\player_profiles.csv
Saved player profiles table to: ./reports\player_profiles.csv
Saved per-player match log to: ./reports\logs\player_match_log.csv
Saved leaderboards and player reports under: ./reports


In [30]:
# filepath: ./build_rf_model_2021_2024.py
"""
Random Forest match outcome model (ATP 2021–2024), leakage-safe, chronological 80/20 split.
Fixes NaN issue by imputing p1_/p2_ BEFORE computing diffs, plus a final numeric imputer.

Run:
    python build_rf_model_2021_2024.py
"""

from __future__ import annotations

import os
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Tuple as TTuple

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# ----------------------------
# Paths & IO helpers
# ----------------------------
PATH_MATCHES = "atp_matches_2021_2024.csv"
OUT_REPORTS = "./reports"
OUT_MODELS = "./models"

def _ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def safe_write_table(df: pd.DataFrame, parquet_path: str, index: bool = False) -> str:
    base, _ = os.path.splitext(parquet_path)
    try:
        df.to_parquet(parquet_path, index=index)
        print(f"Saved: {parquet_path}")
        return parquet_path
    except Exception:
        csv_path = base + ".csv"
        df.to_csv(csv_path, index=index)
        print(f"Parquet engine not available; saved CSV instead: {csv_path}")
        return csv_path

def safe_write_text(text: str, path: str) -> str:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"Saved: {path}")
    return path

# ----------------------------
# Utilities
# ----------------------------
def _safe_int_date(x) -> pd.Timestamp:
    if pd.isna(x):
        return pd.NaT
    s = str(int(x))
    try:
        return pd.to_datetime(s, format="%Y%m%d")
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

# ----------------------------
# Elo + player log (no leakage)
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_int_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []

    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s,
            "tourney_name": r["tourney_name"], "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "is_win": 1, "pre_elo": ew_w, "pre_selo": es_w, "opp_pre_elo": ew_l, "opp_pre_selo": es_l,
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s,
            "tourney_name": r["tourney_name"], "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "is_win": 0, "pre_elo": ew_l, "pre_selo": es_l, "opp_pre_elo": ew_w, "opp_pre_selo": es_w,
            "rank": r[l_rank] if l_rank else np.nan,
        })

        # Elo update
        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        elo_overall[w] = ew_w + K * (1 - exp_w)
        elo_overall[l] = ew_l - K * (1 - exp_w)
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))
        elo_surface[(w, s)] = es_w + KS * (1 - exp_w_s)
        elo_surface[(l, s)] = es_l - KS * (1 - exp_w_s)

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)

    # Rest/workload
    log["prev_date"] = log.groupby("player")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days.clip(lower=0)
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def _rolling_counts(dates: List[pd.Timestamp], window: int) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq)); dq.append(dt)
        return out

    for w in (7, 14, 28):
        log[f"matches_{w}d"] = log.groupby("player")["date"].transform(lambda s: _rolling_counts(list(s), w))

    # Rolling form (prior only)
    log["rolling_10_winrate"] = (
        log.groupby("player")["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    )
    log["rolling_5_winrate"] = (
        log.groupby("player")["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    )

    # Streak (prior)
    def _streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    log["streak"] = log.groupby("player")["is_win"].transform(_streak_prior)

    # Cumulative priors
    log["avg_rest_days"] = log.groupby("player")["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["win_rate"] = log.groupby("player")["is_win"].transform(lambda s: s.shift(1).expanding().mean()).fillna(0.5).clip(0,1)
    log["peak_elo"] = log.groupby("player")["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["current_elo"] = log["pre_elo"]

    # Surface WRs
    log = add_surface_wrs(log)

    # Rank prior
    log["rank_prior"] = log.groupby("player")["rank"].shift(1) if "rank" in log.columns else np.nan

    # H2H prior (Beta(1,1))
    def _h2h_prior(g: pd.DataFrame) -> pd.Series:
        wins = g["is_win"].shift(1).fillna(0)
        csum = wins.cumsum()
        cnt = (~g["is_win"].shift(1).isna()).cumsum()
        return ((1 + csum) / (2 + cnt)).fillna(0.5)
    log["h2h_wr_prior"] = (
        log.sort_values("date").groupby(["player","opp"], sort=False).apply(_h2h_prior).reset_index(level=[0,1], drop=True)
    )

    return log

def add_surface_wrs(log: pd.DataFrame) -> pd.DataFrame:
    log_sorted = log.sort_values(["player","surface","date"])
    wr_surf = (
        log_sorted.groupby(["player","surface"])["is_win"]
                  .transform(lambda s: s.shift(1).expanding().mean())
                  .fillna(0.5).clip(0,1)
    )
    tmp = log_sorted[["player","date","surface"]].copy()
    tmp["wr_surface"] = wr_surf.values

    surf_wide = tmp.pivot_table(index=["player","date"], columns="surface", values="wr_surface", aggfunc="last")
    for s in ["Clay","Grass","Hard"]:
        if s not in surf_wide.columns: surf_wide[s] = np.nan
    surf_wide = surf_wide[["Clay","Grass","Hard"]].sort_values(["player","date"]).reset_index()
    surf_wide[["Clay","Grass","Hard"]] = surf_wide.groupby("player")[["Clay","Grass","Hard"]].ffill()
    surf_wide[["Clay","Grass","Hard"]] = surf_wide[["Clay","Grass","Hard"]].fillna(0.5)

    out = log.merge(surf_wide.rename(columns={"Clay":"wr_Clay","Grass":"wr_Grass","Hard":"wr_Hard"}),
                    on=["player","date"], how="left")
    for c in ["wr_Clay","wr_Grass","wr_Hard"]:
        out[c] = out[c].fillna(0.5)
    return out

# ----------------------------
# Match dataset (impute p1_/p2_ BEFORE diffs)
# ----------------------------
def build_match_dataset(matches: pd.DataFrame, log: pd.DataFrame) -> TTuple[pd.DataFrame, List[str], List[str]]:
    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_int_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    p1 = np.minimum(m["winner_name"], m["loser_name"])
    p2 = np.maximum(m["winner_name"], m["loser_name"])
    y = (m["winner_name"] == p1).astype(int)

    base = pd.DataFrame({
        "match_id": np.arange(len(m)),
        "date": m["tourney_date"],
        "surface": m["surface"].fillna("Other"),
        "tourney_level": m["tourney_level"].fillna("UNK"),
        "round": m["round"].fillna("UNK"),
        "best_of": m["best_of"].fillna(3),
        "p1_name": p1, "p2_name": p2,
        "winner_name": m["winner_name"],
        "y": y,
    })

    key_cols = ["player","opp","date"]
    base_feats = [
        "pre_elo","pre_selo","rest_days","matches_28d","rolling_10_winrate","rolling_5_winrate",
        "streak","avg_rest_days","win_rate","peak_elo","current_elo","wr_Clay","wr_Grass","wr_Hard",
        "rank_prior","h2h_wr_prior"
    ]
    log_keyed = log[key_cols + base_feats].copy().sort_values(key_cols).drop_duplicates(key_cols, keep="last")

    # p1 snapshot
    p1_snap = base.rename(columns={"p1_name":"player","p2_name":"opp"}) \
                 .merge(log_keyed, on=["player","opp","date"], how="left").add_prefix("p1_")
    # p2 snapshot
    p2_snap = base.rename(columns={"p2_name":"player","p1_name":"opp"}) \
                 .merge(log_keyed, on=["player","opp","date"], how="left").add_prefix("p2_")

    ds = base.merge(p1_snap, left_on=["p1_name","p2_name","date"], right_on=["p1_player","p1_opp","p1_date"], how="left") \
             .merge(p2_snap, left_on=["p1_name","p2_name","date"], right_on=["p2_opp","p2_player","p2_date"], how="left")

    # --- Impute p1_/p2_ BEFORE diffs ---
    def _fill_cols(df: pd.DataFrame, prefix: str) -> None:
        cols = [f"{prefix}{c}" for c in base_feats]
        for col in cols:
            if col.endswith(("wr_Clay","wr_Grass","wr_Hard")) or "winrate" in col or col.endswith("win_rate") or "h2h" in col:
                df[col] = df[col].fillna(0.5)
            elif "rest" in col or "matches_28d" in col or "streak" in col:
                df[col] = df[col].fillna(0.0)
            elif "elo" in col or "peak_elo" in col or "current_elo" in col or "selo" in col:
                df[col] = df[col].fillna(1500.0)
            elif "rank" in col:
                df[col] = df[col].fillna(2000.0)
            else:
                df[col] = df[col].fillna(0.0)

    _fill_cols(ds, "p1_")
    _fill_cols(ds, "p2_")

    # Diffs (p1 - p2) after imputation
    for f in base_feats:
        ds[f"diff_{f}"] = ds[f"p1_{f}"] - ds[f"p2_{f}"]

    ds["elo_diff"] = ds["diff_pre_elo"]
    ds["selo_diff"] = ds["diff_pre_selo"]
    ds["rank_diff"] = ds["diff_rank_prior"]

    requested = [
        "diff_avg_rest_days","diff_matches_28d","diff_rolling_10_winrate","diff_rolling_5_winrate",
        "diff_streak","diff_win_rate","diff_peak_elo","diff_current_elo","diff_wr_Clay","diff_wr_Grass","diff_wr_Hard",
    ]
    extras = ["elo_diff","selo_diff","rank_diff","diff_h2h_wr_prior"]
    cat_feats = ["surface","tourney_level","best_of","round"]
    feature_cols = requested + extras + cat_feats

    # Final sanity: ensure no NaN in features
    for c in feature_cols:
        if ds[c].isna().any():
            # For categoricals we already filled; for safety:
            if c in cat_feats:
                ds[c] = ds[c].fillna({"surface":"Other","tourney_level":"UNK","round":"UNK","best_of":3}.get(c, "UNK"))
            else:
                ds[c] = ds[c].fillna(0.0)

    keep_cols = ["match_id","date","p1_name","p2_name","y"] + feature_cols
    ds = ds[keep_cols].sort_values("date").reset_index(drop=True)
    return ds, feature_cols, cat_feats

# ----------------------------
# Train / Evaluate
# ----------------------------
def train_eval_rf(ds: pd.DataFrame, feature_cols: List[str], cat_feats: List[str]) -> TTuple[Pipeline, dict]:
    n = len(ds)
    cutoff_idx = int(round(n * 0.8))
    train = ds.iloc[:cutoff_idx].copy()
    test = ds.iloc[cutoff_idx:].copy()

    y_train = train["y"].values
    y_test = test["y"].values

    cat = [c for c in cat_feats if c in feature_cols]
    num = [c for c in feature_cols if c not in cat]

    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])

    rf = RandomForestClassifier(
        n_estimators=600,
        min_samples_split=6,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
    )

    pipe = Pipeline([("prep", pre), ("rf", rf)])
    pipe.fit(train[feature_cols], y_train)

    p_train = pipe.predict_proba(train[feature_cols])[:,1]
    p_test = pipe.predict_proba(test[feature_cols])[:,1]

    thr = 0.5
    yhat_train = (p_train >= thr).astype(int)
    yhat_test = (p_test >= thr).astype(int)

    metrics = {
        "n_train": int(len(train)),
        "n_test": int(len(test)),
        "cutoff_idx": int(cutoff_idx),
        "cutoff_date": str(ds.iloc[cutoff_idx]["date"].date()) if len(ds) > 0 else "NA",
        "train_acc": float(accuracy_score(y_train, yhat_train)),
        "test_acc": float(accuracy_score(y_test, yhat_test)),
        "train_auc": float(roc_auc_score(y_train, p_train)),
        "test_auc": float(roc_auc_score(y_test, p_test)),
        "train_brier": float(brier_score_loss(y_train, p_train)),
        "test_brier": float(brier_score_loss(y_test, p_test)),
        "cm_test": confusion_matrix(y_test, yhat_test).tolist(),
    }

    # Importances (map back through transformer)
    rf_model = pipe.named_steps["rf"]
    ohe = pipe.named_steps["prep"].named_transformers_["cat"]
    cat_out = list(ohe.get_feature_names_out(cat)) if cat else []
    feature_names = num + cat_out
    importances = rf_model.feature_importances_
    imp_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)

    return pipe, {"metrics": metrics, "feat_importances": imp_df, "splits": (train, test)}

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_REPORTS); _ensure_dir(OUT_MODELS)

    matches = pd.read_csv(PATH_MATCHES)
    log = build_player_log(matches, EloParams())

    ds, feature_cols, cat_feats = build_match_dataset(matches, log)
    ds_path = safe_write_table(ds, os.path.join(OUT_REPORTS, "match_dataset.parquet"), index=False)

    pipe, out = train_eval_rf(ds, feature_cols, cat_feats)

    # Save model
    model_path = os.path.join(OUT_MODELS, "rf_model.joblib")
    dump({"pipeline": pipe, "feature_cols": feature_cols, "cat_feats": cat_feats}, model_path)
    print(f"Saved model: {model_path}")

    # Save metrics & importances
    mt = out["metrics"]
    metrics_txt = "\n".join([
        "RandomForest (chronological 80/20 split)",
        f"Rows train/test: {mt['n_train']} / {mt['n_test']}",
        f"Cutoff date (first test row): {mt['cutoff_date']}",
        f"Train acc:  {mt['train_acc']:.4f}",
        f"Test acc:   {mt['test_acc']:.4f}",
        f"Train AUC:  {mt['train_auc']:.4f}",
        f"Test AUC:   {mt['test_auc']:.4f}",
        f"Train Brier:{mt['train_brier']:.4f}",
        f"Test Brier: {mt['test_brier']:.4f}",
        f"Confusion matrix (test) [[tn, fp],[fn, tp]]: {mt['cm_test']}",
        f"Features used: {feature_cols}",
    ])
    safe_write_text(metrics_txt, os.path.join(OUT_REPORTS, "rf_metrics.txt"))

    imp_path = safe_write_table(out["feat_importances"], os.path.join(OUT_REPORTS, "rf_feature_importances.parquet"), index=False)

    print("Done.")
    print(f"Dataset: {ds_path}")
    print(f"Metrics: {os.path.join(OUT_REPORTS, 'rf_metrics.txt')}")
    print(f"Importances: {imp_path}")
    print(f"Model: {model_path}")

if __name__ == "__main__":
    main()


Parquet engine not available; saved CSV instead: ./reports\match_dataset.csv


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_spar

Saved model: ./models\rf_model.joblib
Saved: ./reports\rf_metrics.txt
Parquet engine not available; saved CSV instead: ./reports\rf_feature_importances.csv
Done.
Dataset: ./reports\match_dataset.csv
Metrics: ./reports\rf_metrics.txt
Importances: ./reports\rf_feature_importances.csv
Model: ./models\rf_model.joblib


In [32]:
# Use this
#  filepath: ./predict_wimbledon_2025_r128.py
"""
Wimbledon 2025 R128 predictions (compact CSV with winner−loser delta metrics).

- Uses ATP 2021–2024 history to build snapshots.
- Rank handling: no 2000 backfill; if either rank missing as-of date, delta_rank=0; else clip to [-500, 500].
- Output columns: match metadata + probabilities + ONLY delta_* metrics (winner − loser).
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
MATCH_PATHS = ["atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

AS_OF_DATE = "2025-07-01"   # Wimbledon 2025
SURFACE = "Grass"
TOURNEY_LEVEL = "G"
ROUND_CODE = "R128"
BEST_OF = 5

OUT_CSV = "./reports/wimbledon2025_R128_predictions.csv"
RANK_CLIP = 500

# ----------------------------
# Bracket (exact order preserved)
# ----------------------------
R128_PAIRS: List[Tuple[str, str]] = [
    ("Jannik Sinner", "Luca Nardi"), 
    ("Chun-Hsin Tseng", "Aleksandar Vukic"),
    ("Pedro Martinez", "George Loffhagen"),
    ("Mariano Navone", "Denis Shapovalov"),
    ("Grigor Dimitrov", "Yoshihito Nishioka"),
    ("Francisco Comesana", "Corentin Moutet"),
    ("Sebastian Ofner", "Hamad Medjedovic"),
    ("Jack Monday", "Tommy Paul"),
    ("Ben Shelton", "Alex Bolt"),
    ("Rinky Hijikata", "David Goffin"),
    ("Aleksandar Kovacevic", "Marton Fucsovics"),
    ("Gael Monfils", "Ugo Humbert"),
    ("Brandon Nakashima", "Bu Yunchaokete"),
    ("Alexander Shevchenko", "Reilly Opelka"),
    ("Joao Faria", "Lorenzo Sonego"),
    ("Nikoloz Basilashvili", "Lorenzo Musetti"),
    ("Jack Draper", "Sebastian Baez"),
    ("Raphael Collignon", "Marin Cilic"),
    ("James McCabe", "Fabian Marozsan"),
    ("Jaume Munar", "Alexander Bublik"),
    ("Flavio Cobolli", "Beibit Zhukayev"),
    ("Tomas Martin Etcheverry", "Jack Pinnington Jones"),
    ("Marcos Giron", "Camilo Ugo Carabelli"),
    ("Hugo Gaston", "Jakub Mensik"),
    ("Alex De Minaur", "Roberto Carballes Baena"),
    ("Arthur Cazaux", "Adam Walton"),
    ("Quentin Halys", "August Holmgren"),
    ("Damir Dzumhur", "Tomas Machac"),
    ("Alex Michelsen", "Miomir Kecmanovic"),
    ("Jesper De Jong", "Christopher Eubanks"),
    ("Daniel Evans", "Jay Clarke"),
    ("Alexandre Muller", "Novak Djokovic"),
    ("Taylor Fritz", "Giovanni Mpetshi Perricard"),
    ("Gabriel Diallo", "Daniel Altmaier"),
    ("Matteo Arnaldi", "Botic van de Zandschulp"),
    ("Brandon Holt", "Alejandro Davidovich Fokina"),
    ("Alexei Popyrin", "Arthur Fery"),
    ("Luciano Darderi", "Roman Safiullin"),
    ("Vit Kopriva", "Jordan Thompson"),
    ("Benjamin Bonzi", "Daniil Medvedev"),
    ("Francisco Cerundolo", "Nuno Borges"),
    ("Billy Harris", "Dusan Lajovic"),
    ("Shintaro Mochizuki", "Giulio Zeppieri"),
    ("Mackenzie McDonald", "Karen Khachanov"),
    ("Matteo Berrettini", "Kamil Majchrzak"),
    ("Ethan Quinn", "Henry Searle"),
    ("Cristian Garin", "Chris Rodesch"),
    ("Arthur Rinderknech", "Alexander Zverev"),
    ("Holger Rune", "Nicolas Jarry"),
    ("Learner Tien", "Nishesh Basavareddy"),
    ("Jacob Fearnley", "Joao Fonseca"),
    ("Jenson Brooksby", "Tallon Griekspoor"),
    ("Jiri Lehecka", "Hugo Dellien"),
    ("Matteo Bellucci", "Oliver Crawford"),
    ("Cameron Norrie", "Roberto Bautista Agut"),
    ("Elmer Moller", "Frances Tiafoe"),
    ("Andrey Rublev", "Laslo Djere"),
    ("Zizou Bergs", "Lloyd Harris"),
    ("Adrian Mannarino", "Christopher O'Connell"),
    ("Valentin Royer", "Stefanos Tsitsipas"),
    ("Felix Auger Aliassime", "James Duckworth"),
    ("Jan Lennard Struff", "Filip Misolic"),
    ("Oliver Tarvet", "Leandro Riedi"),
    ("Fabio Fognini", "Carlos Alcaraz"),
]

# ----------------------------
# Helpers
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

# ----------------------------
# Player history (same as training)
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        # Elo updates
        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        elo_overall[w] = ew_w + K * (1 - exp_w)
        elo_overall[l] = ew_l - K * (1 - exp_w)
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))
        elo_surface[(w, s)] = es_w + KS * (1 - exp_w_s)
        elo_surface[(l, s)] = es_l - KS * (1 - exp_w_s)

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"] = log["opp"].map(canon_name)

    # rest/workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def _rolling_counts(dates: List[pd.Timestamp], window: int) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq)); dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d"] = log.groupby("player_key")["date"].transform(lambda s: _rolling_counts(list(s), w))

    # form & streak (prior-only)
    log["rolling_10_winrate"] = (
        log.groupby("player_key")["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    )
    log["rolling_5_winrate"] = (
        log.groupby("player_key")["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    )
    def _streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    log["streak"] = log.groupby("player_key")["is_win"].transform(_streak_prior)

    # cumulative priors
    log["avg_rest_days"] = log.groupby("player_key")["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["win_rate"]       = log.groupby("player_key")["is_win"].transform(lambda s: s.shift(1).expanding().mean()).fillna(0.5).clip(0,1)
    log["peak_elo"]       = log.groupby("player_key")["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["current_elo"]    = log["pre_elo"]

    # surface WRs (prior), forward-filled
    log_sorted = log.sort_values(["player_key","surface","date"])
    wr_surf = (
        log_sorted.groupby(["player_key","surface"])["is_win"]
                  .transform(lambda s: s.shift(1).expanding().mean())
                  .fillna(0.5).clip(0,1)
    )
    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_surface"] = wr_surf.values
    surf_wide = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_surface", aggfunc="last")
    for s in ["Clay","Grass","Hard"]:
        if s not in surf_wide.columns: surf_wide[s] = np.nan
    surf_wide = surf_wide[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_wide[["Clay","Grass","Hard"]] = surf_wide.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
    surf_wide[["Clay","Grass","Hard"]] = surf_wide[["Clay","Grass","Hard"]].fillna(0.5)
    log = log.merge(
        surf_wide.rename(columns={"Clay":"wr_Clay","Grass":"wr_Grass","Hard":"wr_Hard"}),
        on=["player_key","date"], how="left"
    )
    for c in ["wr_Clay","wr_Grass","wr_Hard"]:
        log[c] = log[c].fillna(0.5)

    # rank prior (NaN when never seen)
    log["rank_prior"] = log.groupby("player_key")["rank"].shift(1)

    return log

# ----------------------------
# Snapshot as-of (by canonical key)
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0, "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        snap.update({
            "pre_elo": float(last["pre_elo"]),
            "current_elo": float(last["pre_elo"]),
            "peak_elo": float(last.get("peak_elo", 1500.0)),
            "win_rate": float(last.get("win_rate", 0.5)),
            "avg_rest_days": float(last.get("avg_rest_days", 20.0)),
            "matches_28d": float(last.get("matches_28d", 0.0)),
            "rolling_10_winrate": float(last.get("rolling_10_winrate", 0.5)),
            "rolling_5_winrate": float(last.get("rolling_5_winrate", 0.5)),
            "streak": float(last.get("streak", 0.0)),
            "wr_Clay": float(last.get("wr_Clay", 0.5)),
            "wr_Grass": float(last.get("wr_Grass", 0.5)),
            "wr_Hard": float(last.get("wr_Hard", 0.5)),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })
        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    # surface-elo snapshot
    sub_surf = log[(log["player_key"] == pk) & (log["surface"] == target_surface) & (log["date"] <= asof)].sort_values("date")
    if not sub_surf.empty:
        snap["pre_selo"] = float(sub_surf["pre_selo"].iloc[-1])

    # H2H prior on keys
    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_CSV)

    model_path = _first_existing(MODEL_PATHS)
    bundle = load(model_path)
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    match_path = _first_existing(MATCH_PATHS)
    matches = pd.read_csv(match_path)
    log = build_player_log(matches, EloParams())

    asof = _safe_date(AS_OF_DATE)

    rows = []
    for i, (A, B) in enumerate(R128_PAIRS, start=1):
        sa = snapshot_asof(log, A, B, asof, SURFACE)
        sb = snapshot_asof(log, B, A, asof, SURFACE)

        # model orientation (alphabetical)
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        # core features for model
        feats = {
            "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_CODE, "best_of": BEST_OF,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # robust rank feature
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        # map back to A/B orientation
        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # compute winner−loser deltas from snapshots
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": AS_OF_DATE,
            "round": ROUND_CODE,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            # ---- winner − loser deltas (concise) ----
            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,
        })

    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)
    _ensure_dir(OUT_CSV)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved predictions (deltas only) to: {OUT_CSV}")
    print(out_df.head(8).to_string(index=False))

if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

Saved predictions (deltas only) to: ./reports/wimbledon2025_R128_predictions.csv
 match_no       date round           player_a           player_b        pred_winner  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-07-01  R128      Jannik Sinner         Luca Nardi      Jannik Sinner    0.866481           0.866481           0.133519 654.716332         130.245017        0.416667       0.528571       0.533333        0.545287         654.716332      570.525262                6.0           -18.549195                       0.8                      1.0          19.0            0.000000       -86.0
        2 2025-07-01  R128    Chun-Hsin Tseng   Aleksandar Vukic   Aleksandar Vukic    0.791988           0.208012         

In [33]:
# filepath: ./predict_wimbledon_2025_r64.py
"""
Wimbledon 2025 R64 predictions (deltas only) with Beta-smoothed win rates.

- WR smoothing reduces small-sample volatility:
  wr_smoothed = (ALPHA + wins) / (ALPHA + BETA + matches), default ALPHA=BETA=5.
- Uses post-R128 metrics for R64 (Elo, surface Elo, WRs, form, streak, workload, etc.).
- Rank handling: neutralize when missing; otherwise clip to [-500, 500].
- Output columns/order match your R128 file; leaves `correct_prediction` blank.
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
MATCH_PATHS = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]
R128_CSV = "./reports/wimbledon2025_R128_predictions_complete.csv"

OUT_CSV_R64 = "./reports/wimbledon2025_R64_predictions.csv"

SURFACE = "Grass"
TOURNEY_NAME = "Wimbledon"
TOURNEY_LEVEL = "G"
BEST_OF = 5
ROUND_R128 = "R128"
ROUND_R64 = "R64"

# Elo/Rank knobs
RANK_CLIP = 500

# Empirical-Bayes smoothing prior strength
SMOOTH_ALPHA = 5.0  # why: shrink extreme WRs toward 0.5 on small samples
SMOOTH_BETA  = 5.0

# Use R128 outcomes to update metrics before R64
INCORPORATE_R128_RESULTS = True


# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"


# ----------------------------
# Player log (prior + post + smoothed WRs)
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    # Elo states
    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        # pre
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        # expected
        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        # post
        new_ew_w = ew_w + K * (1 - exp_w)
        new_ew_l = ew_l - K * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        # commit state
        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"] = log["opp"].map(canon_name)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form & streak (prior vs post)
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)
    log["win_rate_prior_raw"]  = grp["is_win"].transform(lambda s: s.shift(1).expanding().mean()).fillna(0.5).clip(0,1)
    log["win_rate_post_raw"]   = grp["is_win"].transform(lambda s: s.expanding().mean()).fillna(0.5).clip(0,1)

    # surface WRs prior/post (wins & matches for smoothing)
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(wins: pd.Series, matches: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + wins) / (SMOOTH_ALPHA + SMOOTH_BETA + matches)

    wr_prior_sm = smooth(wins_prior, matches_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  matches_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # overall WR smoothing prior/post
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # peaks using post Elo
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)

    # historical rank snapshot
    log["rank_prior"] = log.groupby("player_key")["rank"].shift(1)

    return log


# ----------------------------
# Snapshot (post-aware + smoothed WRs)
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        # choose prior vs post (smoothed where applicable)
        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last["matches_28d_post"] if after_last else last["matches_28d_prior"]
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        # surface-elo & WRs (smoothed)
        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])
            snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        # other surfaces for deltas
        snap["wr_Hard"] = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"] = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    # H2H to as-of
    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap


# ----------------------------
# Build R64 from R128 CSV
# ----------------------------
def derive_r64_pairs(r128: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    r128 = r128.sort_values("match_no").reset_index(drop=True)
    def actual_winner(r) -> str:
        if "correct_prediction" in r and pd.notna(r["correct_prediction"]):
            cp = int(r["correct_prediction"])
            return r["pred_winner"] if cp == 1 else (r["player_b"] if r["pred_winner"] == r["player_a"] else r["player_a"])
        return r["pred_winner"]
    winners = r128.apply(actual_winner, axis=1).tolist()
    pairs: List[Tuple[str,str]] = []
    for i in range(0, len(winners), 2):
        pairs.append((winners[i], winners[i+1]))
    d_r128 = _safe_date(r128["date"].iloc[0])
    d_r64 = d_r128 + pd.Timedelta(days=2)  # simple day offset
    return pairs, d_r128, d_r64

def augment_with_r128_results(base_matches: pd.DataFrame, r128: pd.DataFrame) -> pd.DataFrame:
    r128 = r128.sort_values("match_no").reset_index(drop=True)
    actual_rows = []
    for _, r in r128.iterrows():
        if "correct_prediction" in r and pd.notna(r["correct_prediction"]):
            cp = int(r["correct_prediction"])
            winner = r["pred_winner"] if cp == 1 else (r["player_b"] if r["pred_winner"] == r["player_a"] else r["player_a"])
        else:
            winner = r["pred_winner"]
        loser = r["player_b"] if winner == r["player_a"] else r["player_a"]
        actual_rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": TOURNEY_LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": winner,
            "loser_name": loser,
            "round": ROUND_R128,
            "best_of": BEST_OF,
            "winner_rank": np.nan, "loser_rank": np.nan,
        })
    extra = pd.DataFrame(actual_rows)
    common = [c for c in extra.columns if c in base_matches.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"] if c not in common]
    extra = extra[common + add]
    return pd.concat([base_matches, extra], ignore_index=True, sort=False)


# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_CSV_R64)

    # model
    bundle = load(_first_existing(MODEL_PATHS))
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # inputs
    r128 = pd.read_csv(R128_CSV)
    base_matches = pd.read_csv(_first_existing(MATCH_PATHS))

    # pairs & dates
    r64_pairs, date_r128, date_r64 = derive_r64_pairs(r128)

    # include R128 outcomes so post metrics reflect them
    matches_for_log = augment_with_r128_results(base_matches, r128) if INCORPORATE_R128_RESULTS else base_matches
    log = build_player_log(matches_for_log, EloParams())

    # predict
    rows = []
    for i, (A, B) in enumerate(r64_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_r64, SURFACE)
        sb = snapshot_asof(log, B, A, date_r64, SURFACE)

        # internal alphabetical orientation for the model
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_R64, "best_of": BEST_OF,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # robust rank
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # deltas winner − loser (using smoothed metrics)
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_r64.strftime("%Y-%m-%d"),
            "round": ROUND_R64,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,  # left blank to match your R128 format
        })

    r64_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Match exact column order of your R128 file
    r128_cols = list(pd.read_csv(R128_CSV, nrows=0).columns)
    for c in r128_cols:
        if c not in r64_df.columns:
            r64_df[c] = np.nan
    r64_df = r64_df[r128_cols].copy()

    _ensure_dir(OUT_CSV_R64)
    r64_df.to_csv(OUT_CSV_R64, index=False, encoding="utf-8-sig")
    print(f"Saved R64 predictions with smoothed WRs to: {OUT_CSV_R64}")
    print(r64_df.head(8).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

Saved R64 predictions with smoothed WRs to: ./reports/wimbledon2025_R64_predictions.csv
 match_no       date round          player_a             player_b       pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-07-03   R64     Jannik Sinner     Aleksandar Vukic     Jannik Sinner                 NaN    0.870516           0.870516           0.129484 508.545095          87.388002        0.123276       0.356461       0.281050        0.322535         508.545095      508.545095                0.0            -6.156412                       0.3                      0.2          14.0            0.500000       -84.0
        2 2025-07-03   R64    Pedro Martinez       Mariano Navone    Mariano

In [35]:
# filepath: ./predict_wimbledon_2025_r32.py
"""
Wimbledon 2025 R32 predictions (deltas only), bracket order preserved.

- Uses ATP 2021–2024 + injected R128/R64 outcomes to update Elo/surface-Elo/form/WRs.
- Empirical-Bayes smoothing for WRs: (alpha+wins)/(alpha+beta+matches) with alpha=beta=5.
- Rank: neutralize when missing; else clip to [-500, 500].
- Output columns/order match your R128 CSV; `correct_prediction` left blank.

Adjust paths below if your files live elsewhere.
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
MATCH_PATHS = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

R128_CSV = "./reports/wimbledon2025_R128_predictions_complete.csv"   # template columns + results
R64_CSV  = "./reports/wimbledon2025_R64_predictions_complete.csv"    # source for R32 bracket

OUT_CSV_R32 = "./reports/wimbledon2025_R32_predictions.csv"

SURFACE = "Grass"
TOURNEY_NAME = "Wimbledon"
TOURNEY_LEVEL = "G"
BEST_OF = 5
ROUND_R128 = "R128"
ROUND_R64  = "R64"
ROUND_R32  = "R32"

# Elo/Rank knobs
RANK_CLIP = 500

# Empirical-Bayes smoothing for WRs
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Helpers
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

# ----------------------------
# Player log with prior/post + smoothed WRs
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        # pre
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        # expected
        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        # post
        new_ew_w = ew_w + K * (1 - exp_w)
        new_ew_l = ew_l - K * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"] = log["opp"].map(canon_name)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form & streak
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)
    log["win_rate_prior_raw"]  = grp["is_win"].transform(lambda s: s.shift(1).expanding().mean()).fillna(0.5).clip(0,1)
    log["win_rate_post_raw"]   = grp["is_win"].transform(lambda s: s.expanding().mean()).fillna(0.5).clip(0,1)

    # per-surface wins/matches for smoothing
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(wins: pd.Series, matches: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + wins) / (SMOOTH_ALPHA + SMOOTH_BETA + matches)

    wr_prior_sm = smooth(wins_prior, matches_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  matches_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # peaks
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)

    # historical rank snapshot
    log["rank_prior"] = log.groupby("player_key")["rank"].shift(1)

    return log

# ----------------------------
# Snapshot (prior/post aware + smoothed WRs)
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last["matches_28d_post"] if after_last else last["matches_28d_prior"]
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])
            snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        snap["wr_Hard"] = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"] = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    # H2H to as-of
    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Round utilities
# ----------------------------
def load_round(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df.sort_values("match_no").reset_index(drop=True)

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs: List[Tuple[str,str]] = []
    for i in range(0, len(winners), 2):
        pairs.append((winners[i], winners[i+1]))
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)
    return pairs, d_prev, d_next

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str) -> pd.DataFrame:
    rows = []
    for _, r in round_df.sort_values("match_no").iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": TOURNEY_LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": BEST_OF,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_CSV_R32)

    # model
    bundle = load(_first_existing(MODEL_PATHS))
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # inputs
    r64 = load_round(R64_CSV)
    base_matches = pd.read_csv(_first_existing(MATCH_PATHS))

    # derive R32 bracket & dates
    r32_pairs, date_r64, date_r32 = derive_next_pairs(r64)

    # include R128 and R64 outcomes so metrics reflect both rounds
    matches_for_log = base_matches
    if os.path.exists(R128_CSV):
        r128 = load_round(R128_CSV)
        matches_for_log = augment_with_results(matches_for_log, r128, ROUND_R128)
    matches_for_log = augment_with_results(matches_for_log, r64, ROUND_R64)

    # build log with post metrics + smoothed WRs
    log = build_player_log(matches_for_log, EloParams())

    # predict R32
    rows = []
    for i, (A, B) in enumerate(r32_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_r32, SURFACE)
        sb = snapshot_asof(log, B, A, date_r32, SURFACE)

        # model orientation (alphabetical)
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_R32, "best_of": BEST_OF,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # deltas winner − loser
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_r32.strftime("%Y-%m-%d"),
            "round": ROUND_R32,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,  # fill later
        })

    r32_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Match the exact column order from your R128 file
    r128_cols = list(pd.read_csv(R128_CSV, nrows=0).columns)
    for c in r128_cols:
        if c not in r32_df.columns:
            r32_df[c] = np.nan
    r32_df = r32_df[r128_cols].copy()

    _ensure_dir(OUT_CSV_R32)
    r32_df.to_csv(OUT_CSV_R32, index=False, encoding="utf-8-sig")
    print(f"Saved R32 predictions to: {OUT_CSV_R32}")
    print(r32_df.head(8).to_string(index=False))

if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

Saved R32 predictions to: ./reports/wimbledon2025_R32_predictions.csv
 match_no       date round          player_a         player_b       pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-07-05   R32     Jannik Sinner   Pedro Martinez     Jannik Sinner                 NaN    0.906179           0.906179           0.093821 590.928004         140.102917        0.228381       0.406966       0.202336        0.339553         590.928004      497.603227                0.0            -4.105068                       0.7                      0.6          14.0            0.333333       -41.0
        2 2025-07-05   R32   Grigor Dimitrov  Sebastian Ofner   Grigor Dimitrov                 NaN   

In [36]:
# filepath: ./predict_wimbledon_2025_r16.py
"""
Wimbledon 2025 R16 predictions (deltas only), bracket order preserved.

- Uses ATP 2021–2024 + injected R128/R64/R32 outcomes so Elo, surface-Elo, form,
  and (smoothed) WRs reflect prior rounds.
- Empirical-Bayes smoothing for WRs: (alpha+wins)/(alpha+beta+matches) with alpha=beta=5.
- Rank: neutralize when missing; else clip to [-500, 500].
- Output columns/order match your R128 CSV; `correct_prediction` left blank.
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline


# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
MATCH_PATHS = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

R128_CSV = "./reports/wimbledon2025_R128_predictions_complete.csv"   # has correct_prediction
R64_CSV  = "./reports/wimbledon2025_R64_predictions_complete.csv"    # has correct_prediction
R32_CSV  = "./reports/wimbledon2025_R32_predictions_complete.csv"    # has correct_prediction

OUT_CSV_R16 = "./reports/wimbledon2025_R16_predictions.csv"

SURFACE = "Grass"
TOURNEY_NAME = "Wimbledon"
TOURNEY_LEVEL = "G"
BEST_OF = 5
ROUND_R128 = "R128"
ROUND_R64  = "R64"
ROUND_R32  = "R32"
ROUND_R16  = "R16"

RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0


# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"


# ----------------------------
# Player log (prior/post + smoothed WRs)
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        # pre
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        # expected
        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        # post
        new_ew_w = ew_w + K * (1 - exp_w)
        new_ew_l = ew_l - K * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"] = log["opp"].map(canon_name)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form & streak
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)
    log["win_rate_prior_raw"]  = grp["is_win"].transform(lambda s: s.shift(1).expanding().mean()).fillna(0.5).clip(0,1)
    log["win_rate_post_raw"]   = grp["is_win"].transform(lambda s: s.expanding().mean()).fillna(0.5).clip(0,1)

    # per-surface wins/matches for smoothing
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(wins: pd.Series, matches: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + wins) / (SMOOTH_ALPHA + SMOOTH_BETA + matches)

    wr_prior_sm = smooth(wins_prior, matches_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  matches_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # peaks
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)

    # historical rank snapshot
    log["rank_prior"] = log.groupby("player_key")["rank"].shift(1)

    return log


# ----------------------------
# Snapshot (prior/post aware + smoothed WRs)
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last["matches_28d_post"] if after_last else last["matches_28d_prior"]
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])
            snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        snap["wr_Hard"] = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"] = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap


# ----------------------------
# Round utils
# ----------------------------
def load_round(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df.sort_values("match_no").reset_index(drop=True)

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs: List[Tuple[str,str]] = []
    for i in range(0, len(winners), 2):
        pairs.append((winners[i], winners[i+1]))
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)
    return pairs, d_prev, d_next

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str) -> pd.DataFrame:
    rows = []
    for _, r in round_df.sort_values("match_no").iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": TOURNEY_LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": BEST_OF,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)


# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_CSV_R16)

    # model
    bundle = load(_first_existing(MODEL_PATHS))
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # inputs
    r32 = load_round(R32_CSV)
    base_matches = pd.read_csv(_first_existing(MATCH_PATHS))

    # derive R16 bracket & dates
    r16_pairs, date_r32, date_r16 = derive_next_pairs(r32)

    # include all prior rounds to date (R128, R64, R32)
    matches_for_log = base_matches
    if os.path.exists(R128_CSV):
        r128 = load_round(R128_CSV)
        matches_for_log = augment_with_results(matches_for_log, r128, ROUND_R128)
    if os.path.exists(R64_CSV):
        r64 = load_round(R64_CSV)
        matches_for_log = augment_with_results(matches_for_log, r64, ROUND_R64)
    matches_for_log = augment_with_results(matches_for_log, r32, ROUND_R32)

    # player log
    log = build_player_log(matches_for_log, EloParams())

    # predict R16
    rows = []
    for i, (A, B) in enumerate(r16_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_r16, SURFACE)
        sb = snapshot_asof(log, B, A, date_r16, SURFACE)

        # alphabetical orientation for model
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_R16, "best_of": BEST_OF,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # deltas (winner − loser)
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_r16.strftime("%Y-%m-%d"),
            "round": ROUND_R16,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,  # fill later
        })

    r16_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Column order = R128 header
    r128_cols = list(pd.read_csv(R128_CSV, nrows=0).columns)
    for c in r128_cols:
        if c not in r16_df.columns:
            r16_df[c] = np.nan
    r16_df = r16_df[r128_cols].copy()

    _ensure_dir(OUT_CSV_R16)
    r16_df.to_csv(OUT_CSV_R16, index=False, encoding="utf-8-sig")
    print(f"Saved R16 predictions to: {OUT_CSV_R16}")
    print(r16_df.head(8).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

Saved R16 predictions to: ./reports/wimbledon2025_R16_predictions.csv
 match_no       date round        player_a        player_b     pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-07-07   R16   Jannik Sinner Grigor Dimitrov   Jannik Sinner                 NaN    0.792447           0.792447           0.207553 278.021436          68.948930        0.077573       0.165685       0.099977        0.148220         278.021436      213.275475                0.0            -1.682796                       0.3                      0.2          14.0            0.666667        -8.0
        2 2025-07-07   R16     Ben Shelton  Lorenzo Sonego     Ben Shelton                 NaN    0.605559      

c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

In [37]:
# filepath: ./predict_wimbledon_2025_qf.py
"""
Wimbledon 2025 QF predictions (deltas only), bracket order preserved.

- Uses ATP 2021–2024 + injected R128/R64/R32/R16 outcomes to update Elo/surface-Elo/form/WRs.
- Empirical-Bayes smoothing for WRs: (alpha+wins)/(alpha+beta+matches), default alpha=beta=5.
- Rank: neutralize when missing; else clip to [-500, 500].
- Output columns/order match your R128 CSV; `correct_prediction` left blank.

Adjust the CSV paths below if your files live elsewhere.
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline


# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
MATCH_PATHS = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

R128_CSV = "./reports/wimbledon2025_R128_predictions_complete.csv"
R64_CSV  = "./reports/wimbledon2025_R64_predictions_complete.csv"
R32_CSV  = "./reports/wimbledon2025_R32_predictions_complete.csv"
R16_CSV  = "./reports/wimbledon2025_R16_predictions_complete.csv"

OUT_CSV_QF = "./reports/wimbledon2025_QF_predictions.csv"

SURFACE = "Grass"
TOURNEY_NAME = "Wimbledon"
TOURNEY_LEVEL = "G"
BEST_OF = 5
ROUND_R128 = "R128"
ROUND_R64  = "R64"
ROUND_R32  = "R32"
ROUND_R16  = "R16"
ROUND_QF   = "QF"

RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0


# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"


# ----------------------------
# Player log (prior/post + smoothed WRs)
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        # pre
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        # expected
        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        # post
        new_ew_w = ew_w + K * (1 - exp_w)
        new_ew_l = ew_l - K * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"] = log["opp"].map(canon_name)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form & streak
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)
    log["win_rate_prior_raw"]  = grp["is_win"].transform(lambda s: s.shift(1).expanding().mean()).fillna(0.5).clip(0,1)
    log["win_rate_post_raw"]   = grp["is_win"].transform(lambda s: s.expanding().mean()).fillna(0.5).clip(0,1)

    # per-surface wins/matches for smoothing
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(wins: pd.Series, matches: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + wins) / (SMOOTH_ALPHA + SMOOTH_BETA + matches)

    wr_prior_sm = smooth(wins_prior, matches_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  matches_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # peaks
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)

    # historical rank snapshot
    log["rank_prior"] = log.groupby("player_key")["rank"].shift(1)

    return log


# ----------------------------
# Snapshot (prior/post aware + smoothed WRs)
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last["matches_28d_post"] if after_last else last["matches_28d_prior"]
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])
            snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        snap["wr_Hard"] = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"] = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap


# ----------------------------
# Round utilities
# ----------------------------
def load_round(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing round file: {path}")
    df = pd.read_csv(path)
    return df.sort_values("match_no").reset_index(drop=True)

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs: List[Tuple[str,str]] = []
    for i in range(0, len(winners), 2):
        pairs.append((winners[i], winners[i+1]))
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)
    return pairs, d_prev, d_next

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str) -> pd.DataFrame:
    rows = []
    for _, r in round_df.sort_values("match_no").iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": TOURNEY_LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": BEST_OF,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)


# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_CSV_QF)

    # model
    bundle = load(_first_existing(MODEL_PATHS))
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # inputs
    r16 = load_round(R16_CSV)
    base_matches = pd.read_csv(_first_existing(MATCH_PATHS))

    # derive QF bracket & dates
    qf_pairs, date_r16, date_qf = derive_next_pairs(r16)

    # include all prior outcomes (R128, R64, R32, R16)
    matches_for_log = base_matches
    if os.path.exists(R128_CSV):
        r128 = load_round(R128_CSV)
        matches_for_log = augment_with_results(matches_for_log, r128, ROUND_R128)
    if os.path.exists(R64_CSV):
        r64 = load_round(R64_CSV)
        matches_for_log = augment_with_results(matches_for_log, r64, ROUND_R64)
    if os.path.exists(R32_CSV):
        r32 = load_round(R32_CSV)
        matches_for_log = augment_with_results(matches_for_log, r32, ROUND_R32)
    matches_for_log = augment_with_results(matches_for_log, r16, ROUND_R16)

    # player log with smoothed WRs and post metrics
    log = build_player_log(matches_for_log, EloParams())

    # predict QF
    rows = []
    for i, (A, B) in enumerate(qf_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_qf, SURFACE)
        sb = snapshot_asof(log, B, A, date_qf, SURFACE)

        # alphabetical orientation for the model
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_QF, "best_of": BEST_OF,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # robust rank
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # winner − loser deltas (smoothed + post-aware)
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_qf.strftime("%Y-%m-%d"),
            "round": ROUND_QF,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,  # you fill later
        })

    qf_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Column order = R128 header
    r128_cols = list(pd.read_csv(R128_CSV, nrows=0).columns)
    for c in r128_cols:
        if c not in qf_df.columns:
            qf_df[c] = np.nan
    qf_df = qf_df[r128_cols].copy()

    _ensure_dir(OUT_CSV_QF)
    qf_df.to_csv(OUT_CSV_QF, index=False, encoding="utf-8-sig")
    print(f"Saved QF predictions to: {OUT_CSV_QF}")
    print(qf_df.head(8).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

Saved QF predictions to: ./reports/wimbledon2025_QF_predictions.csv
 match_no       date round       player_a        player_b    pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-07-09    QF  Jannik Sinner     Ben Shelton  Jannik Sinner                 NaN    0.862423           0.862423           0.137577 368.474581         104.630819        0.142119       0.204945       0.206270        0.204310         368.474581      350.364925                0.0            -1.981541                       0.3                      0.2          14.0            0.428571       -20.0
        2 2025-07-09    QF Flavio Cobolli  Novak Djokovic Novak Djokovic                 NaN    0.887490           0.1

In [38]:
# filepath: ./predict_wimbledon_2025_sf.py
"""
Wimbledon 2025 SF predictions (deltas only), bracket order preserved.

- Uses ATP 2021–2024 + injected R128/R64/R32/R16/QF outcomes to update Elo/surface-Elo/form/WRs.
- Empirical-Bayes smoothing for WRs: (alpha+wins)/(alpha+beta+matches), alpha=beta=5.
- Rank: neutralize when missing; else clip to [-500, 500].
- Output columns/order match your R128 CSV; `correct_prediction` left blank.
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline


# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
MATCH_PATHS = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

R128_CSV = "./reports/wimbledon2025_R128_predictions_complete.csv"
R64_CSV  = "./reports/wimbledon2025_R64_predictions_complete.csv"
R32_CSV  = "./reports/wimbledon2025_R32_predictions_complete.csv"
R16_CSV  = "./reports/wimbledon2025_R16_predictions_complete.csv"
QF_CSV   = "./reports/wimbledon2025_QF_predictions_complete.csv"

OUT_CSV_SF = "./reports/wimbledon2025_SF_predictions.csv"

SURFACE = "Grass"
TOURNEY_NAME = "Wimbledon"
TOURNEY_LEVEL = "G"
BEST_OF = 5
ROUND_R128 = "R128"
ROUND_R64  = "R64"
ROUND_R32  = "R32"
ROUND_R16  = "R16"
ROUND_QF   = "QF"
ROUND_SF   = "SF"

RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0


# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"


# ----------------------------
# Player log (prior/post + smoothed WRs)
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        # pre
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        # expected
        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        # post
        new_ew_w = ew_w + K * (1 - exp_w)
        new_ew_l = ew_l - K * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"] = log["opp"].map(canon_name)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form & streak
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)
    log["win_rate_prior_raw"]  = grp["is_win"].transform(lambda s: s.shift(1).expanding().mean()).fillna(0.5).clip(0,1)
    log["win_rate_post_raw"]   = grp["is_win"].transform(lambda s: s.expanding().mean()).fillna(0.5).clip(0,1)

    # per-surface wins/matches for smoothing
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(wins: pd.Series, matches: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + wins) / (SMOOTH_ALPHA + SMOOTH_BETA + matches)

    wr_prior_sm = smooth(wins_prior, matches_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  matches_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # peaks
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)

    # historical rank snapshot
    log["rank_prior"] = log.groupby("player_key")["rank"].shift(1)

    return log


# ----------------------------
# Snapshot (prior/post aware + smoothed WRs)
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last["matches_28d_post"] if after_last else last["matches_28d_prior"]
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])
            snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        snap["wr_Hard"] = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"] = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap


# ----------------------------
# Round utilities
# ----------------------------
def load_round(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing round file: {path}")
    df = pd.read_csv(path)
    return df.sort_values("match_no").reset_index(drop=True)

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs: List[Tuple[str,str]] = []
    for i in range(0, len(winners), 2):
        pairs.append((winners[i], winners[i+1]))
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)
    return pairs, d_prev, d_next

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str) -> pd.DataFrame:
    rows = []
    for _, r in round_df.sort_values("match_no").iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": TOURNEY_LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": BEST_OF,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)


# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_CSV_SF)

    # model
    bundle = load(_first_existing(MODEL_PATHS))
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # inputs
    qf = load_round(QF_CSV)
    base_matches = pd.read_csv(_first_existing(MATCH_PATHS))

    # derive SF bracket & dates
    sf_pairs, date_qf, date_sf = derive_next_pairs(qf)

    # include all prior outcomes (R128, R64, R32, R16, QF)
    matches_for_log = base_matches
    if os.path.exists(R128_CSV):
        r128 = load_round(R128_CSV)
        matches_for_log = augment_with_results(matches_for_log, r128, ROUND_R128)
    if os.path.exists(R64_CSV):
        r64 = load_round(R64_CSV)
        matches_for_log = augment_with_results(matches_for_log, r64, ROUND_R64)
    if os.path.exists(R32_CSV):
        r32 = load_round(R32_CSV)
        matches_for_log = augment_with_results(matches_for_log, r32, ROUND_R32)
    if os.path.exists(R16_CSV):
        r16 = load_round(R16_CSV)
        matches_for_log = augment_with_results(matches_for_log, r16, ROUND_R16)
    matches_for_log = augment_with_results(matches_for_log, qf, ROUND_QF)

    # player log with post metrics + smoothed WRs
    log = build_player_log(matches_for_log, EloParams())

    # predict SF
    rows = []
    for i, (A, B) in enumerate(sf_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_sf, SURFACE)
        sb = snapshot_asof(log, B, A, date_sf, SURFACE)

        # alphabetical orientation for model features
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_SF, "best_of": BEST_OF,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # deltas winner − loser (smoothed + post-aware)
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_sf.strftime("%Y-%m-%d"),
            "round": ROUND_SF,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,  # fill later
        })

    sf_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Column order = R128 header
    r128_cols = list(pd.read_csv(R128_CSV, nrows=0).columns)
    for c in r128_cols:
        if c not in sf_df.columns:
            sf_df[c] = np.nan
    sf_df = sf_df[r128_cols].copy()

    _ensure_dir(OUT_CSV_SF)
    sf_df.to_csv(OUT_CSV_SF, index=False, encoding="utf-8-sig")
    print(f"Saved SF predictions to: {OUT_CSV_SF}")
    print(sf_df.head(8).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

Saved SF predictions to: ./reports/wimbledon2025_SF_predictions.csv
 match_no       date round      player_a       player_b    pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-07-11    SF Jannik Sinner Novak Djokovic  Jannik Sinner                 NaN    0.501822           0.501822           0.498178  93.308510         -55.472823       -0.132664      -0.046423      -0.086330       -0.067727          93.308510       31.148433                0.0            -1.175457                       0.1                      0.0          14.0                 0.0        -3.0
        2 2025-07-11    SF  Taylor Fritz Carlos Alcaraz Carlos Alcaraz                 NaN    0.605735           0.394265 

c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

In [40]:
# filepath: ./predict_wimbledon_2025_f.py
"""
Wimbledon 2025 Final predictions (deltas only), bracket order preserved.

- Uses ATP 2021–2024 + injected R128/R64/R32/R16/QF/SF outcomes so Elo/surface-Elo/form
  and (smoothed) WRs reflect prior rounds.
- Empirical-Bayes smoothing for WRs: (alpha+wins)/(alpha+beta+matches), alpha=beta=5.
- Rank: neutralize when missing; else clip to [-500, 500].
- Output columns/order match your R128 CSV; `correct_prediction` left blank.
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
MATCH_PATHS = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

R128_CSV = "./reports/wimbledon2025_R128_predictions_complete.csv"
R64_CSV  = "./reports/wimbledon2025_R64_predictions_complete.csv"
R32_CSV  = "./reports/wimbledon2025_R32_predictions_complete.csv"
R16_CSV  = "./reports/wimbledon2025_R16_predictions_complete.csv"
QF_CSV   = "./reports/wimbledon2025_QF_predictions_complete.csv"
SF_CSV   = "./reports/wimbledon2025_SF_predictions_complete.csv"   # required

OUT_CSV_F = "./reports/wimbledon2025_F_predictions.csv"

SURFACE = "Grass"
TOURNEY_NAME = "Wimbledon"
TOURNEY_LEVEL = "G"
BEST_OF = 5
ROUND_R128 = "R128"
ROUND_R64  = "R64"
ROUND_R32  = "R32"
ROUND_R16  = "R16"
ROUND_QF   = "QF"
ROUND_SF   = "SF"
ROUND_F    = "F"

RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

# ----------------------------
# Player log (prior/post + smoothed WRs)
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K * (1 - exp_w)
        new_ew_l = ew_l - K * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        # commit state
        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"] = log["opp"].map(canon_name)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form & streak
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)
    log["win_rate_prior_raw"]  = grp["is_win"].transform(lambda s: s.shift(1).expanding().mean()).fillna(0.5).clip(0,1)
    log["win_rate_post_raw"]   = grp["is_win"].transform(lambda s: s.expanding().mean()).fillna(0.5).clip(0,1)

    # per-surface wins/matches for smoothing
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(wins: pd.Series, matches: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + wins) / (SMOOTH_ALPHA + SMOOTH_BETA + matches)

    wr_prior_sm = smooth(wins_prior, matches_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  matches_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # peaks
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)

    # historical rank snapshot
    log["rank_prior"] = log.groupby("player_key")["rank"].shift(1)

    return log

# ----------------------------
# Snapshot (prior/post aware + smoothed WRs)
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last["matches_28d_post"] if after_last else last["matches_28d_prior"]
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])
            snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        snap["wr_Hard"] = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"] = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Round utilities
# ----------------------------
def load_round(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing round file: {path}")
    df = pd.read_csv(path)
    return df.sort_values("match_no").reset_index(drop=True)

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs: List[Tuple[str,str]] = []
    for i in range(0, len(winners), 2):
        pairs.append((winners[i], winners[i+1]))
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)
    return pairs, d_prev, d_next

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str) -> pd.DataFrame:
    rows = []
    for _, r in round_df.sort_values("match_no").iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": TOURNEY_LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": BEST_OF,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_CSV_F)

    # model
    bundle = load(_first_existing(MODEL_PATHS))
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # inputs
    sf = load_round(SF_CSV)
    base_matches = pd.read_csv(_first_existing(MATCH_PATHS))

    # Final bracket & dates
    f_pairs, date_sf, date_f = derive_next_pairs(sf)  # one pair
    if len(f_pairs) != 1:
        raise ValueError(f"Expected 1 SF pairing → 1 Final, got {len(f_pairs)}")

    # include all prior outcomes through SF
    matches_for_log = base_matches
    if os.path.exists(R128_CSV):
        r128 = load_round(R128_CSV)
        matches_for_log = augment_with_results(matches_for_log, r128, ROUND_R128)
    if os.path.exists(R64_CSV):
        r64 = load_round(R64_CSV)
        matches_for_log = augment_with_results(matches_for_log, r64, ROUND_R64)
    if os.path.exists(R32_CSV):
        r32 = load_round(R32_CSV)
        matches_for_log = augment_with_results(matches_for_log, r32, ROUND_R32)
    if os.path.exists(R16_CSV):
        r16 = load_round(R16_CSV)
        matches_for_log = augment_with_results(matches_for_log, r16, ROUND_R16)
    if os.path.exists(QF_CSV):
        qf = load_round(QF_CSV)
        matches_for_log = augment_with_results(matches_for_log, qf, ROUND_QF)
    matches_for_log = augment_with_results(matches_for_log, sf, ROUND_SF)

    # player log
    log = build_player_log(matches_for_log, EloParams())

    # predict Final
    rows = []
    (A, B) = f_pairs[0]

    sa = snapshot_asof(log, A, B, date_f, SURFACE)
    sb = snapshot_asof(log, B, A, date_f, SURFACE)

    # alphabetical orientation for model
    p1, p2 = (A, B) if A <= B else (B, A)
    s1, s2 = (sa, sb) if p1 == A else (sb, sa)

    def d(name: str) -> float:
        return float(s1[name]) - float(s2[name])

    feats = {
        "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_F, "best_of": BEST_OF,
        "elo_diff": d("pre_elo"),
        "selo_diff": d("pre_selo"),
        "diff_wr_Grass": d("wr_Grass"),
        "diff_wr_Hard": d("wr_Hard"),
        "diff_wr_Clay": d("wr_Clay"),
        "diff_win_rate": d("win_rate"),
        "diff_current_elo": d("current_elo"),
        "diff_peak_elo": d("peak_elo"),
        "diff_matches_28d": d("matches_28d"),
        "diff_avg_rest_days": d("avg_rest_days"),
        "diff_rolling_10_winrate": d("rolling_10_winrate"),
        "diff_rolling_5_winrate": d("rolling_5_winrate"),
        "diff_streak": d("streak"),
        "diff_h2h_wr_prior": d("h2h_wr_prior"),
    }
    if sa["rank_missing"] or sb["rank_missing"]:
        feats["rank_diff"] = 0.0
    else:
        feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

    X = {c: feats.get(c, 0.0) for c in feature_cols}
    proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

    prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
    prob_b = 1.0 - prob_a
    winner = A if prob_a >= 0.5 else B
    loser  = B if winner == A else A
    conf = max(prob_a, prob_b)

    # deltas winner − loser (smoothed + post-aware)
    snap_w = sa if winner == A else sb
    snap_l = sb if winner == A else sa

    def delta(col: str) -> float:
        return float(snap_w[col] - snap_l[col])

    delta_rank = 0.0
    if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
        delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

    rows.append({
        "match_no": 1,
        "date": date_f.strftime("%Y-%m-%d"),
        "round": ROUND_F,
        "player_a": A,
        "player_b": B,
        "pred_winner": winner,
        "confidence": conf,
        "prob_player_a_win": prob_a,
        "prob_player_b_win": prob_b,

        "delta_elo": delta("pre_elo"),
        "delta_surface_elo": delta("pre_selo"),
        "delta_wr_Grass": delta("wr_Grass"),
        "delta_wr_Hard": delta("wr_Hard"),
        "delta_wr_Clay": delta("wr_Clay"),
        "delta_win_rate": delta("win_rate"),
        "delta_current_elo": delta("current_elo"),
        "delta_peak_elo": delta("peak_elo"),
        "delta_matches_28d": delta("matches_28d"),
        "delta_avg_rest_days": delta("avg_rest_days"),
        "delta_rolling_10_winrate": delta("rolling_10_winrate"),
        "delta_rolling_5_winrate": delta("rolling_5_winrate"),
        "delta_streak": delta("streak"),
        "delta_h2h_wr_prior": delta("h2h_wr_prior"),
        "delta_rank": delta_rank,

        "correct_prediction": np.nan,  # to be filled later
    })

    f_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Column order = R128 header
    r128_cols = list(pd.read_csv(R128_CSV, nrows=0).columns)
    for c in r128_cols:
        if c not in f_df.columns:
            f_df[c] = np.nan
    f_df = f_df[r128_cols].copy()

    _ensure_dir(OUT_CSV_F)
    f_df.to_csv(OUT_CSV_F, index=False, encoding="utf-8-sig")
    print(f"Saved Final predictions to: {OUT_CSV_F}")
    print(f_df.to_string(index=False))

if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

Saved Final predictions to: ./reports/wimbledon2025_F_predictions.csv
 match_no       date round      player_a       player_b   pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-07-13     F Jannik Sinner Carlos Alcaraz Jannik Sinner                 NaN    0.587093           0.587093           0.412907 138.404505         -48.714849       -0.102842       0.069361      -0.104252       -0.002612         138.404505       96.242176                0.0            -0.462105                       0.2                      0.0          13.0           -0.166667        -2.0


In [43]:
# filepath: ./combine_wimbledon_2025_predictions.py
"""
Combine all completed Wimbledon 2025 prediction CSVs into a single file.

- Preserves bracket order (R128→R64→R32→R16→QF→SF→F, then by match_no).
- Prefers '*_predictions_complete.csv' for each round; can optionally include non-complete files.
- Column layout follows the R128 header when available; any missing columns are added as NaN.
- By default includes all existing round files (complete or not). Use --only-complete to restrict.

Usage (CLI):
    python combine_wimbledon_2025_predictions.py
    python combine_wimbledon_2025_predictions.py --only-complete all
    python combine_wimbledon_2025_predictions.py --out ./reports/wimbledon2025_all_rounds_predictions_combined.csv

Notes:
- Safe in notebooks: uses parse_known_args() so it ignores extraneous argv.
"""

from __future__ import annotations

import argparse
import os
from typing import List, Dict

import pandas as pd

ROUNDS: List[str] = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]
ROUND_ORDER: Dict[str, int] = {r: i for i, r in enumerate(ROUNDS, start=1)}

# Candidate search locations and naming patterns
SEARCH_DIRS = ["./reports", "/mnt/data"]
NAME_PATTERNS = [
    "wimbledon2025_{round}_predictions_complete.csv",  # preferred
    "wimbledon2025_{round}_predictions.csv",           # fallback (if allowed)
]

DEFAULT_OUT = "./reports/wimbledon2025_all_rounds_predictions_combined.csv"


def find_round_file(round_code: str, allow_fallback: bool) -> str | None:
    for d in SEARCH_DIRS:
        preferred = os.path.join(d, NAME_PATTERNS[0].format(round=round_code))
        if os.path.exists(preferred):
            return preferred
    if not allow_fallback:
        return None
    for d in SEARCH_DIRS:
        fallback = os.path.join(d, NAME_PATTERNS[1].format(round=round_code))
        if os.path.exists(fallback):
            return fallback
    return None


def load_round_df(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "round" not in df.columns:
        raise ValueError(f"'round' column missing in {path}")
    if "match_no" not in df.columns:
        raise ValueError(f"'match_no' column missing in {path}")
    return df


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=DEFAULT_OUT, help="Output CSV path")
    ap.add_argument(
        "--only-complete",
        choices=["off", "any", "all"],
        default="off",
        help=(
            "Filter by 'correct_prediction' completeness per file: "
            "'off' = include any file found; "
            "'any' = include file if it has the 'correct_prediction' column and at least one non-null; "
            "'all' = include file only if all rows have non-null 'correct_prediction'."
        ),
    )
    # Safe for notebooks: ignore unknown argv rather than exiting
    args, _unknown = ap.parse_known_args()

    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)

    # Discover files round-by-round
    files: List[str] = []
    for r in ROUNDS:
        p = find_round_file(r, allow_fallback=(args.only_complete == "off"))
        if p:
            files.append(p)

    if not files:
        raise FileNotFoundError("No round prediction CSVs found in ./reports or /mnt/data.")

    print("Including files (in round order):")
    for p in files:
        print("  -", p)

    # Determine reference header from R128 if present; otherwise the widest union we see
    ref_header: List[str] | None = None
    r128_pref = find_round_file("R128", allow_fallback=(args.only_complete == "off"))
    if r128_pref:
        ref_header = list(pd.read_csv(r128_pref, nrows=0).columns)

    round_frames: List[pd.DataFrame] = []
    union_cols: set[str] = set(ref_header or [])

    # First pass: load & optionally filter by 'correct_prediction'
    for path in files:
        try:
            df = load_round_df(path)
        except Exception as e:
            print(f"Skip {path}: {e}")
            continue

        # filter by completeness if requested
        if args.only_complete != "off":
            if "correct_prediction" not in df.columns:
                print(f"Skip {path}: missing 'correct_prediction' for --only-complete={args.only_complete}")
                continue
            if args.only_complete == "any":
                if not df["correct_prediction"].notna().any():
                    print(f"Skip {path}: no non-null 'correct_prediction'")
                    continue
            if args.only_complete == "all":
                if df["correct_prediction"].isna().any():
                    print(f"Skip {path}: some rows missing 'correct_prediction'")
                    continue

        union_cols.update(df.columns)
        round_frames.append(df)

    if not round_frames:
        raise RuntimeError("No qualifying round files after filtering.")

    # Align columns to reference header if available; else use union (sorted with important columns first)
    if ref_header is None:
        preferred = [
            "match_no", "date", "round", "player_a", "player_b",
            "pred_winner", "confidence", "prob_player_a_win", "prob_player_b_win",
        ]
        delta_like = sorted([c for c in union_cols if c.startswith("delta_")])
        rest = [c for c in sorted(union_cols) if c not in preferred + delta_like]
        ref_header = preferred + delta_like + rest

    # Second pass: add missing columns and reorder
    aligned: List[pd.DataFrame] = []
    for df in round_frames:
        for col in ref_header:
            if col not in df.columns:
                df[col] = pd.NA
        aligned.append(df[ref_header].copy())

    # Concatenate and sort by round order + match_no (preserve bracket order per round)
    big = pd.concat(aligned, ignore_index=True)
    big["__round_order__"] = big["round"].map(ROUND_ORDER).fillna(9999).astype(int)

    # Ensure match_no numeric for sorting; keep original text for output
    big["__match_no__"] = pd.to_numeric(big["match_no"], errors="coerce")

    big = big.sort_values(["__round_order__", "__match_no__"], kind="mergesort").reset_index(drop=True)
    big = big.drop(columns=["__round_order__", "__match_no__"])

    # Write out
    big.to_csv(args.out, index=False, encoding="utf-8-sig")
    print(f"\nCombined predictions saved to: {args.out}")
    with pd.option_context("display.max_columns", None, "display.width", 200):
        print(big.head(12).to_string(index=False))


if __name__ == "__main__":
    main()


Including files (in round order):
  - ./reports\wimbledon2025_R128_predictions_complete.csv
  - ./reports\wimbledon2025_R64_predictions_complete.csv
  - ./reports\wimbledon2025_R32_predictions_complete.csv
  - ./reports\wimbledon2025_R16_predictions_complete.csv
  - ./reports\wimbledon2025_QF_predictions_complete.csv
  - ./reports\wimbledon2025_SF_predictions_complete.csv
  - ./reports\wimbledon2025_F_predictions_complete.csv

Combined predictions saved to: ./reports/wimbledon2025_all_rounds_predictions_combined.csv
 match_no     date round             player_a           player_b        pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 7/1/2025  R128        Jannik Sinner         Luca N

In [46]:
# filepath: ./prepare_after_wimbledon_and_predict_canada.py
"""
Post-Wimbledon refresh + Canada Masters R64 predictions (deltas-only, bracket order).

Outputs:
- ./data/match_dataset_post_wimbledon_2025.csv        (new dataset; original untouched)
- ./reports/player_profiles_post_wimbledon_2025.csv   (updated player profiles)
- ./reports/canada2025_R64_predictions.csv            (predictions in Wimbledon format)

Requires:
- Base: atp_matches_2021_2024.csv
- Model: ./models/rf_model.joblib
- Wimbledon results: wimbledon2025_{R128,R64,R32,R16,QF,SF,F}_predictions_complete.csv (as available)
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline


# ----------------------------
# Paths & constants
# ----------------------------
BASE_MATCHES_PATHS = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]

# Wimbledon completed prediction CSVs (include any you have)
WIM_ROUNDS = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]
WIM_SEARCH_DIRS = ["./reports", "/mnt/data"]
def _wim_path(r: str) -> List[str]:
    return [os.path.join(d, f"wimbledon2025_{r}_predictions_complete.csv") for d in WIM_SEARCH_DIRS]

# Use R128 header to align prediction outputs consistently
R128_HEADER_CANDIDATES = [os.path.join(d, "wimbledon2025_R128_predictions_complete.csv") for d in WIM_SEARCH_DIRS]

# New dataset & outputs (will be created)
OUT_MATCH_DATASET = "./data/match_dataset_post_wimbledon_2025.csv"
OUT_PLAYER_PROFILES = "./reports/player_profiles_post_wimbledon_2025.csv"
OUT_CANADA_R64 = "./reports/canada2025_R64_predictions.csv"

# Canada Masters config
CANADA_NAME = "Canada Masters 2025"
CANADA_SURFACE = "Hard"
CANADA_LEVEL = "M"   # Masters 1000
CANADA_ROUND = "R64"
CANADA_BEST_OF = 3
CANADA_AS_OF_DATE = "2025-07-29"  # tweak if needed

# Rank handling
RANK_CLIP = 500

# Beta smoothing for WRs
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0


# ----------------------------
# Utilities
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"


# ----------------------------
# Actual winner from completed prediction CSVs
# ----------------------------
def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    # Fall back to model pick if no correctness label
    return row["pred_winner"]

def load_round_df_any(paths: List[str]) -> pd.DataFrame | None:
    for p in paths:
        if os.path.exists(p):
            df = pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
            return df
    return None

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str, tourney_name: str) -> pd.DataFrame:
    if round_df is None or round_df.empty:
        return base
    rows = []
    for _, r in round_df.iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tourney_name,
            "surface": r.get("surface", "Grass"),  # Wimbledon = Grass
            "tourney_level": r.get("tourney_level", "G"),
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": int(r.get("best_of", 5)),
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    # Keep union of base columns, add any missing essentials
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"] if c not in common]
    out = pd.concat([base, extra[common + add]], ignore_index=True, sort=False)
    return out


# ----------------------------
# Elo log + smoothed WRs
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K * (1 - exp_w)
        new_ew_l = ew_l - K * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"] = log["opp"].map(canon_name)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form & streak
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(wins: pd.Series, matches: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + wins) / (SMOOTH_ALPHA + SMOOTH_BETA + matches)

    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    wr_prior_sm = smooth(wins_prior, matches_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  matches_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # peaks & rank history
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)

    return log


# ----------------------------
# Profiles
# ----------------------------
def compute_player_profiles_from_log(log: pd.DataFrame) -> pd.DataFrame:
    # take the last observed (post-aware) snapshot per player
    last_idx = log.groupby("player_key")["date"].idxmax()
    last = log.loc[last_idx].copy()

    out = pd.DataFrame({
        "player": last["player"].values,
        "last_date": last["date"].values,
        "current_elo": last["post_elo"].values,
        "peak_elo": last["peak_elo_post"].values,
        "win_rate_sm": last["win_rate_post_sm"].values,
        "wr_Clay_sm": last["wr_Clay_post_sm"].values,
        "wr_Grass_sm": last["wr_Grass_post_sm"].values,
        "wr_Hard_sm": last["wr_Hard_post_sm"].values,
        "avg_rest_days": last["avg_rest_days_post"].values if "avg_rest_days_post" in last else np.nan,
        "matches_28d": last["matches_28d_post"].values if "matches_28d_post" in last else np.nan,
        "streak": last["streak_post"].values,
        "n_matches": log.groupby("player_key")["player"].count().reindex(last["player_key"]).values,
    })
    return out.sort_values(["current_elo","win_rate_sm"], ascending=[False, False]).reset_index(drop=True)


# ----------------------------
# Snapshots for prediction
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = _safe_date(CANADA_AS_OF_DATE) > last["date"]  # generic check

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if "avg_rest_days_post" in last and after_last else last.get("avg_rest_days_prior", 20.0)
        m28d = last["matches_28d_post"] if "matches_28d_post" in last and after_last else last.get("matches_28d_prior", 0.0)
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            if target_surface == "Grass":
                snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
            elif target_surface == "Hard":
                snap["wr_Hard"]  = float(last_s["wr_Hard_post_sm"]  if after_last else last_s["wr_Hard_prior_sm"])
            elif target_surface == "Clay":
                snap["wr_Clay"]  = float(last_s["wr_Clay_post_sm"]  if after_last else last_s["wr_Clay_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])

        # fill other WRs from last row (post/prior)
        snap["wr_Hard"] = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"] = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])
        snap["wr_Grass"]= float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    # H2H prior to as-of
    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap


# ----------------------------
# Canada R64 pairings (exact order)
# ----------------------------
CANADA_R64_PAIRS: List[Tuple[str,str]] = [
    ("Alexander Zverev","Adam Walton"),
    ("Tristan Schoolkate","Matteo Arnaldi"),
    ("Tallon Griekspoor","Tomas Martin Etcheverry"),
    ("Jaume Munar","Francisco Cerundolo"),
    ("Daniil Medvedev","Dalibor Svrcina"),
    ("Nicolas Arseneault","Alexei Popyrin"),
    ("Alexandre Muller","Miomir Kecmanovic"),
    ("Giovanni Mpetshi Perricard","Holger Rune"),
    ("Lorenzo Musetti","James Duckworth"),
    ("Tomas Barrios Vera","Alex Michelsen"),
    ("Denis Shapovalov","Learner Tien"),
    ("Reilly Opelka","Tomas Machac"),
    ("Karen Khachanov","Juan Pablo Ficovich"),
    ("Emilio Nava","Ugo Humbert"),
    ("Nuno Borges","Facundo Bagnis"),
    ("Roman Safiullin","Casper Ruud"),
    ("Frances Tiafoe","Yosuke Watanuki"),
    ("Aleksandar Vukic","Cameron Norrie"),
    ("Stefanos Tsitsipas","Christopher O'Connell"),
    ("Francisco Comesana","Alex de Minaur"),
    ("Flavio Cobolli","Alexis Galarneau"),
    ("Fabian Marozsan","Felix Auger-Aliassime"),
    ("Brandon Nakashima","Ethan Quinn"),
    ("Adrian Mannarino","Ben Shelton"),
    ("Andrey Rublev","Hugo Gaston"),
    ("Bu Yunchaokete","Lorenzo Sonego"),
    ("Alejandro Davidovich Fokina","Corentin Moutet"),
    ("Tristan Boyer","Jakub Mensik"),
    ("Arthur Fils","Pablo Carreno Busta"),
    ("Mackenzie McDonald","Jiri Lehecka"),
    ("Gabriel Diallo","Matteo Gigante"),
    ("Roberto Carballes Baena","Taylor Fritz"),
]


# ----------------------------
# Run
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_MATCH_DATASET)
    _ensure_dir(OUT_PLAYER_PROFILES)
    _ensure_dir(OUT_CANADA_R64)

    # 1) Load base dataset (read-only)
    base_path = _first_existing(BASE_MATCHES_PATHS)
    base_matches = pd.read_csv(base_path)

    # 2) Append Wimbledon 2025 actual results (only the files you have)
    wim_all = base_matches.copy()
    for rnd in WIM_ROUNDS:
        df = load_round_df_any(_wim_path(rnd))
        if df is not None:
            wim_all = augment_with_results(wim_all, df, rnd, tourney_name="Wimbledon")
        else:
            print(f"[INFO] Wimbledon round {rnd} file not found; skipping.")

    # De-duplicate safety (should be no overlap with 2021-2024)
    if {"tourney_date","tourney_name","match_num","winner_name","loser_name"}.issubset(wim_all.columns):
        wim_all = wim_all.drop_duplicates(subset=["tourney_date","tourney_name","match_num","winner_name","loser_name"])

    # 3) Save new dataset (original untouched)
    wim_all.to_csv(OUT_MATCH_DATASET, index=False, encoding="utf-8-sig")
    print(f"[OK] Wrote new dataset: {OUT_MATCH_DATASET}  (rows={len(wim_all)})")

    # 4) Rebuild player log & profiles from new dataset
    log = build_player_log(wim_all, EloParams())
    profiles = compute_player_profiles_from_log(log)
    profiles.to_csv(OUT_PLAYER_PROFILES, index=False, encoding="utf-8-sig")
    print(f"[OK] Wrote updated player profiles: {OUT_PLAYER_PROFILES}")

    # 5) Load trained RF
    bundle = load(_first_existing(MODEL_PATHS))
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # 6) Predict Canada Masters R64
    asof = _safe_date(CANADA_AS_OF_DATE)
    pairs = CANADA_R64_PAIRS

    rows = []
    for i, (A, B) in enumerate(pairs, start=1):
        sa = snapshot_asof(log, A, B, asof, CANADA_SURFACE)
        sb = snapshot_asof(log, B, A, asof, CANADA_SURFACE)

        # model orientation (alphabetical)
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": CANADA_SURFACE, "tourney_level": CANADA_LEVEL, "round": CANADA_ROUND, "best_of": CANADA_BEST_OF,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # robust rank
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        # map back to bracket order
        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # winner − loser deltas
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": asof.strftime("%Y-%m-%d"),
            "round": CANADA_ROUND,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,  # to fill later
        })

    canada_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Align columns to Wimbledon R128 header if present
    header = None
    for cand in R128_HEADER_CANDIDATES:
        if os.path.exists(cand):
            header = list(pd.read_csv(cand, nrows=0).columns)
            break
    if header is None:
        # sensible default order
        header = [
            "match_no","date","round","player_a","player_b","pred_winner","confidence",
            "prob_player_a_win","prob_player_b_win",
            "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
            "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
            "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
            "correct_prediction",
        ]
    for c in header:
        if c not in canada_df.columns:
            canada_df[c] = pd.NA
    canada_df = canada_df[header].copy()

    canada_df.to_csv(OUT_CANADA_R64, index=False, encoding="utf-8-sig")
    print(f"[OK] Wrote Canada R64 predictions: {OUT_CANADA_R64}")
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(canada_df.head(8).to_string(index=False))


if __name__ == "__main__":
    main()


[OK] Wrote new dataset: ./data/match_dataset_post_wimbledon_2025.csv  (rows=11839)
[OK] Wrote updated player profiles: ./reports/player_profiles_post_wimbledon_2025.csv


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] Wrote Canada R64 predictions: ./reports/canada2025_R64_predictions.csv
 match_no       date round                   player_a                player_b         pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-07-29   R64           Alexander Zverev             Adam Walton    Alexander Zverev                 NaN    0.850939           0.850939           0.149061 445.878699         359.063461        0.158333       0.400877       0.290738        0.411544         445.878699      404.808039                0.0           -17.538889                  0.600000                      0.6           4.0            0.000000       -92.0
        2 2025-07-29   R64         Tristan Schoolkate       

In [47]:
# filepath: ./predict_canada_2025_r32.py
"""
Canada Masters 2025 - R32 predictions (deltas only, bracket order preserved).

Inputs:
- Model: ./models/rf_model.joblib
- Updated dataset (preferred): ./data/match_dataset_post_wimbledon_2025.csv
  (fallback: atp_matches_2021_2024.csv + injected Wimbledon results)
- Canada R64 predictions: canada2025_R64_predictions_complete.csv (preferred) or canada2025_R64_predictions.csv

Output:
- ./reports/canada2025_R32_predictions.csv
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
BASE_POST_WIM = "./data/match_dataset_post_wimbledon_2025.csv"
BASE_MATCHES_PATHS = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

WIM_ROUNDS = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]
SEARCH_DIRS = ["./reports", "/mnt/data"]
def _wim_path(r: str) -> List[str]:
    return [os.path.join(d, f"wimbledon2025_{r}_predictions_complete.csv") for d in SEARCH_DIRS]

CAN_R64_CANDIDATES = [os.path.join(d, "canada2025_R64_predictions_complete.csv") for d in SEARCH_DIRS] + \
                     [os.path.join(d, "canada2025_R64_predictions.csv") for d in SEARCH_DIRS]

OUT_R32 = "./reports/canada2025_R32_predictions.csv"

# Tournament metadata
TOURNEY_NAME_WIM = "Wimbledon"
TOURNEY_NAME_CAN = "Canada Masters 2025"
SURFACE_CAN = "Hard"
LEVEL_CAN = "M"
ROUND_R64 = "R64"
ROUND_R32 = "R32"
BEST_OF_CAN = 3

# Header alignment
R128_HEADER_CANDS = [os.path.join(d, "wimbledon2025_R128_predictions_complete.csv") for d in SEARCH_DIRS]

# Model feature hygiene
RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Helpers
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]  # fallback; why: lets you simulate if not yet labeled

def load_round_df_any(paths: List[str]) -> pd.DataFrame | None:
    for p in paths:
        if os.path.exists(p):
            return pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
    return None

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str, tname: str, surface: str, level: str, best_of: int) -> pd.DataFrame:
    if round_df is None or round_df.empty:
        return base
    rows = []
    for _, r in round_df.iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tname,
            "surface": surface,
            "tourney_level": level,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": best_of,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs = [(winners[i], winners[i+1]) for i in range(0, len(winners), 2)]
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)  # why: standard gap used earlier
    return pairs, d_prev, d_next

# ----------------------------
# Elo + smoothed WRs
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K * (1 - exp_w)
        new_ew_l = ew_l - K * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"] = log["opp"].map(canon_name)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form & streak
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(wins: pd.Series, matches: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + wins) / (SMOOTH_ALPHA + SMOOTH_BETA + matches)

    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    wr_prior_sm = smooth(wins_prior, matches_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  matches_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # peaks & rank history
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)

    return log

def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last["matches_28d_post"] if "matches_28d_post" in last and after_last else last.get("matches_28d_prior", 0.0)
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            if target_surface == "Hard":
                snap["wr_Hard"] = float(last_s["wr_Hard_post_sm"] if after_last else last_s["wr_Hard_prior_sm"])
            elif target_surface == "Grass":
                snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
            elif target_surface == "Clay":
                snap["wr_Clay"] = float(last_s["wr_Clay_post_sm"] if after_last else last_s["wr_Clay_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])

        # fill remaining WRs from last row
        snap["wr_Hard"] = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"] = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])
        snap["wr_Grass"]= float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_R32)

    # model
    bundle = load(_first_existing(MODEL_PATHS))
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # dataset (prefer updated post-Wimbledon)
    if os.path.exists(BASE_POST_WIM):
        matches = pd.read_csv(BASE_POST_WIM)
    else:
        # fall back: base + all Wimbledon rounds found
        base_path = _first_existing(BASE_MATCHES_PATHS)
        base = pd.read_csv(base_path)
        for rnd in WIM_ROUNDS:
            df = load_round_df_any(_wim_path(rnd))
            if df is not None:
                base = augment_with_results(base, df, rnd, TOURNEY_NAME_WIM, "Grass", "G", 5)
        matches = base

    # Canada R64 file
    r64 = load_round_df_any(CAN_R64_CANDIDATES)
    if r64 is None:
        raise FileNotFoundError("Canada R64 predictions file not found (complete or plain).")

    # R32 bracket & dates
    r32_pairs, date_r64, date_r32 = derive_next_pairs(r64)

    # Inject Canada R64 results so R32 uses post-R64 state
    matches_aug = augment_with_results(matches, r64, ROUND_R64, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)

    # Build player log with smoothed WRs
    log = build_player_log(matches_aug, EloParams())

    # Predict R32
    rows = []
    for i, (A, B) in enumerate(r32_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_r32, SURFACE_CAN)
        sb = snapshot_asof(log, B, A, date_r32, SURFACE_CAN)

        # alphabetical orientation for model
        p1, _p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE_CAN, "tourney_level": LEVEL_CAN, "round": ROUND_R32, "best_of": BEST_OF_CAN,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # winner − loser deltas
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_r32.strftime("%Y-%m-%d"),
            "round": ROUND_R32,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,
        })

    r32_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Align to Wimbledon R128 header for consistency
    header = None
    for cand in R128_HEADER_CANDS:
        if os.path.exists(cand):
            header = list(pd.read_csv(cand, nrows=0).columns)
            break
    if header is None:
        header = [
            "match_no","date","round","player_a","player_b","pred_winner","confidence",
            "prob_player_a_win","prob_player_b_win",
            "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
            "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
            "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
            "correct_prediction",
        ]
    for c in header:
        if c not in r32_df.columns:
            r32_df[c] = pd.NA
    r32_df = r32_df[header].copy()

    _ensure_dir(OUT_R32)
    r32_df.to_csv(OUT_R32, index=False, encoding="utf-8-sig")
    print(f"[OK] Canada R32 predictions → {OUT_R32}")
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(r32_df.head(8).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] Canada R32 predictions → ./reports/canada2025_R32_predictions.csv
 match_no       date round                player_a            player_b             pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-07-31   R32        Alexander Zverev      Matteo Arnaldi        Alexander Zverev                 NaN    0.755557           0.755557           0.244443 265.343204         222.752381        0.225000       0.187929       0.210399        0.213211         265.343204      238.807650                0.0            -4.401642                       0.4                      0.4           0.0            0.000000       -36.0
        2 2025-07-31   R32 Tomas Martin Etcheverry Francisco Cerundolo 

c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

In [48]:
# filepath: ./predict_canada_2025_r16.py
"""
Canada Masters 2025 - R16 predictions (deltas only, bracket order preserved).

Inputs:
- Model: ./models/rf_model.joblib
- Updated dataset (preferred): ./data/match_dataset_post_wimbledon_2025.csv
  (fallback: atp_matches_2021_2024.csv + injected Wimbledon results)
- Canada R64 + R32 predictions (prefer '*_complete.csv'; fallback to plain files)

Output:
- ./reports/canada2025_R16_predictions.csv
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
BASE_POST_WIM = "./data/match_dataset_post_wimbledon_2025.csv"
BASE_MATCHES_PATHS = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

WIM_ROUNDS = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]
SEARCH_DIRS = ["./reports", "/mnt/data"]
def _wim_path(r: str) -> List[str]:
    return [os.path.join(d, f"wimbledon2025_{r}_predictions_complete.csv") for d in SEARCH_DIRS]

CAN_R64_CANDS = [os.path.join(d, "canada2025_R64_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_R64_predictions.csv") for d in SEARCH_DIRS]
CAN_R32_CANDS = [os.path.join(d, "canada2025_R32_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_R32_predictions.csv") for d in SEARCH_DIRS]

OUT_R16 = "./reports/canada2025_R16_predictions.csv"

TOURNEY_NAME_WIM = "Wimbledon"
TOURNEY_NAME_CAN = "Canada Masters 2025"
SURFACE_CAN = "Hard"
LEVEL_CAN = "M"
ROUND_R64 = "R64"
ROUND_R32 = "R32"
ROUND_R16 = "R16"
BEST_OF_CAN = 3

R128_HEADER_CANDS = [os.path.join(d, "wimbledon2025_R128_predictions_complete.csv") for d in SEARCH_DIRS]

RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]  # fallback

def load_round_df_any(paths: List[str]) -> pd.DataFrame | None:
    for p in paths:
        if os.path.exists(p):
            return pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
    return None

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str,
                         tname: str, surface: str, level: str, best_of: int) -> pd.DataFrame:
    if round_df is None or round_df.empty:
        return base
    rows = []
    for _, r in round_df.iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tname,
            "surface": surface,
            "tourney_level": level,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": best_of,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs = [(winners[i], winners[i+1]) for i in range(0, len(winners), 2)]
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)  # ensures post-round state is used
    return pairs, d_prev, d_next

# ----------------------------
# Elo + smoothed WRs
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K * (1 - exp_w)
        new_ew_l = ew_l - K * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"] = log["opp"].map(canon_name)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form & streak
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(wins: pd.Series, matches: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + wins) / (SMOOTH_ALPHA + SMOOTH_BETA + matches)

    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    wr_prior_sm = smooth(wins_prior, matches_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  matches_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # peaks & rank history
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)

    return log

def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last["matches_28d_post"] if "matches_28d_post" in last and after_last else last.get("matches_28d_prior", 0.0)
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            if target_surface == "Hard":
                snap["wr_Hard"] = float(last_s["wr_Hard_post_sm"] if after_last else last_s["wr_Hard_prior_sm"])
            elif target_surface == "Grass":
                snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
            else:
                snap["wr_Clay"] = float(last_s["wr_Clay_post_sm"] if after_last else last_s["wr_Clay_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])

        # fill remaining WRs from last row
        snap["wr_Hard"]  = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"]  = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])
        snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    # H2H prior to as-of
    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_R16)

    # model
    bundle = load(_first_existing(MODEL_PATHS))
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # dataset (prefer post-Wimbledon)
    if os.path.exists(BASE_POST_WIM):
        matches = pd.read_csv(BASE_POST_WIM)
    else:
        base = pd.read_csv(_first_existing(BASE_MATCHES_PATHS))
        for rnd in WIM_ROUNDS:
            df_w = load_round_df_any(_wim_path(rnd))
            if df_w is not None:
                base = augment_with_results(base, df_w, rnd, TOURNEY_NAME_WIM, "Grass", "G", 5)
        matches = base

    # Canada R64 & R32
    r64 = load_round_df_any(CAN_R64_CANDS)
    r32 = load_round_df_any(CAN_R32_CANDS)
    if r32 is None:
        raise FileNotFoundError("Canada R32 predictions file not found (complete or plain).")

    # R16 bracket & dates
    r16_pairs, date_r32, date_r16 = derive_next_pairs(r32)

    # Inject Canada R64 & R32 so R16 uses post-R32 state
    matches_aug = matches.copy()
    if r64 is not None:
        matches_aug = augment_with_results(matches_aug, r64, ROUND_R64, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
    matches_aug = augment_with_results(matches_aug, r32, ROUND_R32, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)

    # Build player log
    log = build_player_log(matches_aug, EloParams())

    # Predict R16
    rows = []
    for i, (A, B) in enumerate(r16_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_r16, SURFACE_CAN)
        sb = snapshot_asof(log, B, A, date_r16, SURFACE_CAN)

        # alphabetical orientation for model
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE_CAN, "tourney_level": LEVEL_CAN, "round": ROUND_R16, "best_of": BEST_OF_CAN,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # winner − loser deltas
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_r16.strftime("%Y-%m-%d"),
            "round": ROUND_R16,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,
        })

    r16_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Align to Wimbledon R128 header
    header = None
    for cand in R128_HEADER_CANDS:
        if os.path.exists(cand):
            header = list(pd.read_csv(cand, nrows=0).columns)
            break
    if header is None:
        header = [
            "match_no","date","round","player_a","player_b","pred_winner","confidence",
            "prob_player_a_win","prob_player_b_win",
            "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
            "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
            "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
            "correct_prediction",
        ]
    for c in header:
        if c not in r16_df.columns:
            r16_df[c] = pd.NA
    r16_df = r16_df[header].copy()

    _ensure_dir(OUT_R16)
    r16_df.to_csv(OUT_R16, index=False, encoding="utf-8-sig")
    print(f"[OK] Canada R16 predictions → {OUT_R16}")
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(r16_df.head(8).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] Canada R16 predictions → ./reports/canada2025_R16_predictions.csv
 match_no       date round         player_a                    player_b      pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-02   R16 Alexander Zverev         Francisco Cerundolo Alexander Zverev                 NaN    0.831837           0.831837           0.168163 219.842010         234.088890        0.163462       0.237022       0.152690        0.196020         219.842010      202.146713                0.0            -2.619895                       0.2                      0.0           0.0           -0.333333       -28.0
        2 2025-08-02   R16   Alexei Popyrin                 Holger Rune      Holger

c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

In [50]:
# filepath: ./predict_canada_2025_qf.py
"""
Canada Masters 2025 - QF predictions (deltas only, bracket order preserved).

Inputs:
- Model: ./models/rf_model.joblib
- Updated dataset (preferred): ./data/match_dataset_post_wimbledon_2025.csv
  (fallback: atp_matches_2021_2024.csv + injected Wimbledon results)
- Completed Canada rounds to-date: R64, R32, R16 (prefer '*_complete.csv'; fallback to plain)

Output:
- ./reports/canada2025_QF_predictions.csv
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
BASE_POST_WIM = "./data/match_dataset_post_wimbledon_2025.csv"
BASE_MATCHES_PATHS = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

WIM_ROUNDS = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]
SEARCH_DIRS = ["./reports", "/mnt/data"]
def _wim_path(r: str) -> List[str]:
    return [os.path.join(d, f"wimbledon2025_{r}_predictions_complete.csv") for d in SEARCH_DIRS]

CAN_R64_CANDS = [os.path.join(d, "canada2025_R64_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_R64_predictions.csv") for d in SEARCH_DIRS]
CAN_R32_CANDS = [os.path.join(d, "canada2025_R32_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_R32_predictions.csv") for d in SEARCH_DIRS]
CAN_R16_CANDS = [os.path.join(d, "canada2025_R16_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_R16_predictions.csv") for d in SEARCH_DIRS]

OUT_QF = "./reports/canada2025_QF_predictions.csv"

TOURNEY_NAME_WIM = "Wimbledon"
TOURNEY_NAME_CAN = "Canada Masters 2025"
SURFACE_CAN = "Hard"
LEVEL_CAN = "M"
ROUND_R64 = "R64"
ROUND_R32 = "R32"
ROUND_R16 = "R16"
ROUND_QF  = "QF"
BEST_OF_CAN = 3

R128_HEADER_CANDS = [os.path.join(d, "wimbledon2025_R128_predictions_complete.csv") for d in SEARCH_DIRS]

RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]  # fallback simulation

def load_round_df_any(paths: List[str]) -> pd.DataFrame | None:
    for p in paths:
        if os.path.exists(p):
            return pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
    return None

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str,
                         tname: str, surface: str, level: str, best_of: int) -> pd.DataFrame:
    if round_df is None or round_df.empty:  # why: allow partial availability
        return base
    rows = []
    for _, r in round_df.iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tname,
            "surface": surface,
            "tourney_level": level,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": best_of,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs = [(winners[i], winners[i+1]) for i in range(0, len(winners), 2)]
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)  # why: ensure post-round metrics are used
    return pairs, d_prev, d_next

# ----------------------------
# Elo + smoothed WRs
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[(w)] = new_ew_w; elo_overall[(l)] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"] = log["opp"].map(canon_name)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form & streak
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(wins: pd.Series, matches: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + wins) / (SMOOTH_ALPHA + SMOOTH_BETA + matches)

    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    wr_prior_sm = smooth(wins_prior, matches_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  matches_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # peaks & rank history
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)

    return log

def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last["matches_28d_post"] if "matches_28d_post" in last and after_last else last.get("matches_28d_prior", 0.0)
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            if target_surface == "Hard":
                snap["wr_Hard"] = float(last_s["wr_Hard_post_sm"] if after_last else last_s["wr_Hard_prior_sm"])
            elif target_surface == "Grass":
                snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
            else:
                snap["wr_Clay"] = float(last_s["wr_Clay_post_sm"] if after_last else last_s["wr_Clay_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])

        # fill remaining WRs from last row
        snap["wr_Hard"]  = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"]  = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])
        snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    # H2H prior to as-of
    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_QF)

    # model
    bundle = load(_first_existing(MODEL_PATHS))
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # dataset (prefer post-Wimbledon)
    if os.path.exists(BASE_POST_WIM):
        matches = pd.read_csv(BASE_POST_WIM)
    else:
        base = pd.read_csv(_first_existing(BASE_MATCHES_PATHS))
        for rnd in WIM_ROUNDS:
            df_w = load_round_df_any(_wim_path(rnd))
            if df_w is not None:
                base = augment_with_results(base, df_w, rnd, TOURNEY_NAME_WIM, "Grass", "G", 5)
        matches = base

    # Canada rounds to inject
    r64 = load_round_df_any(CAN_R64_CANDS)
    r32 = load_round_df_any(CAN_R32_CANDS)
    r16 = load_round_df_any(CAN_R16_CANDS)
    if r16 is None:
        raise FileNotFoundError("Canada R16 predictions file not found (complete or plain).")

    # QF bracket & dates
    qf_pairs, date_r16, date_qf = derive_next_pairs(r16)

    # Inject Canada R64 + R32 + R16 so QF uses post-R16 state
    matches_aug = matches.copy()
    if r64 is not None:
        matches_aug = augment_with_results(matches_aug, r64, ROUND_R64, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
    if r32 is not None:
        matches_aug = augment_with_results(matches_aug, r32, ROUND_R32, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
    matches_aug = augment_with_results(matches_aug, r16, ROUND_R16, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)

    # Build player log
    log = build_player_log(matches_aug, EloParams())

    # Predict QF
    rows = []
    for i, (A, B) in enumerate(qf_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_qf, SURFACE_CAN)
        sb = snapshot_asof(log, B, A, date_qf, SURFACE_CAN)

        # alphabetical orientation for model
        p1, _p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE_CAN, "tourney_level": LEVEL_CAN, "round": ROUND_QF, "best_of": BEST_OF_CAN,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # winner − loser deltas
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_qf.strftime("%Y-%m-%d"),
            "round": ROUND_QF,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,
        })

    qf_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Align to Wimbledon R128 header
    header = None
    for cand in R128_HEADER_CANDS:
        if os.path.exists(cand):
            header = list(pd.read_csv(cand, nrows=0).columns)
            break
    if header is None:
        header = [
            "match_no","date","round","player_a","player_b","pred_winner","confidence",
            "prob_player_a_win","prob_player_b_win",
            "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
            "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
            "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
            "correct_prediction",
        ]
    for c in header:
        if c not in qf_df.columns:
            qf_df[c] = pd.NA
    qf_df = qf_df[header].copy()

    _ensure_dir(OUT_QF)
    qf_df.to_csv(OUT_QF, index=False, encoding="utf-8-sig")
    print(f"[OK] Canada QF predictions → {OUT_QF}")
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(qf_df.head(8).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] Canada QF predictions → ./reports/canada2025_QF_predictions.csv
 match_no       date round         player_a        player_b      pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-04    QF Alexander Zverev  Alexei Popyrin Alexander Zverev                 NaN    0.729271           0.729271           0.270729 181.750252         161.124993        0.267857       0.196311       0.255487        0.242694         181.750252      206.044382                0.0            -4.109534                       0.2                      0.0           0.0            0.333333       -22.0
        2 2025-08-04    QF   Alex Michelsen Karen Khachanov  Karen Khachanov                 NaN    0.631671 

In [51]:
# filepath: ./predict_canada_2025_sf.py
"""
Canada Masters 2025 - SF predictions (deltas only, bracket order preserved).

Inputs:
- Model: ./models/rf_model.joblib
- Updated dataset (preferred): ./data/match_dataset_post_wimbledon_2025.csv
  (fallback: atp_matches_2021_2024.csv + injected Wimbledon results)
- Completed Canada rounds to date: R64, R32, R16, QF (prefer '*_complete.csv'; fallback to plain)

Output:
- ./reports/canada2025_SF_predictions.csv
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
BASE_POST_WIM = "./data/match_dataset_post_wimbledon_2025.csv"
BASE_MATCHES_PATHS = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

WIM_ROUNDS = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]
SEARCH_DIRS = ["./reports", "/mnt/data"]
def _wim_path(r: str) -> List[str]:
    return [os.path.join(d, f"wimbledon2025_{r}_predictions_complete.csv") for d in SEARCH_DIRS]

CAN_R64_CANDS = [os.path.join(d, "canada2025_R64_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_R64_predictions.csv") for d in SEARCH_DIRS]
CAN_R32_CANDS = [os.path.join(d, "canada2025_R32_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_R32_predictions.csv") for d in SEARCH_DIRS]
CAN_R16_CANDS = [os.path.join(d, "canada2025_R16_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_R16_predictions.csv") for d in SEARCH_DIRS]
CAN_QF_CANDS  = [os.path.join(d, "canada2025_QF_predictions_complete.csv")  for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_QF_predictions.csv")  for d in SEARCH_DIRS]

OUT_SF = "./reports/canada2025_SF_predictions.csv"

TOURNEY_NAME_WIM = "Wimbledon"
TOURNEY_NAME_CAN = "Canada Masters 2025"
SURFACE_CAN = "Hard"
LEVEL_CAN = "M"
ROUND_R64 = "R64"
ROUND_R32 = "R32"
ROUND_R16 = "R16"
ROUND_QF  = "QF"
ROUND_SF  = "SF"
BEST_OF_CAN = 3

R128_HEADER_CANDS = [os.path.join(d, "wimbledon2025_R128_predictions_complete.csv") for d in SEARCH_DIRS]

RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]  # why: lets simulate when not labeled

def load_round_df_any(paths: List[str]) -> pd.DataFrame | None:
    for p in paths:
        if os.path.exists(p):
            return pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
    return None

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str,
                         tname: str, surface: str, level: str, best_of: int) -> pd.DataFrame:
    if round_df is None or round_df.empty:
        return base
    rows = []
    for _, r in round_df.iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tname,
            "surface": surface,
            "tourney_level": level,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": best_of,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs = [(winners[i], winners[i+1]) for i in range(0, len(winners), 2)]
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)  # why: ensure post-round metrics are visible
    return pairs, d_prev, d_next

# ----------------------------
# Elo + smoothed WRs
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[(w)] = new_ew_w; elo_overall[(l)] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"]    = log["opp"].map(canon_name)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form & streak
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(wins: pd.Series, matches: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + wins) / (SMOOTH_ALPHA + SMOOTH_BETA + matches)

    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    wr_prior_sm = smooth(wins_prior, matches_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  matches_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # peaks & rank history
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)

    return log

def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last["matches_28d_post"] if "matches_28d_post" in last and after_last else last.get("matches_28d_prior", 0.0)
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            if target_surface == "Hard":
                snap["wr_Hard"] = float(last_s["wr_Hard_post_sm"] if after_last else last_s["wr_Hard_prior_sm"])
            elif target_surface == "Grass":
                snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
            else:
                snap["wr_Clay"]  = float(last_s["wr_Clay_post_sm"]  if after_last else last_s["wr_Clay_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])

        # fill remaining WRs from last row
        snap["wr_Hard"]  = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"]  = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])
        snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    # H2H prior to as-of
    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_SF)

    # model
    bundle = load(_first_existing(MODEL_PATHS))
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # dataset (prefer post-Wimbledon)
    if os.path.exists(BASE_POST_WIM):
        matches = pd.read_csv(BASE_POST_WIM)
    else:
        base = pd.read_csv(_first_existing(BASE_MATCHES_PATHS))
        for rnd in WIM_ROUNDS:
            df_w = load_round_df_any(_wim_path(rnd))
            if df_w is not None:
                base = augment_with_results(base, df_w, rnd, TOURNEY_NAME_WIM, "Grass", "G", 5)
        matches = base

    # Canada rounds to inject for state through QF
    r64 = load_round_df_any(CAN_R64_CANDS)
    r32 = load_round_df_any(CAN_R32_CANDS)
    r16 = load_round_df_any(CAN_R16_CANDS)
    qf  = load_round_df_any(CAN_QF_CANDS)
    if qf is None:
        raise FileNotFoundError("Canada QF predictions file not found (complete or plain).")

    # SF bracket & dates
    sf_pairs, date_qf, date_sf = derive_next_pairs(qf)

    # Inject Canada R64 + R32 + R16 + QF so SF uses post-QF state
    matches_aug = matches.copy()
    if r64 is not None:
        matches_aug = augment_with_results(matches_aug, r64, ROUND_R64, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
    if r32 is not None:
        matches_aug = augment_with_results(matches_aug, r32, ROUND_R32, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
    if r16 is not None:
        matches_aug = augment_with_results(matches_aug, r16, ROUND_R16, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
    matches_aug = augment_with_results(matches_aug, qf,  ROUND_QF,  TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)

    # Build player log
    log = build_player_log(matches_aug, EloParams())

    # Predict SF
    rows = []
    for i, (A, B) in enumerate(sf_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_sf, SURFACE_CAN)
        sb = snapshot_asof(log, B, A, date_sf, SURFACE_CAN)

        # alphabetical orientation for model
        p1, _p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE_CAN, "tourney_level": LEVEL_CAN, "round": ROUND_SF, "best_of": BEST_OF_CAN,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # winner − loser deltas
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_sf.strftime("%Y-%m-%d"),
            "round": ROUND_SF,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,
        })

    sf_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Align to Wimbledon R128 header
    header = None
    for cand in R128_HEADER_CANDS:
        if os.path.exists(cand):
            header = list(pd.read_csv(cand, nrows=0).columns)
            break
    if header is None:
        header = [
            "match_no","date","round","player_a","player_b","pred_winner","confidence",
            "prob_player_a_win","prob_player_b_win",
            "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
            "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
            "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
            "correct_prediction",
        ]
    for c in header:
        if c not in sf_df.columns:
            sf_df[c] = pd.NA
    sf_df = sf_df[header].copy()

    _ensure_dir(OUT_SF)
    sf_df.to_csv(OUT_SF, index=False, encoding="utf-8-sig")
    print(f"[OK] Canada SF predictions → {OUT_SF}")
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(sf_df.to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] Canada SF predictions → ./reports/canada2025_SF_predictions.csv
 match_no       date round         player_a        player_b      pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-06    SF Alexander Zverev Karen Khachanov Alexander Zverev                 NaN    0.649814           0.649814           0.350186 108.914118         103.983909        0.049242       0.092768       0.160668        0.113585         108.914118      126.969495               -2.0            -1.442704                       0.0                      0.0           0.0            0.600000       -19.0
        2 2025-08-06    SF      Ben Shelton    Taylor Fritz     Taylor Fritz                 NaN    0.594577 

In [53]:
# filepath: ./predict_canada_2025_f.py
"""
Canada Masters 2025 - Final predictions (deltas only, bracket order preserved).

Inputs:
- Model: ./models/rf_model.joblib
- Updated dataset (preferred): ./data/match_dataset_post_wimbledon_2025.csv
  (fallback: atp_matches_2021_2024.csv + injected Wimbledon results)
- Completed Canada rounds to date: R64, R32, R16, QF, SF (prefer '*_complete.csv'; fallback to plain)

Output:
- ./reports/canada2025_F_predictions.csv
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
BASE_POST_WIM = "./data/match_dataset_post_wimbledon_2025.csv"
BASE_MATCHES_PATHS = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

WIM_ROUNDS = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]
SEARCH_DIRS = ["./reports", "/mnt/data"]
def _wim_path(r: str) -> List[str]:
    return [os.path.join(d, f"wimbledon2025_{r}_predictions_complete.csv") for d in SEARCH_DIRS]

# Canada files by round (prefer complete; fallback plain)
CAN_R64_CANDS = [os.path.join(d, "canada2025_R64_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_R64_predictions.csv") for d in SEARCH_DIRS]
CAN_R32_CANDS = [os.path.join(d, "canada2025_R32_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_R32_predictions.csv") for d in SEARCH_DIRS]
CAN_R16_CANDS = [os.path.join(d, "canada2025_R16_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_R16_predictions.csv") for d in SEARCH_DIRS]
CAN_QF_CANDS  = [os.path.join(d, "canada2025_QF_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_QF_predictions.csv") for d in SEARCH_DIRS]
CAN_SF_CANDS  = [os.path.join(d, "canada2025_SF_predictions_complete.csv") for d in SEARCH_DIRS] + \
                [os.path.join(d, "canada2025_SF_predictions.csv") for d in SEARCH_DIRS]

OUT_F = "./reports/canada2025_F_predictions.csv"

TOURNEY_NAME_WIM = "Wimbledon"
TOURNEY_NAME_CAN = "Canada Masters 2025"
SURFACE_CAN = "Hard"
LEVEL_CAN = "M"
ROUND_R64 = "R64"
ROUND_R32 = "R32"
ROUND_R16 = "R16"
ROUND_QF  = "QF"
ROUND_SF  = "SF"
ROUND_F   = "F"
BEST_OF_CAN = 3

# Column alignment (match Wimbledon R128 header if available)
R128_HEADER_CANDS = [os.path.join(d, "wimbledon2025_R128_predictions_complete.csv") for d in SEARCH_DIRS]

# Feature hygiene
RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str:
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]  # fallback when unlabeled

def load_round_df_any(paths: List[str]) -> pd.DataFrame | None:
    for p in paths:
        if os.path.exists(p):
            return pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
    return None

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str,
                         tname: str, surface: str, level: str, best_of: int) -> pd.DataFrame:
    if round_df is None or round_df.empty:
        return base
    rows = []
    for _, r in round_df.iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tname,
            "surface": surface,
            "tourney_level": level,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": best_of,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs = [(winners[i], winners[i+1]) for i in range(0, len(winners), 2)]
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)  # lets post-round state settle for snapshots
    return pairs, d_prev, d_next

# ----------------------------
# Elo + smoothed WRs
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[(w)] = new_ew_w; elo_overall[(l)] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)
    log["opp_key"]    = log["opp"].map(canon_name)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form & streak
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(wins: pd.Series, matches: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + wins) / (SMOOTH_ALPHA + SMOOTH_BETA + matches)

    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    wr_prior_sm = smooth(wins_prior, matches_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  matches_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # peaks & rank history
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)

    return log

def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon_name(player); ok = canon_name(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last["matches_28d_post"] if "matches_28d_post" in last and after_last else last.get("matches_28d_prior", 0.0)
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            if target_surface == "Hard":
                snap["wr_Hard"] = float(last_s["wr_Hard_post_sm"] if after_last else last_s["wr_Hard_prior_sm"])
            elif target_surface == "Grass":
                snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
            else:
                snap["wr_Clay"] = float(last_s["wr_Clay_post_sm"] if after_last else last_s["wr_Clay_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])

        # fill remaining WRs from last row
        snap["wr_Hard"]  = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"]  = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])
        snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    # H2H prior to as-of
    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_F)

    # model
    bundle = load(_first_existing(MODEL_PATHS))
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # dataset (prefer post-Wimbledon)
    if os.path.exists(BASE_POST_WIM):
        matches = pd.read_csv(BASE_POST_WIM)
    else:
        base = pd.read_csv(_first_existing(BASE_MATCHES_PATHS))
        for rnd in WIM_ROUNDS:
            df_w = load_round_df_any(_wim_path(rnd))
            if df_w is not None:
                base = augment_with_results(base, df_w, rnd, TOURNEY_NAME_WIM, "Grass", "G", 5)
        matches = base

    # Canada rounds to inject up to SF
    r64 = load_round_df_any(CAN_R64_CANDS)
    r32 = load_round_df_any(CAN_R32_CANDS)
    r16 = load_round_df_any(CAN_R16_CANDS)
    qf  = load_round_df_any(CAN_QF_CANDS)
    sf  = load_round_df_any(CAN_SF_CANDS)
    if sf is None:
        raise FileNotFoundError("Canada SF predictions file not found (complete or plain).")

    # Final bracket (one pairing) & dates
    f_pairs, date_sf, date_f = derive_next_pairs(sf)  # returns one pair for Finals

    # Inject Canada rounds so Finals uses post-SF state
    matches_aug = matches.copy()
    if r64 is not None:
        matches_aug = augment_with_results(matches_aug, r64, ROUND_R64, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
    if r32 is not None:
        matches_aug = augment_with_results(matches_aug, r32, ROUND_R32, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
    if r16 is not None:
        matches_aug = augment_with_results(matches_aug, r16, ROUND_R16, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
    if qf is not None:
        matches_aug = augment_with_results(matches_aug, qf,  ROUND_QF,  TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
    matches_aug = augment_with_results(matches_aug, sf,  ROUND_SF,  TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)

    # Build player log
    log = build_player_log(matches_aug, EloParams())

    # Predict Final
    rows = []
    # there will be exactly one match
    for i, (A, B) in enumerate(f_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_f, SURFACE_CAN)
        sb = snapshot_asof(log, B, A, date_f, SURFACE_CAN)

        # alphabetical orientation for model
        p1, _p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE_CAN, "tourney_level": LEVEL_CAN, "round": ROUND_F, "best_of": BEST_OF_CAN,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # robust rank
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # winner − loser deltas
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_f.strftime("%Y-%m-%d"),
            "round": ROUND_F,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,
        })

    f_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Align to Wimbledon R128 header for consistency
    header = None
    for cand in R128_HEADER_CANDS:
        if os.path.exists(cand):
            header = list(pd.read_csv(cand, nrows=0).columns)
            break
    if header is None:
        header = [
            "match_no","date","round","player_a","player_b","pred_winner","confidence",
            "prob_player_a_win","prob_player_b_win",
            "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
            "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
            "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
            "correct_prediction",
        ]
    for c in header:
        if c not in f_df.columns:
            f_df[c] = pd.NA
    f_df = f_df[header].copy()

    _ensure_dir(OUT_F)
    f_df.to_csv(OUT_F, index=False, encoding="utf-8-sig")
    print(f"[OK] Canada Final predictions → {OUT_F}")
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(f_df.to_string(index=False))


if __name__ == "__main__":
    main()


[OK] Canada Final predictions → ./reports/canada2025_F_predictions.csv
 match_no       date round        player_a    player_b pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-08     F Karen Khachanov Ben Shelton Ben Shelton                 NaN    0.570652           0.429348           0.570652 -24.481018         -32.523171       -0.040043      -0.016799      -0.100744       -0.035603         -24.481018      -24.481018                0.0             0.758653                       0.0                      0.0           0.0                 0.0         0.0


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

In [56]:
# filepath: ./combine_canada_2025_predictions.py
"""
Combine all *complete* Canada Masters 2025 prediction CSVs into a single file.

- Includes only files named: canada2025_{R64|R32|R16|QF|SF|F}_predictions_complete.csv
- Preserves bracket order: round rank (R64→R32→R16→QF→SF→F), then match_no (stable).
- Aligns columns to R64 header if available; otherwise builds a union with preferred columns first.
- Output: ./reports/canada2025_all_rounds_predictions_combined.csv
"""

from __future__ import annotations

import os
from typing import List, Dict
import pandas as pd

ROUNDS: List[str] = ["R64", "R32", "R16", "QF", "SF", "F"]
ROUND_ORDER: Dict[str, int] = {r: i for i, r in enumerate(ROUNDS, start=1)}

SEARCH_DIRS = ["./reports", "/mnt/data"]
NAME_PATTERN = "canada2025_{round}_predictions_complete.csv"
OUT_PATH = "./reports/canada2025_all_rounds_predictions_combined.csv"


def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)


def _find_round_file(round_code: str) -> str | None:
    fname = NAME_PATTERN.format(round=round_code)
    for d in SEARCH_DIRS:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return None


def _load_round_df(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "round" not in df.columns or "match_no" not in df.columns:
        raise ValueError(f"Missing required columns in {path} (need 'round' and 'match_no').")
    return df


def main() -> None:
    _ensure_dir(OUT_PATH)

    files: List[str] = []
    for r in ROUNDS:
        p = _find_round_file(r)
        if p:
            files.append(p)
        else:
            print(f"[warn] Missing complete file for {r}")

    if not files:
        raise FileNotFoundError("No complete Canada 2025 round CSVs found in ./reports or /mnt/data.")

    # Use R64 header if present; otherwise union of all columns
    r64_path = _find_round_file("R64")
    ref_header: List[str] | None = None
    if r64_path:
        ref_header = list(pd.read_csv(r64_path, nrows=0).columns)

    # Load all and build union
    frames: List[pd.DataFrame] = []
    union_cols: set[str] = set(ref_header or [])
    for p in files:
        try:
            df = _load_round_df(p)
        except Exception as e:
            print(f"[skip] {p}: {e}")
            continue
        union_cols.update(df.columns)
        frames.append(df)

    if not frames:
        raise RuntimeError("No valid round files loaded.")

    # Build header if we didn't have R64's
    if ref_header is None:
        preferred = [
            "match_no", "date", "round", "player_a", "player_b",
            "pred_winner", "confidence", "prob_player_a_win", "prob_player_b_win",
        ]
        delta_like = sorted([c for c in union_cols if c.startswith("delta_")])
        rest = [c for c in union_cols if c not in preferred + delta_like]
        # Keep 'correct_prediction' near the end if present
        if "correct_prediction" in rest:
            rest.remove("correct_prediction")
            rest.append("correct_prediction")
        ref_header = preferred + delta_like + sorted(rest)

    # Align and concat
    aligned: List[pd.DataFrame] = []
    for df in frames:
        for col in ref_header:
            if col not in df.columns:
                df[col] = pd.NA
        aligned.append(df[ref_header].copy())

    big = pd.concat(aligned, ignore_index=True)

    # Stable sort: round order then numeric match_no
    big["__round_rank__"] = big["round"].map(ROUND_ORDER).fillna(9999).astype(int)
    big["__m__"] = pd.to_numeric(big["match_no"], errors="coerce")

    big = big.sort_values(["__round_rank__", "__m__"], kind="mergesort").drop(columns=["__round_rank__", "__m__"]).reset_index(drop=True)

    big.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
    print(f"[ok] Combined Canada predictions → {OUT_PATH}")
    with pd.option_context("display.max_columns", None, "display.width", 200):
        print(big.head(12).to_string(index=False))


if __name__ == "__main__":
    main()


[ok] Combined Canada predictions → ./reports/canada2025_all_rounds_predictions_combined.csv
 match_no      date round                   player_a                player_b         pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 7/29/2025   R64           Alexander Zverev             Adam Walton    Alexander Zverev                   1    0.850939           0.850939           0.149061 445.878699         359.063461        0.158333       0.400877       0.290738        0.411544         445.878699      404.808039                  0           -17.538889                  0.600000                      0.6             4            0.000000         -92
        2 7/29/2025   R64         Tristan Scho

In [59]:
# filepath: ./update_logs_and_profiles_after_canada.py
"""
Update match logs and player profiles after injecting Canada Masters 2025 results.

Outputs:
- ./data/match_dataset_post_wimbledon_2025_canada.csv
- ./reports/logs/player_match_log_post_wimbledon_canada.csv (+ .parquet if engine available)
- ./reports/player_profiles_post_wimbledon_canada.csv

Base inputs (not modified):
- Preferred: ./data/match_dataset_post_wimbledon_2025.csv
- Fallback:  ./atp_matches_2021_2024.csv  (+ injected Wimbledon results if found under ./reports)

Canada results are read from: canada2025_{R64|R32|R16|QF|SF|F}_predictions_complete.csv
(looked up in ./reports and /mnt/data)
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

# ----------------------------
# Paths & constants
# ----------------------------
PREF_BASE = "./data/match_dataset_post_wimbledon_2025.csv"
FALLBACK_BASES = ["./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

SEARCH_DIRS = ["./reports", "/mnt/data"]

# Wimbledon predictions (if we need to construct post-Wimbledon base)
WIM_ROUNDS = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]

# Canada predictions (*_complete.csv only)
CAN_ROUNDS = ["R64", "R32", "R16", "QF", "SF", "F"]

OUT_MATCHES = "./data/match_dataset_post_wimbledon_2025_canada.csv"
OUT_LOGS_CSV = "./reports/logs/player_match_log_post_wimbledon_canada.csv"
OUT_LOGS_PARQUET = "./reports/logs/player_match_log_post_wimbledon_canada.parquet"
OUT_PROFILES = "./reports/player_profiles_post_wimbledon_canada.csv"

TOURNEY_NAME_WIM = "Wimbledon"
TOURNEY_NAME_CAN = "Canada Masters 2025"
SURFACE_WIM = "Grass"; LEVEL_WIM = "G"; BEST_OF_WIM = 5
SURFACE_CAN = "Hard";  LEVEL_CAN = "M"; BEST_OF_CAN = 3

# Elo & smoothing
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> str | None:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", (s or "")).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).strip().lower()
    return s

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def _find_pred_file(prefix: str, round_code: str) -> str | None:
    fname = f"{prefix}2025_{round_code}_predictions_complete.csv"
    for d in SEARCH_DIRS:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return None

def _load_round_df(path: str) -> pd.DataFrame:
    df = pd.read_csv(path).sort_values("match_no").reset_index(drop=True)
    need = ["date","match_no","player_a","player_b","pred_winner"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing '{c}' in {path}")
    return df

def _actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]

def _augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str,
                          tname: str, surface: str, level: str, best_of: int) -> pd.DataFrame:
    if round_df is None or round_df.empty:
        return base
    rows = []
    for _, r in round_df.iterrows():
        w = _actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tname,
            "surface": surface,
            "tourney_level": level,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": best_of,
            # ranks unknown here; logs handle missing gracefully
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    # Retain/append only columns known to base
    common = [c for c in extra.columns if c in base.columns]
    add_min = ["tourney_date","tourney_name","surface","tourney_level","match_num",
               "winner_name","loser_name","round","best_of"]
    add = [c for c in add_min if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

# ----------------------------
# Logs builder (pre/post Elo, smoothed WRs, workload, form)
# ----------------------------
def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num",
           "winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    BASE, K, KS = params.base, params.k_overall, params.k_surface
    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "post_elo": new_ew_w,
            "pre_selo": es_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "post_elo": new_ew_l,
            "pre_selo": es_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        # update state
        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon_name)

    # workload & rest
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7, 14, 28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    # form (prior/post) and streaks
    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # cumulative priors/post
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # overall smoothed WRs
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    matches_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    matches_post_ov  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)

    log["win_rate_prior_sm"] = smooth(wins_prior_ov, matches_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  matches_post_ov).clip(0,1)

    # per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # peaks & rank history
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)

    return log

# ----------------------------
# Profiles
# ----------------------------
def compute_player_profiles(log: pd.DataFrame) -> pd.DataFrame:
    log = log.sort_values(["player_key","date"], kind="mergesort")
    last_idx = log.groupby("player_key")["date"].idxmax()
    last_rows = log.loc[last_idx].copy()

    # per-surface latest post_selo
    surf_last = (log.sort_values(["player_key","surface","date"])
                   .groupby(["player_key","surface"])
                   .tail(1)[["player_key","surface","post_selo"]])
    surf_piv = surf_last.pivot(index="player_key", columns="surface", values="post_selo").rename(
        columns={"Clay":"selo_Clay","Grass":"selo_Grass","Hard":"selo_Hard"}
    )
    for c in ["selo_Clay","selo_Grass","selo_Hard"]:
        if c not in surf_piv.columns: surf_piv[c] = np.nan

    profiles = (last_rows
        .set_index("player_key")[[
            "player","date","post_elo","peak_elo_post",
            "win_rate_post_sm","wr_Clay_post_sm","wr_Grass_post_sm","wr_Hard_post_sm",
            "rolling_10_winrate_post","rolling_5_winrate_post","streak_post",
            "avg_rest_days_post","matches_28d_post"
        ]]
        .rename(columns={
            "player": "name",
            "date": "last_match_date",
            "post_elo": "current_elo",
            "peak_elo_post": "peak_elo",
            "win_rate_post_sm": "overall_wr",
            "wr_Clay_post_sm": "wr_Clay",
            "wr_Grass_post_sm": "wr_Grass",
            "wr_Hard_post_sm": "wr_Hard",
            "rolling_10_winrate_post": "form10_wr",
            "rolling_5_winrate_post": "form5_wr",
            "streak_post": "streak",
            "avg_rest_days_post": "avg_rest_days",
            "matches_28d_post": "matches_28d",
        }))

    profiles = profiles.join(surf_piv, how="left")
    for c in ["selo_Clay","selo_Grass","selo_Hard"]:
        profiles[c] = profiles[c].fillna(1500.0)

    # tidy
    profiles = profiles.reset_index().sort_values("name")
    # clip/clean
    for c in ["wr_Clay","wr_Grass","wr_Hard","overall_wr","form10_wr","form5_wr"]:
        profiles[c] = profiles[c].clip(0,1)
    profiles["avg_rest_days"] = profiles["avg_rest_days"].clip(0,60)

    return profiles[[
        "name","last_match_date","current_elo","peak_elo",
        "selo_Clay","selo_Grass","selo_Hard",
        "wr_Clay","wr_Grass","wr_Hard","overall_wr",
        "form10_wr","form5_wr","streak","avg_rest_days","matches_28d"
    ]]

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    # Base matches
    if os.path.exists(PREF_BASE):
        matches = pd.read_csv(PREF_BASE)
        print(f"[base] Loaded {PREF_BASE} ({len(matches):,} rows)")
    else:
        fallback = _first_existing(FALLBACK_BASES)
        if fallback is None:
            raise FileNotFoundError("No base matches found.")
        matches = pd.read_csv(fallback)
        print(f"[base] Loaded {fallback} ({len(matches):,} rows)")
        # inject Wimbledon if present
        for r in WIM_ROUNDS:
            p = _find_pred_file("wimbledon", r)
            if p:
                df = _load_round_df(p)
                matches = _augment_with_results(matches, df, r, TOURNEY_NAME_WIM, SURFACE_WIM, LEVEL_WIM, BEST_OF_WIM)
                print(f"[wim] +{len(df)} rows from {os.path.basename(p)}")

    # Inject Canada
    added = 0
    for r in CAN_ROUNDS:
        p = _find_pred_file("canada", r)
        if p:
            df = _load_round_df(p)
            before = len(matches)
            matches = _augment_with_results(matches, df, r, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
            delta = len(matches) - before
            added += delta
            print(f"[can] +{delta} rows from {os.path.basename(p)}")
        else:
            print(f"[can] No file for {r}")

    # Save updated dataset (post-Wimbledon + Canada)
    _ensure_dir(OUT_MATCHES)
    matches.to_csv(OUT_MATCHES, index=False, encoding="utf-8-sig")
    print(f"[ok] wrote {OUT_MATCHES} ({len(matches):,} rows)")

    # Build logs
    log = build_player_log(matches, EloParams())
    _ensure_dir(OUT_LOGS_CSV)
    log.to_csv(OUT_LOGS_CSV, index=False, encoding="utf-8-sig")
    print(f"[ok] wrote {OUT_LOGS_CSV} ({len(log):,} rows)")

    # Parquet (optional)
    try:
        log.to_parquet(OUT_LOGS_PARQUET, index=False)
        print(f"[ok] wrote {OUT_LOGS_PARQUET}")
    except Exception as e:
        print(f"[skip] parquet: {e}")

    # Profiles
    prof = compute_player_profiles(log)
    _ensure_dir(OUT_PROFILES)
    prof.to_csv(OUT_PROFILES, index=False, encoding="utf-8-sig")
    print(f"[ok] wrote {OUT_PROFILES} ({len(prof):,} players)")

    # Small preview
    with pd.option_context("display.max_columns", None, "display.width", 140):
        print("\n[preview] profiles:")
        print(prof.head(12).to_string(index=False))


if __name__ == "__main__":
    main()


[base] Loaded ./data/match_dataset_post_wimbledon_2025.csv (11,839 rows)
[can] +32 rows from canada2025_R64_predictions_complete.csv
[can] +16 rows from canada2025_R32_predictions_complete.csv
[can] +8 rows from canada2025_R16_predictions_complete.csv
[can] +4 rows from canada2025_QF_predictions_complete.csv
[can] +2 rows from canada2025_SF_predictions_complete.csv
[can] +1 rows from canada2025_F_predictions_complete.csv
[ok] wrote ./data/match_dataset_post_wimbledon_2025_canada.csv (11,902 rows)
[ok] wrote ./reports/logs/player_match_log_post_wimbledon_canada.csv (23,804 rows)
[skip] parquet: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for par

In [61]:
# filepath: ./predict_cincinnati_2025_r64.py
"""
Cincinnati (Western & Southern Open) 2025 - R64 predictions (deltas only, bracket order preserved).

Inputs:
- Model bundle: ./models/rf_model.joblib
- Preferred dataset: ./data/match_dataset_post_wimbledon_2025_canada.csv
  Fallback: atp_matches_2021_2024.csv (+ inject Wimbledon/Canada if *_complete.csv present)

Output:
- ./reports/cincinnati2025_R64_predictions.csv
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]

PREF_DATA = "./data/match_dataset_post_wimbledon_2025_canada.csv"
BASES = ["./data/match_dataset_post_wimbledon_2025.csv",
         "./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

SEARCH_DIRS = ["./reports", "/mnt/data"]
WIM_ROUNDS = ["R128","R64","R32","R16","QF","SF","F"]
CAN_ROUNDS = ["R64","R32","R16","QF","SF","F"]

TOURNEY_NAME_WIM = "Wimbledon"
SURFACE_WIM = "Grass"; LEVEL_WIM = "G"; BEST_OF_WIM = 5

TOURNEY_NAME_CAN = "Canada Masters 2025"
SURFACE_CAN = "Hard";  LEVEL_CAN = "M"; BEST_OF_CAN = 3

TOURNEY_NAME_CIN = "Cincinnati 2025"
SURFACE_CIN = "Hard"; LEVEL_CIN = "M"; BEST_OF_CIN = 3
ROUND_CIN = "R64"
DATE_CIN = "2025-08-13"  # adjust if necessary

OUT_CSV = "./reports/cincinnati2025_R64_predictions.csv"

# Prefer aligning to Canada R64, else Wimbledon R128 headers
R64_HEADER_CANDS  = [os.path.join(d, "canada2025_R64_predictions_complete.csv") for d in SEARCH_DIRS]
R128_HEADER_CANDS = [os.path.join(d, "wimbledon2025_R128_predictions_complete.csv") for d in SEARCH_DIRS]

# Feature hygiene
RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Cincinnati R64 pairs (exact order from user)
# ----------------------------
CIN_R64_PAIRS: List[Tuple[str, str]] = [
    ("Jannik Sinner", "Daniel Elahi Galan"),
    ("Sebastian Baez", "Gabriel Diallo"),
    ("Tomas Machac", "Adrian Mannarino"),
    ("Pedro Martinez", "Tommy Paul"),
    ("Casper Ruud", "Arthur Rinderknech"),
    ("Tomas Martin Etcheverry", "Felix Auger Aliassime"),
    ("Stefanos Tsitsipas", "Fabian Marozsan"),
    ("Benjamin Bonzi", "Lorenzo Musetti"),
    ("Taylor Fritz", "Emilio Nava"),
    ("Zizou Bergs", "Lorenzo Sonego"),
    ("Alejandro Davidovich Fokina", "Joao Fonseca"),
    ("Terence Atmane", "Flavio Cobolli"),
    ("Frances Tiafoe", "Roberto Carballes Baena"),
    ("Coleman Wong", "Ugo Humbert"),
    ("Alex Michelsen", "Corentin Moutet"),
    ("Roman Safiullin", "Holger Rune"),
    ("Ben Shelton", "Camilo Ugo Carabelli"),
    ("Roberto Bautista Agut", "Cameron Norrie"),
    ("Jiri Lehecka", "Tristan Boyer"),
    ("Adam Walton", "Daniil Medvedev"),
    ("Karen Khachanov", "Valentin Royer"),
    ("Jenson Brooksby", "Arthur Cazaux"),
    ("Brandon Nakashima", "Alexander Blockx"),
    ("Nishesh Basavareddy", "Alexander Zverev"),
    ("Alex De Minaur", "Reilly Opelka"),
    ("Francisco Comesana", "Luciano Darderi"),
    ("Alexei Popyrin", "Martin Landaluce"),
    ("Learner Tien", "Andrey Rublev"),
    ("Jakub Mensik", "Ethan Quinn"),
    ("Luca Nardi", "Denis Shapovalov"),
    ("Tallon Griekspoor", "Hamad Medjedovic"),
    ("Damir Dzumhur", "Carlos Alcaraz"),
]

# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def find_pred_file(prefix: str, round_code: str, complete_only=True) -> Optional[str]:
    names = [f"{prefix}2025_{round_code}_predictions_complete.csv"] if complete_only \
            else [f"{prefix}2025_{round_code}_predictions_complete.csv", f"{prefix}2025_{round_code}_predictions.csv"]
    for d in SEARCH_DIRS:
        for nm in names:
            p = os.path.join(d, nm)
            if os.path.exists(p):
                return p
    return None

def load_round_df(p: str) -> pd.DataFrame:
    df = pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
    need = ["date","match_no","player_a","player_b","pred_winner"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing '{c}' in {p}")
    return df

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str,
                         tname: str, surface: str, level: str, best_of: int) -> pd.DataFrame:
    if round_df is None or round_df.empty:
        return base
    rows = []
    for _, r in round_df.iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tname,
            "surface": surface,
            "tourney_level": level,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": best_of,
            "winner_rank": np.nan, "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

# ----------------------------
# Elo & logs
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon)
    log["opp_key"]    = log["opp"].map(canon)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)

    log["win_rate_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # per-surface WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)

    return log

# ----------------------------
# Snapshot & prediction
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]
        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last["matches_28d_post"] if "matches_28d_post" in last and after_last else last.get("matches_28d_prior", 0.0)
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            if target_surface == "Hard":
                snap["wr_Hard"] = float(last_s["wr_Hard_post_sm"] if after_last else last_s["wr_Hard_prior_sm"])
            elif target_surface == "Grass":
                snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
            else:
                snap["wr_Clay"]  = float(last_s["wr_Clay_post_sm"]  if after_last else last_s["wr_Clay_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])

        snap["wr_Hard"]  = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"]  = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])
        snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_CSV)

    # Load model
    bundle_path = _first_existing(MODEL_PATHS)
    if not bundle_path:
        raise FileNotFoundError("Model bundle not found under ./models or /mnt/data/models.")
    bundle = load(bundle_path)
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # Load base matches (prefer post-Canada)
    if os.path.exists(PREF_DATA):
        matches = pd.read_csv(PREF_DATA)
    else:
        base_path = _first_existing(BASES)
        if not base_path:
            raise FileNotFoundError("No base dataset found.")
        base = pd.read_csv(base_path)
        # Inject Wimbledon & Canada if *_complete.csv exist
        for r in WIM_ROUNDS:
            p = find_pred_file("wimbledon", r, complete_only=True)
            if p:
                base = augment_with_results(base, load_round_df(p), r, TOURNEY_NAME_WIM, SURFACE_WIM, LEVEL_WIM, BEST_OF_WIM)
        for r in CAN_ROUNDS:
            p = find_pred_file("canada", r, complete_only=True)
            if p:
                base = augment_with_results(base, load_round_df(p), r, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
        matches = base

    # Build player log
    log = build_player_log(matches, EloParams())
    asof = _safe_date(DATE_CIN)

    # Predict
    rows = []
    for i, (A, B) in enumerate(CIN_R64_PAIRS, start=1):
        sa = snapshot_asof(log, A, B, asof, SURFACE_CIN)
        sb = snapshot_asof(log, B, A, asof, SURFACE_CIN)

        # alphabetical orientation for model
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE_CIN, "tourney_level": LEVEL_CIN, "round": ROUND_CIN, "best_of": BEST_OF_CIN,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # winner − loser deltas
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": asof.strftime("%Y-%m-%d"),
            "round": ROUND_CIN,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,
        })

    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Column alignment: prefer Canada R64 header, else Wimbledon R128, else default
    header = None
    for cand in R64_HEADER_CANDS + R128_HEADER_CANDS:
        if os.path.exists(cand):
            header = list(pd.read_csv(cand, nrows=0).columns)
            break
    if header is None:
        header = [
            "match_no","date","round","player_a","player_b","pred_winner","confidence",
            "prob_player_a_win","prob_player_b_win",
            "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
            "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
            "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
            "correct_prediction",
        ]
    for c in header:
        if c not in out_df.columns:
            out_df[c] = pd.NA
    out_df = out_df[header].copy()

    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] Cincinnati R64 predictions → {OUT_CSV}  (matches: {len(out_df)})")
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(out_df.head(10).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] Cincinnati R64 predictions → ./reports/cincinnati2025_R64_predictions.csv  (matches: 32)
 match_no       date round                player_a              player_b           pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-13   R64           Jannik Sinner    Daniel Elahi Galan         Jannik Sinner                 NaN    0.854621           0.854621           0.145379 696.285710         553.596087        0.193582       0.393498       0.300310        0.379510         696.285710      559.450561                0.0            -7.236014                       0.7                      0.6          22.0            0.500000      -123.0
        2 2025-08-13   R64          Sebastian Ba

In [62]:
# filepath: ./predict_cincinnati_2025_r32.py
"""
Cincinnati (Western & Southern Open) 2025 - R32 predictions (deltas only, bracket order preserved).

Inputs:
- Model: ./models/rf_model.joblib
- Preferred dataset: ./data/match_dataset_post_wimbledon_2025_canada.csv
  (fallback: base + injected Wimbledon/Canada results if available)
- Cincinnati R64 predictions: prefer ./reports/cincinnati2025_R64_predictions_complete.csv, else ..._R64_predictions.csv

Output:
- ./reports/cincinnati2025_R32_predictions.csv
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]

PREF_DATA = "./data/match_dataset_post_wimbledon_2025_canada.csv"
BASES = ["./data/match_dataset_post_wimbledon_2025.csv",
         "./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

SEARCH_DIRS = ["./reports", "/mnt/data"]

# Prior events (only needed if PREF_DATA missing)
WIM_ROUNDS = ["R128","R64","R32","R16","QF","SF","F"]
CAN_ROUNDS = ["R64","R32","R16","QF","SF","F"]
TOURNEY_NAME_WIM = "Wimbledon"; SURFACE_WIM = "Grass"; LEVEL_WIM = "G"; BEST_OF_WIM = 5
TOURNEY_NAME_CAN = "Canada Masters 2025"; SURFACE_CAN = "Hard"; LEVEL_CAN = "M"; BEST_OF_CAN = 3

# Cincinnati constants
TOURNEY_NAME_CIN = "Cincinnati 2025"
SURFACE_CIN = "Hard"; LEVEL_CIN = "M"; BEST_OF_CIN = 3
ROUND_R64 = "R64"
ROUND_R32 = "R32"

# Output
OUT_CSV = "./reports/cincinnati2025_R32_predictions.csv"

# Header alignment preference
CIN_R64_HDR = [os.path.join(d, "cincinnati2025_R64_predictions_complete.csv") for d in SEARCH_DIRS] + \
              [os.path.join(d, "cincinnati2025_R64_predictions.csv") for d in SEARCH_DIRS]
CAN_R64_HDR = [os.path.join(d, "canada2025_R64_predictions_complete.csv") for d in SEARCH_DIRS]
WIM_R128_HDR = [os.path.join(d, "wimbledon2025_R128_predictions_complete.csv") for d in SEARCH_DIRS]

# Hygiene
RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def find_pred_file(prefix: str, round_code: str, complete_only=True) -> Optional[str]:
    names = [f"{prefix}2025_{round_code}_predictions_complete.csv"] if complete_only \
            else [f"{prefix}2025_{round_code}_predictions_complete.csv", f"{prefix}2025_{round_code}_predictions.csv"]
    for d in SEARCH_DIRS:
        for nm in names:
            p = os.path.join(d, nm)
            if os.path.exists(p):
                return p
    return None

def load_round_df(p: str) -> pd.DataFrame:
    df = pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
    need = ["date","match_no","player_a","player_b","pred_winner"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing '{c}' in {p}")
    return df

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str,
                         tname: str, surface: str, level: str, best_of: int) -> pd.DataFrame:
    if round_df is None or round_df.empty:
        return base
    rows = []
    for _, r in round_df.iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tname,
            "surface": surface,
            "tourney_level": level,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": best_of,
            "winner_rank": np.nan, "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs = [(winners[i], winners[i+1]) for i in range(0, len(winners), 2)]
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)  # why: ensures snapshots use post-round state
    return pairs, d_prev, d_next

# ----------------------------
# Elo/logs
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    rows: List[Dict] = []
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon)
    log["opp_key"]    = log["opp"].map(canon)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)

    log["win_rate_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # per-surface WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        for df in (surf_prior, surf_post):
            if srf not in df.columns: df[srf] = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_values(["player_key","date"]).reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)

    return log

# ----------------------------
# Snapshot & features
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last.get("matches_28d_post" if after_last else "matches_28d_prior", 0.0)
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            if target_surface == "Hard":
                snap["wr_Hard"] = float(last_s["wr_Hard_post_sm"] if after_last else last_s["wr_Hard_prior_sm"])
            elif target_surface == "Grass":
                snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
            else:
                snap["wr_Clay"]  = float(last_s["wr_Clay_post_sm"]  if after_last else last_s["wr_Clay_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])

        # fill remaining WRs
        snap["wr_Hard"]  = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"]  = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])
        snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_CSV)

    # model
    bundle = load(_first_existing(MODEL_PATHS) or MODEL_PATHS[0])
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # dataset
    if os.path.exists(PREF_DATA):
        matches = pd.read_csv(PREF_DATA)
    else:
        base_path = _first_existing(BASES)
        if not base_path:
            raise FileNotFoundError("No base dataset found.")
        base = pd.read_csv(base_path)
        # inject Wimbledon & Canada if present
        for r in WIM_ROUNDS:
            p = find_pred_file("wimbledon", r, complete_only=True)
            if p:
                base = augment_with_results(base, load_round_df(p), r, TOURNEY_NAME_WIM, SURFACE_WIM, LEVEL_WIM, BEST_OF_WIM)
        for r in CAN_ROUNDS:
            p = find_pred_file("canada", r, complete_only=True)
            if p:
                base = augment_with_results(base, load_round_df(p), r, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
        matches = base

    # Cincinnati R64 predictions (derive R32)
    cin_r64 = None
    for d in SEARCH_DIRS:
        for nm in ["cincinnati2025_R64_predictions_complete.csv", "cincinnati2025_R64_predictions.csv"]:
            p = os.path.join(d, nm)
            if os.path.exists(p):
                cin_r64 = pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
                break
        if cin_r64 is not None:
            break
    if cin_r64 is None or cin_r64.empty:
        raise FileNotFoundError("Cincinnati R64 predictions file not found in ./reports or /mnt/data.")

    r32_pairs, date_r64, date_r32 = derive_next_pairs(cin_r64)

    # Inject R64 so R32 uses post-R64 state
    matches_aug = augment_with_results(matches, cin_r64, ROUND_R64, TOURNEY_NAME_CIN, SURFACE_CIN, LEVEL_CIN, BEST_OF_CIN)

    # Build player log
    log = build_player_log(matches_aug, EloParams())

    # Predict R32
    rows = []
    for i, (A, B) in enumerate(r32_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_r32, SURFACE_CIN)
        sb = snapshot_asof(log, B, A, date_r32, SURFACE_CIN)

        # alphabetical orientation for model
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE_CIN, "tourney_level": LEVEL_CIN, "round": ROUND_R32, "best_of": BEST_OF_CIN,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        loser  = B if winner == A else A
        conf = max(prob_a, prob_b)

        # winner − loser deltas
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_r32.strftime("%Y-%m-%d"),
            "round": ROUND_R32,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": prob_b,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,
        })

    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Align columns to Cincinnati R64 header if available; else Canada R64; else Wimbledon R128; else default
    header = None
    for cand in CIN_R64_HDR + CAN_R64_HDR + WIM_R128_HDR:
        if os.path.exists(cand):
            header = list(pd.read_csv(cand, nrows=0).columns)
            break
    if header is None:
        header = [
            "match_no","date","round","player_a","player_b","pred_winner","confidence",
            "prob_player_a_win","prob_player_b_win",
            "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
            "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
            "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
            "correct_prediction",
        ]
    for c in header:
        if c not in out_df.columns:
            out_df[c] = pd.NA
    out_df = out_df[header].copy()

    _ensure_dir(OUT_CSV)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] Cincinnati R32 predictions → {OUT_CSV} (matches: {len(out_df)})")
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(out_df.head(10).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] Cincinnati R32 predictions → ./reports/cincinnati2025_R32_predictions.csv (matches: 16)
 match_no       date round           player_a              player_b           pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-15   R32      Jannik Sinner        Gabriel Diallo         Jannik Sinner                 NaN    0.822618           0.822618           0.177382 561.937899         464.553875        0.288820       0.301802       0.235595        0.320284         561.937899      561.937899               -2.0           -17.042176                       0.4                      0.4          21.0            0.000000       -85.0
        2 2025-08-15   R32   Adrian Mannarino            To

In [64]:
# filepath: ./predict_cincinnati_2025_r16.py
"""
Cincinnati (Western & Southern Open) 2025 - R16 predictions (deltas only, bracket order preserved).

Inputs:
- Model: ./models/rf_model.joblib
- Preferred dataset: ./data/match_dataset_post_wimbledon_2025_canada.csv
  (fallback: base + injected Wimbledon/Canada results if available)
- Cincinnati R64/R32 predictions: prefer '*_complete.csv', else plain

Output:
- ./reports/cincinnati2025_R16_predictions.csv
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]

PREF_DATA = "./data/match_dataset_post_wimbledon_2025_canada.csv"
BASES = ["./data/match_dataset_post_wimbledon_2025.csv",
         "./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

SEARCH_DIRS = ["./reports", "/mnt/data"]

# Prior events (only needed if PREF_DATA missing)
WIM_ROUNDS = ["R128","R64","R32","R16","QF","SF","F"]
CAN_ROUNDS = ["R64","R32","R16","QF","SF","F"]
TOURNEY_NAME_WIM = "Wimbledon"; SURFACE_WIM = "Grass"; LEVEL_WIM = "G"; BEST_OF_WIM = 5
TOURNEY_NAME_CAN = "Canada Masters 2025"; SURFACE_CAN = "Hard"; LEVEL_CAN = "M"; BEST_OF_CAN = 3

# Cincinnati constants
TOURNEY_NAME_CIN = "Cincinnati 2025"
SURFACE_CIN = "Hard"; LEVEL_CIN = "M"; BEST_OF_CIN = 3
ROUND_R64 = "R64"
ROUND_R32 = "R32"
ROUND_R16 = "R16"

# Output
OUT_CSV = "./reports/cincinnati2025_R16_predictions.csv"

# Header alignment preference
CIN_R32_HDR = [os.path.join(d, "cincinnati2025_R32_predictions_complete.csv") for d in SEARCH_DIRS] + \
              [os.path.join(d, "cincinnati2025_R32_predictions.csv") for d in SEARCH_DIRS]
CIN_R64_HDR = [os.path.join(d, "cincinnati2025_R64_predictions_complete.csv") for d in SEARCH_DIRS] + \
              [os.path.join(d, "cincinnati2025_R64_predictions.csv") for d in SEARCH_DIRS]
CAN_R64_HDR = [os.path.join(d, "canada2025_R64_predictions_complete.csv") for d in SEARCH_DIRS]
WIM_R128_HDR = [os.path.join(d, "wimbledon2025_R128_predictions_complete.csv") for d in SEARCH_DIRS]

# Hygiene
RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def find_pred_file(prefix: str, round_code: str, complete_only=True) -> Optional[str]:
    names = [f"{prefix}2025_{round_code}_predictions_complete.csv"] if complete_only \
            else [f"{prefix}2025_{round_code}_predictions_complete.csv", f"{prefix}2025_{round_code}_predictions.csv"]
    for d in SEARCH_DIRS:
        for nm in names:
            p = os.path.join(d, nm)
            if os.path.exists(p):
                return p
    return None

def load_round_df(p: str) -> pd.DataFrame:
    df = pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
    need = ["date","match_no","player_a","player_b","pred_winner"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing '{c}' in {p}")
    return df

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str,
                         tname: str, surface: str, level: str, best_of: int) -> pd.DataFrame:
    if round_df is None or round_df.empty:
        return base
    rows = []
    for _, r in round_df.iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tname,
            "surface": surface,
            "tourney_level": level,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": best_of,
            "winner_rank": np.nan, "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs = [(winners[i], winners[i+1]) for i in range(0, len(winners), 2)]
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)
    return pairs, d_prev, d_next

# ----------------------------
# Elo/logs (fixed surface WR pivots)
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon)
    log["opp_key"]    = log["opp"].map(canon)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)

    log["win_rate_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # per-surface smoothed WRs (fixed)
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")

    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_prior.columns: surf_prior[srf] = np.nan
        if srf not in surf_post.columns:  surf_post[srf]  = np.nan

    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()

    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(
        surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
        on=["player_key","date"], how="left"
    )
    log = log.merge(
        surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
        on=["player_key","date"], how="left"
    )

    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)

    return log

# ----------------------------
# Snapshot & features
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last.get("matches_28d_post" if after_last else "matches_28d_prior", 0.0)
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            if target_surface == "Hard":
                snap["wr_Hard"] = float(last_s["wr_Hard_post_sm"] if after_last else last_s["wr_Hard_prior_sm"])
            elif target_surface == "Grass":
                snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
            else:
                snap["wr_Clay"]  = float(last_s["wr_Clay_post_sm"]  if after_last else last_s["wr_Clay_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])

        snap["wr_Hard"]  = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"]  = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])
        snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_CSV)

    # model
    bundle = load(_first_existing(MODEL_PATHS) or MODEL_PATHS[0])
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # dataset
    if os.path.exists(PREF_DATA):
        matches = pd.read_csv(PREF_DATA)
    else:
        base_path = _first_existing(BASES)
        if not base_path:
            raise FileNotFoundError("No base dataset found.")
        base = pd.read_csv(base_path)
        # inject Wimbledon & Canada if present
        for r in WIM_ROUNDS:
            p = find_pred_file("wimbledon", r, complete_only=True)
            if p:
                base = augment_with_results(base, load_round_df(p), r, TOURNEY_NAME_WIM, SURFACE_WIM, LEVEL_WIM, BEST_OF_WIM)
        for r in CAN_ROUNDS:
            p = find_pred_file("canada", r, complete_only=True)
            if p:
                base = augment_with_results(base, load_round_df(p), r, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
        matches = base

    # Cincinnati prior rounds
    cin_r64 = None; cin_r32 = None
    for d in SEARCH_DIRS:
        p64c = os.path.join(d, "cincinnati2025_R64_predictions_complete.csv")
        p64  = os.path.join(d, "cincinnati2025_R64_predictions.csv")
        if os.path.exists(p64c): cin_r64 = pd.read_csv(p64c).sort_values("match_no").reset_index(drop=True)
        elif os.path.exists(p64): cin_r64 = pd.read_csv(p64).sort_values("match_no").reset_index(drop=True)

        p32c = os.path.join(d, "cincinnati2025_R32_predictions_complete.csv")
        p32  = os.path.join(d, "cincinnati2025_R32_predictions.csv")
        if os.path.exists(p32c): cin_r32 = pd.read_csv(p32c).sort_values("match_no").reset_index(drop=True)
        elif os.path.exists(p32): cin_r32 = pd.read_csv(p32).sort_values("match_no").reset_index(drop=True)

    if cin_r32 is None or cin_r32.empty:
        raise FileNotFoundError("Cincinnati R32 predictions file not found in ./reports or /mnt/data.")
    if cin_r64 is None or cin_r64.empty:
        print("[warn] Cincinnati R64 not found; proceeding with R32 only. (State may be slightly off)")

    # Derive R16 pairs & date
    r16_pairs, date_r32, date_r16 = derive_next_pairs(cin_r32)

    # Inject prior rounds so R16 uses post-R32 state
    matches_aug = matches.copy()
    if cin_r64 is not None and not cin_r64.empty:
        matches_aug = augment_with_results(matches_aug, cin_r64, ROUND_R64, TOURNEY_NAME_CIN, SURFACE_CIN, LEVEL_CIN, BEST_OF_CIN)
    matches_aug = augment_with_results(matches_aug, cin_r32, ROUND_R32, TOURNEY_NAME_CIN, SURFACE_CIN, LEVEL_CIN, BEST_OF_CIN)

    # Build player log
    log = build_player_log(matches_aug, EloParams())

    # Predict R16
    rows = []
    for i, (A, B) in enumerate(r16_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_r16, SURFACE_CIN)
        sb = snapshot_asof(log, B, A, date_r16, SURFACE_CIN)

        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE_CIN, "tourney_level": LEVEL_CIN, "round": ROUND_R16, "best_of": BEST_OF_CIN,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        conf = max(prob_a, prob_b)

        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_r16.strftime("%Y-%m-%d"),
            "round": ROUND_R16,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": 1.0 - prob_a,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,
        })

    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Align columns to Cincinnati R32 header if available; else Cincinnati R64; else Canada R64; else Wimbledon R128; else default
    header = None
    for cand in CIN_R32_HDR + CIN_R64_HDR + CAN_R64_HDR + WIM_R128_HDR:
        if os.path.exists(cand):
            header = list(pd.read_csv(cand, nrows=0).columns)
            break
    if header is None:
        header = [
            "match_no","date","round","player_a","player_b","pred_winner","confidence",
            "prob_player_a_win","prob_player_b_win",
            "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
            "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
            "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
            "correct_prediction",
        ]
    for c in header:
        if c not in out_df.columns:
            out_df[c] = pd.NA
    out_df = out_df[header].copy()

    _ensure_dir(OUT_CSV)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] Cincinnati R16 predictions → {OUT_CSV} (matches: {len(out_df)})")
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(out_df.head(10).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] Cincinnati R16 predictions → ./reports/cincinnati2025_R16_predictions.csv (matches: 8)
 match_no       date round              player_a         player_b           pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-17   R16         Jannik Sinner Adrian Mannarino         Jannik Sinner                 NaN    0.823013           0.823013           0.176987 549.463024         437.970253        0.168372       0.281218       0.456808        0.293304         549.463024      394.520347               -1.0            -2.228731                       0.4                      0.4          21.0            0.500000       -53.0
        2 2025-08-17   R16 Felix Auger Aliassime   Benjamin Bonz

In [65]:
# filepath: ./predict_cincinnati_2025_qf.py
"""
Cincinnati (Western & Southern Open) 2025 - QF predictions (deltas only, bracket order preserved).

Inputs:
- Model: ./models/rf_model.joblib
- Preferred dataset: ./data/match_dataset_post_wimbledon_2025_canada.csv
  (fallback: base + injected Wimbledon/Canada results if available)
- Cincinnati R64/R32/R16 predictions: prefer '*_complete.csv', else plain

Output:
- ./reports/cincinnati2025_QF_predictions.csv
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]

PREF_DATA = "./data/match_dataset_post_wimbledon_2025_canada.csv"
BASES = ["./data/match_dataset_post_wimbledon_2025.csv",
         "./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

SEARCH_DIRS = ["./reports", "/mnt/data"]

# Prior events (only needed if PREF_DATA missing)
WIM_ROUNDS = ["R128","R64","R32","R16","QF","SF","F"]
CAN_ROUNDS = ["R64","R32","R16","QF","SF","F"]
TOURNEY_NAME_WIM = "Wimbledon"; SURFACE_WIM = "Grass"; LEVEL_WIM = "G"; BEST_OF_WIM = 5
TOURNEY_NAME_CAN = "Canada Masters 2025"; SURFACE_CAN = "Hard"; LEVEL_CAN = "M"; BEST_OF_CAN = 3

# Cincinnati constants
TOURNEY_NAME_CIN = "Cincinnati 2025"
SURFACE_CIN = "Hard"; LEVEL_CIN = "M"; BEST_OF_CIN = 3
ROUND_R64 = "R64"
ROUND_R32 = "R32"
ROUND_R16 = "R16"
ROUND_QF  = "QF"

# Output
OUT_CSV = "./reports/cincinnati2025_QF_predictions.csv"

# Header alignment preference (use earlier rounds to align columns)
CIN_R16_HDR = [os.path.join(d, "cincinnati2025_R16_predictions_complete.csv") for d in SEARCH_DIRS] + \
              [os.path.join(d, "cincinnati2025_R16_predictions.csv") for d in SEARCH_DIRS]
CIN_R32_HDR = [os.path.join(d, "cincinnati2025_R32_predictions_complete.csv") for d in SEARCH_DIRS] + \
              [os.path.join(d, "cincinnati2025_R32_predictions.csv") for d in SEARCH_DIRS]
CIN_R64_HDR = [os.path.join(d, "cincinnati2025_R64_predictions_complete.csv") for d in SEARCH_DIRS] + \
              [os.path.join(d, "cincinnati2025_R64_predictions.csv") for d in SEARCH_DIRS]
CAN_R64_HDR = [os.path.join(d, "canada2025_R64_predictions_complete.csv") for d in SEARCH_DIRS]
WIM_R128_HDR = [os.path.join(d, "wimbledon2025_R128_predictions_complete.csv") for d in SEARCH_DIRS]

# Hygiene
RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def find_pred_file(prefix: str, round_code: str, complete_only=True) -> Optional[str]:
    names = [f"{prefix}2025_{round_code}_predictions_complete.csv"] if complete_only \
            else [f"{prefix}2025_{round_code}_predictions_complete.csv", f"{prefix}2025_{round_code}_predictions.csv"]
    for d in SEARCH_DIRS:
        for nm in names:
            p = os.path.join(d, nm)
            if os.path.exists(p):
                return p
    return None

def load_round_df(p: str) -> pd.DataFrame:
    df = pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
    need = ["date","match_no","player_a","player_b","pred_winner"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing '{c}' in {p}")
    return df

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str,
                         tname: str, surface: str, level: str, best_of: int) -> pd.DataFrame:
    if round_df is None or round_df.empty:
        return base
    rows = []
    for _, r in round_df.iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tname,
            "surface": surface,
            "tourney_level": level,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": best_of,
            "winner_rank": np.nan, "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs = [(winners[i], winners[i+1]) for i in range(0, len(winners), 2)]
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)  # snapshots use post-round state
    return pairs, d_prev, d_next

# ----------------------------
# Elo/logs (fixed surface WR pivots)
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon)
    log["opp_key"]    = log["opp"].map(canon)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)

    log["win_rate_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # per-surface smoothed WRs (fixed)
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")

    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_prior.columns: surf_prior[srf] = np.nan
        if srf not in surf_post.columns:  surf_post[srf]  = np.nan

    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()

    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(
        surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
        on=["player_key","date"], how="left"
    )
    log = log.merge(
        surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
        on=["player_key","date"], how="left"
    )

    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)

    return log

# ----------------------------
# Snapshot & features
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last.get("matches_28d_post" if after_last else "matches_28d_prior", 0.0)
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            if target_surface == "Hard":
                snap["wr_Hard"] = float(last_s["wr_Hard_post_sm"] if after_last else last_s["wr_Hard_prior_sm"])
            elif target_surface == "Grass":
                snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
            else:
                snap["wr_Clay"]  = float(last_s["wr_Clay_post_sm"]  if after_last else last_s["wr_Clay_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])

        snap["wr_Hard"]  = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"]  = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])
        snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_CSV)

    # model
    bundle = load(_first_existing(MODEL_PATHS) or MODEL_PATHS[0])
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # dataset
    if os.path.exists(PREF_DATA):
        matches = pd.read_csv(PREF_DATA)
    else:
        base_path = _first_existing(BASES)
        if not base_path:
            raise FileNotFoundError("No base dataset found.")
        base = pd.read_csv(base_path)
        # inject Wimbledon & Canada if present
        for r in WIM_ROUNDS:
            p = find_pred_file("wimbledon", r, complete_only=True)
            if p:
                base = augment_with_results(base, load_round_df(p), r, TOURNEY_NAME_WIM, SURFACE_WIM, LEVEL_WIM, BEST_OF_WIM)
        for r in CAN_ROUNDS:
            p = find_pred_file("canada", r, complete_only=True)
            if p:
                base = augment_with_results(base, load_round_df(p), r, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
        matches = base

    # Cincinnati prior rounds
    cin_r64 = None; cin_r32 = None; cin_r16 = None
    for d in SEARCH_DIRS:
        p64c = os.path.join(d, "cincinnati2025_R64_predictions_complete.csv")
        p64  = os.path.join(d, "cincinnati2025_R64_predictions.csv")
        if os.path.exists(p64c): cin_r64 = pd.read_csv(p64c).sort_values("match_no").reset_index(drop=True)
        elif os.path.exists(p64): cin_r64 = pd.read_csv(p64).sort_values("match_no").reset_index(drop=True)

        p32c = os.path.join(d, "cincinnati2025_R32_predictions_complete.csv")
        p32  = os.path.join(d, "cincinnati2025_R32_predictions.csv")
        if os.path.exists(p32c): cin_r32 = pd.read_csv(p32c).sort_values("match_no").reset_index(drop=True)
        elif os.path.exists(p32): cin_r32 = pd.read_csv(p32).sort_values("match_no").reset_index(drop=True)

        p16c = os.path.join(d, "cincinnati2025_R16_predictions_complete.csv")
        p16  = os.path.join(d, "cincinnati2025_R16_predictions.csv")
        if os.path.exists(p16c): cin_r16 = pd.read_csv(p16c).sort_values("match_no").reset_index(drop=True)
        elif os.path.exists(p16): cin_r16 = pd.read_csv(p16).sort_values("match_no").reset_index(drop=True)

    if cin_r16 is None or cin_r16.empty:
        raise FileNotFoundError("Cincinnati R16 predictions file not found in ./reports or /mnt/data.")
    if cin_r32 is None or cin_r32.empty:
        print("[warn] Cincinnati R32 not found; injecting R16 only (state may be slightly off).")
    if cin_r64 is None or cin_r64.empty:
        print("[warn] Cincinnati R64 not found; injecting later rounds only (state may be slightly off).")

    # Derive QF pairs & date
    qf_pairs, date_r16, date_qf = derive_next_pairs(cin_r16)

    # Inject prior rounds so QF uses post-R16 state
    matches_aug = matches.copy()
    if cin_r64 is not None and not cin_r64.empty:
        matches_aug = augment_with_results(matches_aug, cin_r64, ROUND_R64, TOURNEY_NAME_CIN, SURFACE_CIN, LEVEL_CIN, BEST_OF_CIN)
    if cin_r32 is not None and not cin_r32.empty:
        matches_aug = augment_with_results(matches_aug, cin_r32, ROUND_R32, TOURNEY_NAME_CIN, SURFACE_CIN, LEVEL_CIN, BEST_OF_CIN)
    matches_aug = augment_with_results(matches_aug, cin_r16, ROUND_R16, TOURNEY_NAME_CIN, SURFACE_CIN, LEVEL_CIN, BEST_OF_CIN)

    # Build player log
    log = build_player_log(matches_aug, EloParams())

    # Predict QF
    rows = []
    for i, (A, B) in enumerate(qf_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_qf, SURFACE_CIN)
        sb = snapshot_asof(log, B, A, date_qf, SURFACE_CIN)

        # alphabetical orientation for model
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE_CIN, "tourney_level": LEVEL_CIN, "round": ROUND_QF, "best_of": BEST_OF_CIN,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # robust rank
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        prob_b = 1.0 - prob_a
        winner = A if prob_a >= 0.5 else B
        conf = max(prob_a, prob_b)

        # winner − loser deltas
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_qf.strftime("%Y-%m-%d"),
            "round": ROUND_QF,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": 1.0 - prob_a,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,
        })

    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Align columns to Cincinnati R16 header if available; else R32; else R64; else Canada R64; else Wimbledon R128; else default
    header = None
    for cand in CIN_R16_HDR + CIN_R32_HDR + CIN_R64_HDR + CAN_R64_HDR + WIM_R128_HDR:
        if os.path.exists(cand):
            header = list(pd.read_csv(cand, nrows=0).columns)
            break
    if header is None:
        header = [
            "match_no","date","round","player_a","player_b","pred_winner","confidence",
            "prob_player_a_win","prob_player_b_win",
            "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
            "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
            "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
            "correct_prediction",
        ]
    for c in header:
        if c not in out_df.columns:
            out_df[c] = pd.NA
    out_df = out_df[header].copy()

    _ensure_dir(OUT_CSV)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] Cincinnati QF predictions → {OUT_CSV} (matches: {len(out_df)})")
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(out_df.head(10).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] Cincinnati QF predictions → ./reports/cincinnati2025_QF_predictions.csv (matches: 4)
 match_no       date round       player_a              player_b      pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-19    QF  Jannik Sinner Felix Auger Aliassime    Jannik Sinner                 NaN    0.783338           0.783338           0.216662 425.836791         368.660035        0.161836       0.175794       0.124923        0.171410         425.836791      297.365073                0.0            -0.939704                       0.4                      0.2          21.0           -0.600000       -18.0
        2 2025-08-19    QF Terence Atmane           Holger Rune      Holger Rune

In [78]:
# filepath: ./predict_cincinnati_2025_sf.py
"""
Cincinnati (Western & Southern Open) 2025 - SF predictions (deltas only, bracket order preserved).

Inputs:
- Model: ./models/rf_model.joblib
- Preferred dataset: ./data/match_dataset_post_wimbledon_2025_canada.csv
  (fallback: base + injected Wimbledon/Canada results if available)
- Cincinnati R64/R32/R16/QF predictions: prefer '*_complete.csv', else plain

Output:
- ./reports/cincinnati2025_SF_predictions.csv
"""

from __future__ import annotations

import os
import re
import unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ----------------------------
# Config
# ----------------------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]

PREF_DATA = "./data/match_dataset_post_wimbledon_2025_canada.csv"
BASES = ["./data/match_dataset_post_wimbledon_2025.csv",
         "./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]

SEARCH_DIRS = ["./reports", "/mnt/data"]

# Prior events (only needed if PREF_DATA missing)
WIM_ROUNDS = ["R128","R64","R32","R16","QF","SF","F"]
CAN_ROUNDS = ["R64","R32","R16","QF","SF","F"]
TOURNEY_NAME_WIM = "Wimbledon"; SURFACE_WIM = "Grass"; LEVEL_WIM = "G"; BEST_OF_WIM = 5
TOURNEY_NAME_CAN = "Canada Masters 2025"; SURFACE_CAN = "Hard"; LEVEL_CAN = "M"; BEST_OF_CAN = 3

# Cincinnati constants
TOURNEY_NAME_CIN = "Cincinnati 2025"
SURFACE_CIN = "Hard"; LEVEL_CIN = "M"; BEST_OF_CIN = 3
ROUND_R64 = "R64"
ROUND_R32 = "R32"
ROUND_R16 = "R16"
ROUND_QF  = "QF"
ROUND_SF  = "SF"

# Output
OUT_CSV = "./reports/cincinnati2025_SF_predictions.csv"

# Header alignment preference (use earlier rounds to align columns)
CIN_QF_HDR  = [os.path.join(d, "cincinnati2025_QF_predictions_complete.csv") for d in SEARCH_DIRS] + \
              [os.path.join(d, "cincinnati2025_QF_predictions.csv") for d in SEARCH_DIRS]
CIN_R16_HDR = [os.path.join(d, "cincinnati2025_R16_predictions_complete.csv") for d in SEARCH_DIRS] + \
              [os.path.join(d, "cincinnati2025_R16_predictions.csv") for d in SEARCH_DIRS]
CIN_R32_HDR = [os.path.join(d, "cincinnati2025_R32_predictions_complete.csv") for d in SEARCH_DIRS] + \
              [os.path.join(d, "cincinnati2025_R32_predictions.csv") for d in SEARCH_DIRS]
CIN_R64_HDR = [os.path.join(d, "cincinnati2025_R64_predictions_complete.csv") for d in SEARCH_DIRS] + \
              [os.path.join(d, "cincinnati2025_R64_predictions.csv") for d in SEARCH_DIRS]
CAN_R64_HDR = [os.path.join(d, "canada2025_R64_predictions_complete.csv") for d in SEARCH_DIRS]
WIM_R128_HDR= [os.path.join(d, "wimbledon2025_R128_predictions_complete.csv") for d in SEARCH_DIRS]

# Hygiene
RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ----------------------------
# Utils
# ----------------------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def find_pred_file(prefix: str, round_code: str, complete_only=True) -> Optional[str]:
    names = [f"{prefix}2025_{round_code}_predictions_complete.csv"] if complete_only \
            else [f"{prefix}2025_{round_code}_predictions_complete.csv", f"{prefix}2025_{round_code}_predictions.csv"]
    for d in SEARCH_DIRS:
        for nm in names:
            p = os.path.join(d, nm)
            if os.path.exists(p):
                return p
    return None

def load_round_df(p: str) -> pd.DataFrame:
    df = pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
    need = ["date","match_no","player_a","player_b","pred_winner"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing '{c}' in {p}")
    return df

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str,
                         tname: str, surface: str, level: str, best_of: int) -> pd.DataFrame:
    if round_df is None or round_df.empty: return base
    rows = []
    for _, r in round_df.iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tname,
            "surface": surface,
            "tourney_level": level,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": best_of,
            "winner_rank": np.nan, "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs = [(winners[i], winners[i+1]) for i in range(0, len(winners), 2)]
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)  # why: ensures snapshots use post-round state
    return pairs, d_prev, d_next

# ----------------------------
# Elo/logs (fixed surface WR pivots)
# ----------------------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]

        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))

        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({
            "player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
            "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[w_rank] if w_rank else np.nan,
        })
        rows.append({
            "player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
            "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
            "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
            "rank": r[l_rank] if l_rank else np.nan,
        })

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon)
    log["opp_key"]    = log["opp"].map(canon)

    # workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, inclusive=True))

    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # overall WR smoothing
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)

    log["win_rate_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # per-surface smoothed WRs (fixed)
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)

    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")

    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_prior.columns: surf_prior[srf] = np.nan
        if srf not in surf_post.columns:  surf_post[srf]  = np.nan

    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()

    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)

    log = log.merge(
        surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
        on=["player_key","date"], how="left"
    )
    log = log.merge(
        surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
        on=["player_key","date"], how="left"
    )

    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)

    return log

# ----------------------------
# Snapshot & features
# ----------------------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {
        "pre_elo": 1500.0, "pre_selo": 1500.0,
        "rest_days": 30.0, "matches_28d": 0.0,
        "rolling_10_winrate": 0.5, "rolling_5_winrate": 0.5, "streak": 0.0,
        "avg_rest_days": 20.0, "win_rate": 0.5, "peak_elo": 1500.0, "current_elo": 1500.0,
        "wr_Clay": 0.5, "wr_Grass": 0.5, "wr_Hard": 0.5,
        "rank_prior": np.nan, "rank_missing": True,
        "h2h_wr_prior": 0.5,
    }
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]

        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        current_elo = pre_elo
        peak_elo = last["peak_elo_post"] if after_last else last["peak_elo_prior"]
        win_rate = last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]
        avg_rest = last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]
        m28d = last.get("matches_28d_post" if after_last else "matches_28d_prior", 0.0)
        r10 = last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]
        r5  = last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]
        streak = last["streak_post"] if after_last else last["streak_prior"]

        snap.update({
            "pre_elo": float(pre_elo),
            "current_elo": float(current_elo),
            "peak_elo": float(peak_elo),
            "win_rate": float(win_rate),
            "avg_rest_days": float(avg_rest),
            "matches_28d": float(m28d),
            "rolling_10_winrate": float(r10),
            "rolling_5_winrate": float(r5),
            "streak": float(streak),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })

        sub_surf = sub[sub["surface"] == target_surface]
        if not sub_surf.empty:
            last_s = sub_surf.iloc[-1]
            snap["pre_selo"] = float(last_s["post_selo"] if after_last else last_s["pre_selo"])
            if target_surface == "Hard":
                snap["wr_Hard"] = float(last_s["wr_Hard_post_sm"] if after_last else last_s["wr_Hard_prior_sm"])
            elif target_surface == "Grass":
                snap["wr_Grass"] = float(last_s["wr_Grass_post_sm"] if after_last else last_s["wr_Grass_prior_sm"])
            else:
                snap["wr_Clay"]  = float(last_s["wr_Clay_post_sm"]  if after_last else last_s["wr_Clay_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])

        snap["wr_Hard"]  = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"]  = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])
        snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])

        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False

    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)

    return snap

# ----------------------------
# Main
# ----------------------------
def main() -> None:
    _ensure_dir(OUT_CSV)

    # model
    bundle = load(_first_existing(MODEL_PATHS) or MODEL_PATHS[0])
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # dataset
    if os.path.exists(PREF_DATA):
        matches = pd.read_csv(PREF_DATA)
    else:
        base_path = _first_existing(BASES)
        if not base_path:
            raise FileNotFoundError("No base dataset found.")
        base = pd.read_csv(base_path)
        # inject Wimbledon & Canada if present
        for r in WIM_ROUNDS:
            p = find_pred_file("wimbledon", r, complete_only=True)
            if p:
                base = augment_with_results(base, load_round_df(p), r, TOURNEY_NAME_WIM, SURFACE_WIM, LEVEL_WIM, BEST_OF_WIM)
        for r in CAN_ROUNDS:
            p = find_pred_file("canada", r, complete_only=True)
            if p:
                base = augment_with_results(base, load_round_df(p), r, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
        matches = base

    # Cincinnati prior rounds
    cin_r64 = cin_r32 = cin_r16 = cin_qf = None
    for d in SEARCH_DIRS:
        p64c = os.path.join(d, "cincinnati2025_R64_predictions_complete.csv")
        p64  = os.path.join(d, "cincinnati2025_R64_predictions.csv")
        if os.path.exists(p64c): cin_r64 = pd.read_csv(p64c).sort_values("match_no").reset_index(drop=True)
        elif os.path.exists(p64): cin_r64 = pd.read_csv(p64).sort_values("match_no").reset_index(drop=True)

        p32c = os.path.join(d, "cincinnati2025_R32_predictions_complete.csv")
        p32  = os.path.join(d, "cincinnati2025_R32_predictions.csv")
        if os.path.exists(p32c): cin_r32 = pd.read_csv(p32c).sort_values("match_no").reset_index(drop=True)
        elif os.path.exists(p32): cin_r32 = pd.read_csv(p32).sort_values("match_no").reset_index(drop=True)

        p16c = os.path.join(d, "cincinnati2025_R16_predictions_complete.csv")
        p16  = os.path.join(d, "cincinnati2025_R16_predictions.csv")
        if os.path.exists(p16c): cin_r16 = pd.read_csv(p16c).sort_values("match_no").reset_index(drop=True)
        elif os.path.exists(p16): cin_r16 = pd.read_csv(p16).sort_values("match_no").reset_index(drop=True)

        pqfc = os.path.join(d, "cincinnati2025_QF_predictions_complete.csv")
        pqf  = os.path.join(d, "cincinnati2025_QF_predictions.csv")
        if os.path.exists(pqfc): cin_qf = pd.read_csv(pqfc).sort_values("match_no").reset_index(drop=True)
        elif os.path.exists(pqf): cin_qf = pd.read_csv(pqf).sort_values("match_no").reset_index(drop=True)

    if cin_qf is None or cin_qf.empty:
        raise FileNotFoundError("Cincinnati QF predictions file not found in ./reports or /mnt/data.")

    # Derive SF pairs & date
    sf_pairs, date_qf, date_sf = derive_next_pairs(cin_qf)

    # Inject prior rounds so SF uses post-QF state
    matches_aug = matches.copy()
    if cin_r64 is not None and not cin_r64.empty:
        matches_aug = augment_with_results(matches_aug, cin_r64, ROUND_R64, TOURNEY_NAME_CIN, SURFACE_CIN, LEVEL_CIN, BEST_OF_CIN)
    if cin_r32 is not None and not cin_r32.empty:
        matches_aug = augment_with_results(matches_aug, cin_r32, ROUND_R32, TOURNEY_NAME_CIN, SURFACE_CIN, LEVEL_CIN, BEST_OF_CIN)
    if cin_r16 is not None and not cin_r16.empty:
        matches_aug = augment_with_results(matches_aug, cin_r16, ROUND_R16, TOURNEY_NAME_CIN, SURFACE_CIN, LEVEL_CIN, BEST_OF_CIN)
    matches_aug = augment_with_results(matches_aug, cin_qf, ROUND_QF, TOURNEY_NAME_CIN, SURFACE_CIN, LEVEL_CIN, BEST_OF_CIN)

    # Build player log
    log = build_player_log(matches_aug, EloParams())

    # Predict SF
    rows = []
    for i, (A, B) in enumerate(sf_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_sf, SURFACE_CIN)
        sb = snapshot_asof(log, B, A, date_sf, SURFACE_CIN)

        # alphabetical orientation for model
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(name: str) -> float:
            return float(s1[name]) - float(s2[name])

        feats = {
            "surface": SURFACE_CIN, "tourney_level": LEVEL_CIN, "round": ROUND_SF, "best_of": BEST_OF_CIN,
            "elo_diff": d("pre_elo"),
            "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"),
            "diff_wr_Hard": d("wr_Hard"),
            "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"),
            "diff_current_elo": d("current_elo"),
            "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"),
            "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"),
            "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"),
            "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # robust rank (why: don't let missing ranks dominate)
        if sa["rank_missing"] or sb["rank_missing"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])

        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        winner = A if prob_a >= 0.5 else B
        conf = max(prob_a, 1.0 - prob_a)

        # winner − loser deltas
        snap_w = sa if winner == A else sb
        snap_l = sb if winner == A else sa

        def delta(col: str) -> float:
            return float(snap_w[col] - snap_l[col])

        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i,
            "date": date_sf.strftime("%Y-%m-%d"),
            "round": ROUND_SF,
            "player_a": A,
            "player_b": B,
            "pred_winner": winner,
            "confidence": conf,
            "prob_player_a_win": prob_a,
            "prob_player_b_win": 1.0 - prob_a,

            "delta_elo": delta("pre_elo"),
            "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"),
            "delta_wr_Hard": delta("wr_Hard"),
            "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"),
            "delta_current_elo": delta("current_elo"),
            "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"),
            "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"),
            "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"),
            "delta_h2h_wr_prior": delta("h2h_wr_prior"),
            "delta_rank": delta_rank,

            "correct_prediction": np.nan,
        })

    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Align columns to earlier Cincinnati headers; fallback to Canada/Wimbledon; else default
    header = None
    for cand in CIN_QF_HDR + CIN_R16_HDR + CIN_R32_HDR + CIN_R64_HDR + CAN_R64_HDR + WIM_R128_HDR:
        if os.path.exists(cand):
            header = list(pd.read_csv(cand, nrows=0).columns)
            break
    if header is None:
        header = [
            "match_no","date","round","player_a","player_b","pred_winner","confidence",
            "prob_player_a_win","prob_player_b_win",
            "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
            "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
            "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
            "correct_prediction",
        ]
    for c in header:
        if c not in out_df.columns:
            out_df[c] = pd.NA
    out_df = out_df[header].copy()

    _ensure_dir(OUT_CSV)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] Cincinnati SF predictions → {OUT_CSV} (matches: {len(out_df)})")
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(out_df.head(10).to_string(index=False))


if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] Cincinnati SF predictions → ./reports/cincinnati2025_SF_predictions.csv (matches: 2)
 match_no       date round         player_a       player_b      pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-21    SF    Jannik Sinner Terence Atmane    Jannik Sinner                 NaN    0.880970           0.880970           0.119030 557.139705         475.663260        0.217391       0.346111       0.190141        0.321429         557.139705      557.139705                0.0           -17.324786                       0.4                      0.2          21.0            0.000000      -160.0
        2 2025-08-21    SF Alexander Zverev Carlos Alcaraz Alexander Zverev               

In [79]:
# filepath: ./predict_cincinnati_2025_f.py
"""
Cincinnati (Western & Southern Open) 2025 - Final prediction (deltas only, bracket order preserved).
Writes: ./reports/cincinnati2025_F_predictions.csv
"""

from __future__ import annotations

import os, re, unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ---------------- Config ----------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
PREF_DATA = "./data/match_dataset_post_wimbledon_2025_canada.csv"
BASES = ["./data/match_dataset_post_wimbledon_2025.csv",
         "./atp_matches_2021_2024.csv", "/mnt/data/atp_matches_2021_2024.csv"]
SEARCH_DIRS = ["./reports", "/mnt/data"]

# Earlier slams/masters (only if PREF_DATA missing)
WIM_ROUNDS = ["R128","R64","R32","R16","QF","SF","F"]
CAN_ROUNDS = ["R64","R32","R16","QF","SF","F"]
TOURNEY_NAME_WIM, SURFACE_WIM, LEVEL_WIM, BEST_OF_WIM = "Wimbledon", "Grass", "G", 5
TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN = "Canada Masters 2025", "Hard", "M", 3

# Cincinnati constants
TOURNEY_NAME_CIN, SURFACE_CIN, LEVEL_CIN, BEST_OF_CIN = "Cincinnati 2025", "Hard", "M", 3
ROUND_R64, ROUND_R32, ROUND_R16, ROUND_QF, ROUND_SF, ROUND_F = "R64","R32","R16","QF","SF","F"

OUT_CSV = "./reports/cincinnati2025_F_predictions.csv"

RANK_CLIP = 500
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# -------------- Utils --------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def find_pred_file(prefix: str, round_code: str, complete_only=True) -> Optional[str]:
    names = [f"{prefix}2025_{round_code}_predictions_complete.csv"] if complete_only \
            else [f"{prefix}2025_{round_code}_predictions_complete.csv", f"{prefix}2025_{round_code}_predictions.csv"]
    for d in SEARCH_DIRS:
        for nm in names:
            p = os.path.join(d, nm)
            if os.path.exists(p):
                return p
    return None

def load_round_df(p: str) -> pd.DataFrame:
    df = pd.read_csv(p).sort_values("match_no").reset_index(drop=True)
    for c in ["date","match_no","player_a","player_b","pred_winner"]:
        if c not in df.columns:
            raise ValueError(f"Missing '{c}' in {p}")
    return df

def actual_winner(row: pd.Series) -> str:
    if "correct_prediction" in row and pd.notna(row["correct_prediction"]):
        cp = int(row["correct_prediction"])
        return row["pred_winner"] if cp == 1 else (row["player_b"] if row["pred_winner"] == row["player_a"] else row["player_a"])
    return row["pred_winner"]

def augment_with_results(base: pd.DataFrame, round_df: pd.DataFrame, round_code: str,
                         tname: str, surface: str, level: str, best_of: int) -> pd.DataFrame:
    if round_df is None or round_df.empty: return base
    rows = []
    for _, r in round_df.iterrows():
        w = actual_winner(r)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": tname,
            "surface": surface,
            "tourney_level": level,
            "match_num": int(r["match_no"]),
            "winner_name": w,
            "loser_name": l,
            "round": round_code,
            "best_of": best_of,
            "winner_rank": np.nan, "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

def derive_next_pairs(prior_df: pd.DataFrame) -> Tuple[List[Tuple[str,str]], pd.Timestamp, pd.Timestamp]:
    winners = prior_df.apply(actual_winner, axis=1).tolist()
    pairs = [(winners[i], winners[i+1]) for i in range(0, len(winners), 2)]
    d_prev = _safe_date(prior_df["date"].iloc[0])
    d_next = d_prev + pd.Timedelta(days=2)
    return pairs, d_prev, d_next

# -------------- Elo/logs --------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    need = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in need:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    elo_overall: Dict[str, float] = {}
    elo_surface: Dict[Tuple[str,str], float] = {}
    BASE, K, KS = params.base, params.k_overall, params.k_surface

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)
        exp_w   = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))
        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)
        rows.append({"player": w,"opp": l,"date": d,"surface": s,"is_win": 1,
                     "pre_elo": ew_w,"pre_selo": es_w,"post_elo": new_ew_w,"post_selo": new_es_w,
                     "tourney_level": r["tourney_level"],"round": r["round"],"best_of": r["best_of"],
                     "rank": r[w_rank] if w_rank else np.nan})
        rows.append({"player": l,"opp": w,"date": d,"surface": s,"is_win": 0,
                     "pre_elo": ew_l,"pre_selo": es_l,"post_elo": new_ew_l,"post_selo": new_es_l,
                     "tourney_level": r["tourney_level"],"round": r["round"],"best_of": r["best_of"],
                     "rank": r[l_rank] if l_rank else np.nan})
        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon)
    log["opp_key"]    = log["opp"].map(canon)

    # workload & form
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0, 60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0)); dq.append(dt)
        return out

    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, True))

    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # smoothed overall WRs
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)
    log["win_rate_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["win_rate_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # smoothed per-surface WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)
    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_prior.columns: surf_prior[srf] = np.nan
        if srf not in surf_post.columns:  surf_post[srf]  = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)
    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)
    return log

# -------------- Snapshot --------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {"pre_elo":1500.0,"pre_selo":1500.0,"rest_days":30.0,"matches_28d":0.0,
            "rolling_10_winrate":0.5,"rolling_5_winrate":0.5,"streak":0.0,
            "avg_rest_days":20.0,"win_rate":0.5,"peak_elo":1500.0,"current_elo":1500.0,
            "wr_Clay":0.5,"wr_Grass":0.5,"wr_Hard":0.5,"rank_prior":np.nan,"rank_missing":True,
            "h2h_wr_prior":0.5}
    if not sub.empty:
        last = sub.iloc[-1]; after_last = asof > last["date"]
        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        snap.update({
            "pre_elo": float(pre_elo), "current_elo": float(pre_elo),
            "peak_elo": float(last["peak_elo_post"] if after_last else last["peak_elo_prior"]),
            "win_rate": float(last["win_rate_post_sm"] if after_last else last["win_rate_prior_sm"]),
            "avg_rest_days": float(last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]),
            "matches_28d": float(last.get("matches_28d_post" if after_last else "matches_28d_prior", 0.0)),
            "rolling_10_winrate": float(last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]),
            "rolling_5_winrate": float(last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]),
            "streak": float(last["streak_post"] if after_last else last["streak_prior"]),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
        })
        sub_s = sub[sub["surface"] == target_surface]
        if not sub_s.empty:
            ls = sub_s.iloc[-1]
            snap["pre_selo"] = float(ls["post_selo"] if after_last else ls["pre_selo"])
            if target_surface == "Hard":
                snap["wr_Hard"] = float(ls["wr_Hard_post_sm"] if after_last else ls["wr_Hard_prior_sm"])
            elif target_surface == "Grass":
                snap["wr_Grass"] = float(ls["wr_Grass_post_sm"] if after_last else ls["wr_Grass_prior_sm"])
            else:
                snap["wr_Clay"]  = float(ls["wr_Clay_post_sm"]  if after_last else ls["wr_Clay_prior_sm"])
        else:
            snap["pre_selo"] = float(last["post_selo"] if after_last else last["pre_selo"])
        snap["wr_Hard"]  = float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"])
        snap["wr_Clay"]  = float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"])
        snap["wr_Grass"] = float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"])
        rp = sub["rank_prior"].dropna()
        if not rp.empty:
            snap["rank_prior"] = float(rp.iloc[-1]); snap["rank_missing"] = False
    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)
    return snap

# -------------- Main --------------
def main() -> None:
    _ensure_dir(OUT_CSV)

    # model
    bundle = load(_first_existing(MODEL_PATHS) or MODEL_PATHS[0])
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # dataset
    if os.path.exists(PREF_DATA):
        matches = pd.read_csv(PREF_DATA)
    else:
        base_path = _first_existing(BASES)
        if not base_path: raise FileNotFoundError("No base dataset found.")
        base = pd.read_csv(base_path)
        for r in WIM_ROUNDS:
            p = find_pred_file("wimbledon", r, complete_only=True)
            if p: base = augment_with_results(base, load_round_df(p), r, TOURNEY_NAME_WIM, SURFACE_WIM, LEVEL_WIM, BEST_OF_WIM)
        for r in CAN_ROUNDS:
            p = find_pred_file("canada", r, complete_only=True)
            if p: base = augment_with_results(base, load_round_df(p), r, TOURNEY_NAME_CAN, SURFACE_CAN, LEVEL_CAN, BEST_OF_CAN)
        matches = base

    # prior CIN rounds
    cin = {}
    for rd in [ROUND_R64, ROUND_R32, ROUND_R16, ROUND_QF, ROUND_SF]:
        p = find_pred_file("cincinnati", rd, complete_only=False)
        if p: cin[rd] = load_round_df(p)
    if ROUND_SF not in cin or cin[ROUND_SF].empty:
        raise FileNotFoundError("Cincinnati SF predictions file not found.")

    # derive Final pairs/date
    f_pairs, date_sf, date_f = derive_next_pairs(cin[ROUND_SF])

    # inject prior rounds
    matches_aug = matches.copy()
    for rd in [ROUND_R64, ROUND_R32, ROUND_R16, ROUND_QF, ROUND_SF]:
        if rd in cin and not cin[rd].empty:
            matches_aug = augment_with_results(matches_aug, cin[rd], rd, TOURNEY_NAME_CIN, SURFACE_CIN, LEVEL_CIN, BEST_OF_CIN)

    # logs
    log = build_player_log(matches_aug, EloParams())

    # predict Final
    rows = []
    for i, (A, B) in enumerate(f_pairs, start=1):
        sa = snapshot_asof(log, A, B, date_f, SURFACE_CIN)
        sb = snapshot_asof(log, B, A, date_f, SURFACE_CIN)
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)
        def d(k: str) -> float: return float(s1[k] - s2[k])
        feats = {
            "surface": SURFACE_CIN, "tourney_level": LEVEL_CIN, "round": ROUND_F, "best_of": BEST_OF_CIN,
            "elo_diff": d("pre_elo"), "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"), "diff_wr_Hard": d("wr_Hard"), "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"), "diff_current_elo": d("current_elo"), "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"), "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"), "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"), "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        feats["rank_diff"] = 0.0 if (sa["rank_missing"] or sb["rank_missing"]) else float(np.clip(s1["rank_prior"] - s2["rank_prior"], -RANK_CLIP, RANK_CLIP))
        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])
        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        winner = A if prob_a >= 0.5 else B
        conf = max(prob_a, 1.0 - prob_a)
        snap_w, snap_l = (sa, sb) if winner == A else (sb, sa)
        def delta(k: str) -> float: return float(snap_w[k] - snap_l[k])
        delta_rank = 0.0
        if (not snap_w["rank_missing"]) and (not snap_l["rank_missing"]):
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))
        rows.append({
            "match_no": i, "date": date_f.strftime("%Y-%m-%d"), "round": ROUND_F,
            "player_a": A, "player_b": B, "pred_winner": winner,
            "confidence": conf, "prob_player_a_win": prob_a, "prob_player_b_win": 1.0 - prob_a,
            "delta_elo": delta("pre_elo"), "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"), "delta_wr_Hard": delta("wr_Hard"), "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"), "delta_current_elo": delta("current_elo"), "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"), "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"), "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"), "delta_h2h_wr_prior": delta("h2h_wr_prior"), "delta_rank": delta_rank,
            "correct_prediction": np.nan,
        })
    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # header align (prefer SF/QF/R16/R32/R64)
    header = None
    for cand in [find_pred_file("cincinnati", ROUND_SF, complete_only=False),
                 find_pred_file("cincinnati", ROUND_QF, complete_only=False),
                 find_pred_file("cincinnati", ROUND_R16, complete_only=False),
                 find_pred_file("cincinnati", ROUND_R32, complete_only=False),
                 find_pred_file("cincinnati", ROUND_R64, complete_only=False)]:
        if cand and os.path.exists(cand):
            header = list(pd.read_csv(cand, nrows=0).columns)
            break
    if header is None:
        header = ["match_no","date","round","player_a","player_b","pred_winner","confidence",
                  "prob_player_a_win","prob_player_b_win","delta_elo","delta_surface_elo","delta_wr_Grass",
                  "delta_wr_Hard","delta_wr_Clay","delta_win_rate","delta_current_elo","delta_peak_elo",
                  "delta_matches_28d","delta_avg_rest_days","delta_rolling_10_winrate","delta_rolling_5_winrate",
                  "delta_streak","delta_h2h_wr_prior","delta_rank","correct_prediction"]
    for c in header:
        if c not in out_df.columns: out_df[c] = pd.NA
    out_df = out_df[header].copy()

    _ensure_dir(OUT_CSV)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] Cincinnati Final prediction → {OUT_CSV}")
    with pd.option_context("display.max_columns", None, "display.width", 170):
        print(out_df.to_string(index=False))

if __name__ == "__main__":
    main()




c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] Cincinnati Final prediction → ./reports/cincinnati2025_F_predictions.csv
 match_no       date round      player_a       player_b   pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-23     F Jannik Sinner Carlos Alcaraz Jannik Sinner                 NaN    0.672746           0.672746           0.327254 138.757087         159.997845       -0.078063        0.06505      -0.104252        0.000413         138.757087      108.848698                0.0            -0.457276                       0.1                      0.0          21.0           -0.076923        -2.0


In [80]:
# filepath: ./combine_cincinnati_2025_predictions.py
"""
Combine all Cincinnati 2025 round prediction CSVs into one file, preserving bracket order.

Searches ./reports and /mnt/data for:
  cincinnati2025_{R64,R32,R16,QF,SF,F}_predictions_complete.csv (preferred)
  cincinnati2025_{round}_predictions.csv (fallback)

Writes: ./reports/cincinnati2025_all_rounds_predictions_combined.csv
"""

from __future__ import annotations

import os
from typing import List, Dict
import pandas as pd

ROUNDS: List[str] = ["R64","R32","R16","QF","SF","F"]
ROUND_ORDER: Dict[str,int] = {r:i for i,r in enumerate(ROUNDS, start=1)}
SEARCH_DIRS = ["./reports", "/mnt/data"]
OUT = "./reports/cincinnati2025_all_rounds_predictions_combined.csv"

def first_existing(path_list: List[str]) -> str | None:
    for p in path_list:
        if os.path.exists(p): return p
    return None

def find_round_file(round_code: str) -> str | None:
    preferred = [os.path.join(d, f"cincinnati2025_{round_code}_predictions_complete.csv") for d in SEARCH_DIRS]
    fallback  = [os.path.join(d, f"cincinnati2025_{round_code}_predictions.csv")           for d in SEARCH_DIRS]
    return first_existing(preferred) or first_existing(fallback)

def load_round(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "round" not in df.columns: raise ValueError(f"'round' missing in {path}")
    if "match_no" not in df.columns: raise ValueError(f"'match_no' missing in {path}")
    return df

def main() -> None:
    os.makedirs(os.path.dirname(OUT) or ".", exist_ok=True)

    files = []
    for r in ROUNDS:
        p = find_round_file(r)
        if p: files.append(p)
    if not files:
        raise FileNotFoundError("No Cincinnati round prediction CSVs found.")

    # Reference header from first available (prefer earliest round with widest columns)
    ref_header = None
    for r in ["R64","R32","R16","QF","SF","F"]:
        p = find_round_file(r)
        if p:
            ref_header = list(pd.read_csv(p, nrows=0).columns)
            break

    frames = []
    union_cols: set[str] = set(ref_header or [])
    for p in files:
        df = load_round(p)
        union_cols.update(df.columns)
        frames.append(df)

    if ref_header is None:
        preferred = ["match_no","date","round","player_a","player_b","pred_winner","confidence",
                     "prob_player_a_win","prob_player_b_win"]
        deltas = sorted([c for c in union_cols if str(c).startswith("delta_")])
        rest = [c for c in sorted(union_cols) if c not in preferred + deltas]
        ref_header = preferred + deltas + rest

    aligned = []
    for df in frames:
        for c in ref_header:
            if c not in df.columns: df[c] = pd.NA
        aligned.append(df[ref_header].copy())

    big = pd.concat(aligned, ignore_index=True)
    big["__ro__"] = big["round"].map(ROUND_ORDER).fillna(9999).astype(int)
    big["__mo__"] = pd.to_numeric(big["match_no"], errors="coerce")
    big = big.sort_values(["__ro__","__mo__"], kind="mergesort").drop(columns=["__ro__","__mo__"]).reset_index(drop=True)

    big.to_csv(OUT, index=False, encoding="utf-8-sig")
    print(f"[OK] Combined Cincinnati predictions → {OUT}")
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(big.head(12).to_string(index=False))

if __name__ == "__main__":
    main()

[OK] Combined Cincinnati predictions → ./reports/cincinnati2025_all_rounds_predictions_combined.csv
 match_no      date round                    player_a              player_b           pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 8/13/2025   R64               Jannik Sinner    Daniel Elahi Galan         Jannik Sinner                   1    0.854621           0.854621           0.145379 696.285710         553.596087        0.193582       0.393498       0.300310        0.379510         696.285710      559.450561                  0            -7.236014                       0.7                      0.6            22            0.500000        -123
        2 8/13/2025   R64           

In [81]:
# filepath: ./update_post_cincinnati_and_build_profiles.py
"""
Append Cincinnati 2025 results to a new dataset and rebuild player profiles (exact schema).

Usage (Jupyter/CLI-safe):
    python update_post_cincinnati_and_build_profiles.py \
        --strict-actual \
        --override ./reports/cincinnati2025_results_override.csv

Key flags:
  --base           Path to starting dataset (default: prefer post-Canada file).
  --strict-actual  Require actual winners (via correct_prediction or override); else fail.
  --override       CSV mapping {round,match_no,actual_winner}; takes precedence per match.
  --out-dataset    Output dataset path (default: ./data/match_dataset_post_cincinnati_2025.csv)
  --out-profiles   Output profiles path (default: ./reports/player_profiles_latest.csv)
"""

from __future__ import annotations

import argparse
import os
import re
import sys
import unicodedata
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# ---------- Config ----------
PREF_BASES = [
    "./data/match_dataset_post_wimbledon_2025_canada.csv",
    "./data/match_dataset_post_wimbledon_2025.csv",
    "./atp_matches_2021_2024.csv",
    "/mnt/data/atp_matches_2021_2024.csv",
]
SEARCH_DIRS = ["./reports", "/mnt/data"]
ROUNDS = ["R64", "R32", "R16", "QF", "SF", "F"]

TOURNEY_NAME = "Cincinnati 2025"
SURFACE = "Hard"
LEVEL = "M"
BEST_OF = 3

DEFAULT_OUT_DATASET = "./data/match_dataset_post_cincinnati_2025.csv"
DEFAULT_OUT_PROFILES = "./reports/player_profiles_latest.csv"

# Elo & smoothing
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ---------- Utils ----------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def find_cincy_round(round_code: str) -> Optional[str]:
    prefs = [f"cincinnati2025_{round_code}_predictions_complete.csv",
             f"cincinnati2025_{round_code}_predictions.csv"]
    for d in SEARCH_DIRS:
        for nm in prefs:
            p = os.path.join(d, nm)
            if os.path.exists(p):
                return p
    return None

def load_round_df(path: str) -> pd.DataFrame:
    df = pd.read_csv(path).sort_values("match_no").reset_index(drop=True)
    for c in ["date","match_no","player_a","player_b","pred_winner"]:
        if c not in df.columns:
            raise ValueError(f"Missing '{c}' in {path}")
    return df

def load_override(path: Optional[str]) -> Dict[Tuple[str,int], str]:
    if not path: return {}
    ov = pd.read_csv(path)
    # Accept columns: round,match_no,actual_winner (or winner)
    cand_name = "actual_winner" if "actual_winner" in ov.columns else ("winner" if "winner" in ov.columns else None)
    if cand_name is None or "round" not in ov.columns or "match_no" not in ov.columns:
        raise ValueError("Override CSV must have columns: round,match_no,actual_winner (or winner)")
    out: Dict[Tuple[str,int], str] = {}
    for _, r in ov.iterrows():
        try:
            out[(str(r["round"]).strip(), int(r["match_no"]))] = str(r[cand_name]).strip()
        except Exception:
            continue
    return out

def winner_from_row(r: pd.Series, round_code: str, override: Dict[Tuple[str,int], str],
                    strict_actual: bool) -> str:
    key = (round_code, int(r["match_no"]))
    if key in override:
        return override[key]
    # If we have correctness, honor it
    if "correct_prediction" in r and pd.notna(r["correct_prediction"]):
        cp = int(r["correct_prediction"])
        return r["pred_winner"] if cp == 1 else (r["player_b"] if r["pred_winner"] == r["player_a"] else r["player_a"])
    # If strict, do not assume predicted winner
    if strict_actual:
        raise ValueError(f"Missing actual winner for {round_code} match_no={r['match_no']}. "
                         f"Provide override or fill 'correct_prediction'.")
    # Else use prediction
    return r["pred_winner"]

def append_round_results(base: pd.DataFrame, rnd: pd.DataFrame, round_code: str,
                         override: Dict[Tuple[str,int], str], strict_actual: bool) -> pd.DataFrame:
    if rnd is None or rnd.empty: return base
    rows = []
    for _, r in rnd.iterrows():
        w = winner_from_row(r, round_code, override, strict_actual)
        l = r["player_b"] if w == r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": str(w),
            "loser_name": str(l),
            "round": round_code,
            "best_of": BEST_OF,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    extra["surface"] = extra["surface"].map(normalize_surface)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common + add]], ignore_index=True, sort=False)

# ---------- Log builder (post-win streak is computed from real results) ----------
def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    req = ["tourney_date","tourney_name","surface","tourney_level","match_num","winner_name","loser_name","round","best_of"]
    for c in req:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    BASE, K, KS = params.base, params.k_overall, params.k_surface
    elo_overall: Dict[str,float] = {}
    elo_surface: Dict[Tuple[str,str],float] = {}

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)
        exp_w   = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))
        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({"player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
                     "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[w_rank] if w_rank else np.nan})
        rows.append({"player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
                     "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[l_rank] if l_rank else np.nan})

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon)

    # Rest/workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0,60)

    # Workload counts (prior/post)
    from collections import deque
    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0))
            dq.append(dt)
        return out
    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, True))

    grp = log.groupby("player_key")
    # Form (prior/post)
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)

    # Streaks (why: display WHO is actually on a winning run)
    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # Averages & smoothed WRs
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["wr_overall_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["wr_overall_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # Per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_prior.columns: surf_prior[srf] = np.nan
        if srf not in surf_post.columns:  surf_post[srf]  = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)
    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # Peaks
    log["peak_elo_prior"] = log.groupby("player_key")["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = log.groupby("player_key")["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    return log

# ---------- Profiles (exact schema) ----------
def compute_profiles_schema(log: pd.DataFrame) -> pd.DataFrame:
    idx_last = log.groupby("player_key")["date"].idxmax()
    last = log.loc[idx_last].copy()

    def latest_surface_selo(df: pd.DataFrame, surface: str) -> pd.Series:
        sub = df[df["surface"] == surface]
        if sub.empty:
            return pd.Series(1500.0, index=df["player_key"].unique())
        idx = sub.groupby("player_key")["date"].idxmax()
        sub_last = sub.loc[idx, ["player_key","post_selo"]].set_index("player_key")["post_selo"]
        all_keys = pd.Index(df["player_key"].unique(), name="player_key")
        return sub_last.reindex(all_keys).fillna(1500.0)

    all_keys = last["player_key"].unique()
    df_all = log[log["player_key"].isin(all_keys)]
    selo_clay  = latest_surface_selo(df_all, "Clay")
    selo_grass = latest_surface_selo(df_all, "Grass")
    selo_hard  = latest_surface_selo(df_all, "Hard")

    prof = pd.DataFrame({
        "name": last["player"].values,
        "last_match_date": pd.to_datetime(last["date"]).dt.strftime("%-m/%-d/%Y" if os.name != "nt" else "%m/%d/%Y").values,
        "current_elo": last["post_elo"].astype(float).values,
        "peak_elo": last["peak_elo_post"].astype(float).values,
        "selo_Clay": selo_clay.reindex(last["player_key"]).values.astype(float),
        "selo_Grass": selo_grass.reindex(last["player_key"]).values.astype(float),
        "selo_Hard": selo_hard.reindex(last["player_key"]).values.astype(float),
        "wr_Clay": last["wr_Clay_post_sm"].astype(float).fillna(0.5).values,
        "wr_Grass": last["wr_Grass_post_sm"].astype(float).fillna(0.5).values,
        "wr_Hard": last["wr_Hard_post_sm"].astype(float).fillna(0.5).values,
        "overall_wr": last["wr_overall_post_sm"].astype(float).values,
        "form10_wr": last["rolling_10_winrate_post"].astype(float).values,
        "form5_wr": last["rolling_5_winrate_post"].astype(float).values,
        "streak": last["streak_post"].astype(float).values,
        "avg_rest_days": last["avg_rest_days_post"].astype(float).values,
        "matches_28d": last["matches_28d_post"].astype(float).values,
    })

    cols = [
        "name","last_match_date","current_elo","peak_elo",
        "selo_Clay","selo_Grass","selo_Hard",
        "wr_Clay","wr_Grass","wr_Hard",
        "overall_wr","form10_wr","form5_wr","streak","avg_rest_days","matches_28d"
    ]
    prof = prof[cols].copy()

    # Fills
    for c in ["selo_Clay","selo_Grass","selo_Hard"]:
        prof[c] = prof[c].fillna(1500.0)
    for c in ["wr_Clay","wr_Grass","wr_Hard","overall_wr","form10_wr","form5_wr"]:
        prof[c] = prof[c].fillna(0.5)
    for c in ["streak","avg_rest_days","matches_28d"]:
        prof[c] = prof[c].fillna(0.0)

    # Sort: stronger first
    prof = prof.sort_values(["current_elo","overall_wr"], ascending=[False, False]).reset_index(drop=True)
    return prof

# ---------- Main ----------
def main() -> None:
    ap = argparse.ArgumentParser(add_help=True)
    ap.add_argument("--base", default=None, help="Input base dataset CSV (post-Canada preferred)")
    ap.add_argument("--out-dataset", default=DEFAULT_OUT_DATASET)
    ap.add_argument("--out-profiles", default=DEFAULT_OUT_PROFILES)
    ap.add_argument("--override", default=None, help="CSV with columns: round,match_no,actual_winner")
    ap.add_argument("--strict-actual", action="store_true",
                    help="Require actual winners via correct_prediction or override (no prediction fallback).")
    args, _ = ap.parse_known_args()

    base_path = args.base or first_existing(PREF_BASES)
    if not base_path:
        raise FileNotFoundError("No base dataset found. Provide --base.")
    matches = pd.read_csv(base_path)

    override_map = load_override(args.override)

    # Append CIN results with correct winners
    for rc in ROUNDS:
        p = find_cincy_round(rc)
        if not p:
            if args.strict_actual:
                raise FileNotFoundError(f"Missing Cincinnati file for round {rc} in ./reports or /mnt/data")
            else:
                print(f"[warn] Cincinnati {rc} file not found; skipping.")
                continue
        rnd = load_round_df(p)
        matches = append_round_results(matches, rnd, rc, override_map, args.strict_actual)

    # Save dataset
    _ensure_dir(args.out_dataset)
    matches.to_csv(args.out_dataset, index=False, encoding="utf-8-sig")
    print(f"[OK] Updated dataset → {args.out_dataset} (rows: {len(matches)})")

    # Build logs & profiles
    log = build_player_log(matches, EloParams())
    profiles = compute_profiles_schema(log)

    _ensure_dir(args.out_profiles)
    profiles.to_csv(args.out_profiles, index=False, encoding="utf-8-sig")
    print(f"[OK] Player profiles → {args.out_profiles} (players: {len(profiles)})")

    # Quick sanity peek: show finalist trail if F exists
    try:
        f_path = find_cincy_round("F")
        if f_path:
            f_df = pd.read_csv(f_path).sort_values("match_no")
            fins = set(f_df["player_a"]).union(set(f_df["player_b"]))
            fins_canon = {canon(x) for x in fins}
            last_dates = log[log["player_key"].isin(fins_canon)].groupby("player_key")["date"].max()
            print("\n[check] Cincinnati finalists last dates:")
            for pk, dt in last_dates.items():
                nm = log.loc[log["player_key"] == pk, "player"].iloc[-1]
                streak = log.loc[(log["player_key"] == pk) & (log["date"] == dt), "streak_post"].iloc[-1]
                print(f"  - {nm}: last_date={dt.date()}  streak={int(streak)}")
    except Exception as e:
        print(f"[check] finalist peek skipped: {e}")

if __name__ == "__main__":
    main()


[OK] Updated dataset → ./data/match_dataset_post_cincinnati_2025.csv (rows: 11965)
[OK] Player profiles → ./reports/player_profiles_latest.csv (players: 737)

[check] Cincinnati finalists last dates:
  - Carlos Alcaraz: last_date=2025-08-23  streak=6
  - Jannik Sinner: last_date=2025-08-23  streak=-1


In [87]:
# filepath: ./predict_usopen_2025_r128.py
"""
Predict US Open 2025 R128 using the trained 2021–2024 RF model (post-Cincinnati state).

- Preserves EXACT bracket order as listed in USO_R128_PAIRS below.
- Outputs a deltas-only CSV identical to earlier tournament formats.

Writes:
  ./reports/usopen2025_R128_predictions.csv
"""

from __future__ import annotations

import os, re, unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ---------------- Config ----------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]

PREF_DATASETS = [
    "./data/match_dataset_post_cincinnati_2025.csv",               # preferred
    "./data/match_dataset_post_wimbledon_2025_canada.csv",
    "./data/match_dataset_post_wimbledon_2025.csv",
    "./atp_matches_2021_2024.csv",
    "/mnt/data/atp_matches_2021_2024.csv",
]

AS_OF_DATE = "2025-08-24"
TOURNEY_NAME = "US Open 2025"
SURFACE = "Hard"
TOURNEY_LEVEL = "G"
ROUND_CODE = "R128"
BEST_OF = 5

OUT_CSV = "./reports/usopen2025_R128_predictions.csv"
RANK_CLIP = 500  # cap diffs; 0 when either rank missing
SMOOTH_ALPHA = 5.0
SMOOTH_BETA  = 5.0

# ---------------- US Open 2025 – R128 pairs (exact bracket order) ----------------
USO_R128_PAIRS: List[Tuple[str, str]] = [
    ("Jannik Sinner", "Vit Kopriva"),
    ("Alexei Popyrin", "Emil Ruusuvuori"),
    ("Valentin Royer", "Bu Yunchaokete"),
    ("Marton Fucsovics", "Denis Shapovalov"),
    ("Alexander Bublik", "Marin Cilic"),
    ("Lorenzo Sonego", "Tristan Schoolkate"),
    ("Nuno Borges", "Brandon Holt"),
    ("Elmer Moller", "Tommy Paul"),
    ("Lorenzo Musetti", "Giovanni Mpetshi Perricard"),
    ("Quentin Halys", "David Goffin"),
    ("Jenson Brooksby", "Aleksandar Vukic"),
    ("Francesco Passaro", "Flavio Cobolli"),
    ("Gabriel Diallo", "Damir Dzumhur"),
    ("Jaume Munar", "Jaime Faria"),
    ("Zizou Bergs", "Chun-Hsin Tseng"),
    ("Federico Agustin Gomez", "Jack Draper"),
    ("Alexander Zverev", "Alejandro Tabilo"),
    ("Roberto Bautista Agut", "Jacob Fearnley"),
    ("Gael Monfils", "Roman Safiullin"),
    ("Billy Harris", "Felix Auger Aliassime"),
    ("Ugo Humbert", "Adam Walton"),
    ("Aleksandar Kovacevic", "Coleman Wong"),
    ("James Duckworth", "Tristan Boyer"),
    ("Dino Prizmic", "Andrey Rublev"),
    ("Karen Khachanov", "Nishesh Basavareddy"),
    ("Hugo Dellien", "Kamil Majchrzak"),
    ("Leandro Riedi", "Pedro Martinez"),
    ("Matteo Arnaldi", "Francisco Cerundolo"),
    ("Stefanos Tsitsipas", "Alexandre Muller"),
    ("Daniel Altmaier", "Hamad Medjedovic"),
    ("Hugo Gaston", "Shintaro Mochizuki"),
    ("Christopher O'Connell", "Alex De Minaur"),
    ("Novak Djokovic", "Learner Tien"),
    ("Zachary Svajda", "Zsombor Piros"),
    ("Cameron Norrie", "Sebastian Korda"),
    ("Francisco Comesana", "Alex Michelsen"),
    ("Frances Tiafoe", "Yoshihito Nishioka"),
    ("Martin Damm", "Darwin Blanch"),
    ("Jan-Lennard Struff", "Mackenzie McDonald"),
    ("Botic van de Zandschulp", "Holger Rune"),
    ("Jakub Mensik", "Nicolas Jarry"),
    ("Ugo Blanchet", "Fabian Marozsan"),
    ("Joao Fonseca", "Miomir Kecmanovic"),
    ("Luca Nardi", "Tomas Machac"),
    ("Brandon Nakashima", "Jesper De Jong"),
    ("Jerome Kym", "Ethan Quinn"),
    ("Sebastian Baez", "Lloyd Harris"),
    ("Emilio Nava", "Taylor Fritz"),
    ("Ben Shelton", "Ignacio Buse"),
    ("Pablo Carreno Busta", "Pablo Llamas Ruiz"),
    ("Jordan Thompson", "Corentin Moutet"),
    ("Adrian Mannarino", "Tallon Griekspoor"),
    ("Jiri Lehecka", "Borna Coric"),
    ("Camilo Ugo Carabelli", "Tomas Martin Etcheverry"),
    ("Daniel Elahi Galan", "Raphael Collignon"),
    ("Sebastian Ofner", "Casper Ruud"),
    ("Daniil Medvedev", "Benjamin Bonzi"),
    ("Mariano Navone", "Marcos Giron"),
    ("Roberto Carballes Baena", "Arthur Rinderknech"),
    ("Alexander Shevchenko", "Alejandro Davidovich Fokina"),
    ("Luciano Darderi", "Rinky Hijikata"),
    ("Stefan Dostanic", "Eliot Spizzirri"),
    ("Mattia Bellucci", "Juncheng Shang"),
    ("Reilly Opelka", "Carlos Alcaraz"),
]

# ---------------- Helpers (same as our Cincinnati/Wimbledon predictors) ----------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii","ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    need = ["tourney_date","tourney_name","surface","tourney_level","match_num",
            "winner_name","loser_name","round","best_of"]
    for c in need:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    BASE, K, KS = params.base, params.k_overall, params.k_surface
    elo_overall: Dict[str,float] = {}
    elo_surface: Dict[Tuple[str,str],float] = {}

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)
        exp_w   = 1.0 / (1.0 + 10 ** ((ew_l - ew_w) / 400))
        exp_w_s = 1.0 / (1.0 + 10 ** ((es_l - es_w) / 400))
        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({"player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
                     "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[w_rank] if w_rank else np.nan})
        rows.append({"player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
                     "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[l_rank] if l_rank else np.nan})

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon)
    log["opp_key"]    = log["opp"].map(canon)

    # Rest/workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0,60)

    # Rolling counts
    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out = []
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0)); dq.append(dt)
        return out
    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, True))

    grp = log.groupby("player_key")
    # Form (prior/post)
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)

    # Streaks
    def streak_prior(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.shift(1).fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    def streak_post(series: pd.Series) -> pd.Series:
        out = []; cur = 0
        for v in series.fillna(-1).astype(int):
            if v == 1: cur = cur + 1 if cur >= 0 else 1
            elif v == 0: cur = cur - 1 if cur <= 0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)

    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # Averages & smoothed WRs
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)

    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["wr_overall_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["wr_overall_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # Per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)

    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values
    tmp["wr_post_sm"]  = wr_post_sm.values

    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_prior.columns: surf_prior[srf] = np.nan
        if srf not in surf_post.columns:  surf_post[srf]  = np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)
    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # Peaks & rank
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)
    return log

def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str, float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"] == pk) & (log["date"] <= asof)].sort_values("date")
    snap = {"pre_elo":1500.0,"pre_selo":1500.0,"rest_days":30.0,"matches_28d":0.0,
            "rolling_10_winrate":0.5,"rolling_5_winrate":0.5,"streak":0.0,
            "avg_rest_days":20.0,"win_rate":0.5,"peak_elo":1500.0,"current_elo":1500.0,
            "wr_Clay":0.5,"wr_Grass":0.5,"wr_Hard":0.5,"rank_prior":np.nan,"rank_missing":True,
            "h2h_wr_prior":0.5}
    if not sub.empty:
        last = sub.iloc[-1]; after_last = asof > last["date"]
        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        snap.update({
            "pre_elo": float(pre_elo), "current_elo": float(pre_elo),
            "peak_elo": float(last["peak_elo_post"] if after_last else last["peak_elo_prior"]),
            "win_rate": float(last["wr_overall_post_sm"] if after_last else last["wr_overall_prior_sm"]),
            "avg_rest_days": float(last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]),
            "matches_28d": float(last.get("matches_28d_post" if after_last else "matches_28d_prior", 0.0)),
            "rolling_10_winrate": float(last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]),
            "rolling_5_winrate": float(last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]),
            "streak": float(last["streak_post"] if after_last else last["streak_prior"]),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
            "wr_Clay": float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"]),
            "wr_Grass": float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"]),
            "wr_Hard": float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"]),
        })
        sub_s = sub[sub["surface"] == target_surface]
        if not sub_s.empty:
            ls = sub_s.iloc[-1]
            snap["pre_selo"] = float(ls["post_selo"] if after_last else ls["pre_selo"])
    h2h = log[(log["player_key"] == pk) & (log["opp_key"] == ok) & (log["date"] <= asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1 + wins) / (2 + cnt)
    return snap

# ---------------- Main ----------------
def main() -> None:
    os.makedirs(os.path.dirname(OUT_CSV) or ".", exist_ok=True)

    # model
    model_path = _first_existing(MODEL_PATHS)
    if not model_path: raise FileNotFoundError("rf_model.joblib not found in ./models or /mnt/data/models")
    bundle = load(model_path)
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # dataset
    data_path = _first_existing(PREF_DATASETS)
    if not data_path: raise FileNotFoundError("No input dataset found.")
    matches = pd.read_csv(data_path)

    # logs
    log = build_player_log(matches, EloParams())
    asof = _safe_date(AS_OF_DATE)

    rows = []
    for i, (A, B) in enumerate(USO_R128_PAIRS, start=1):
        sa = snapshot_asof(log, A, B, asof, SURFACE)
        sb = snapshot_asof(log, B, A, asof, SURFACE)

        # model orientation alphabetical (p1/p2), but we keep bracket order via match_no
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)

        def d(k: str) -> float: return float(s1[k] - s2[k])

        feats = {
            "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_CODE, "best_of": BEST_OF,
            "elo_diff": d("pre_elo"), "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"), "diff_wr_Hard": d("wr_Hard"), "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"), "diff_current_elo": d("current_elo"), "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"), "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"), "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"), "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # robust rank diff
        if sa.get("rank_prior") is np.nan or sb.get("rank_prior") is np.nan or sa["rank_prior"] != sa["rank_prior"] or sb["rank_prior"] != sb["rank_prior"]:
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip((s1["rank_prior"] - s2["rank_prior"]), -RANK_CLIP, RANK_CLIP))

        # ensure all model features exist
        X = {c: feats.get(c, 0.0) for c in feature_cols}

        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:, 1][0])
        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        winner = A if prob_a >= 0.5 else B
        conf = max(prob_a, 1.0 - prob_a)

        # deltas winner - loser
        snap_w, snap_l = (sa, sb) if winner == A else (sb, sa)
        def delta(k: str) -> float: return float(snap_w[k] - snap_l[k])
        # rank delta
        if (snap_w.get("rank_prior") is np.nan or snap_l.get("rank_prior") is np.nan or
            snap_w["rank_prior"] != snap_w["rank_prior"] or snap_l["rank_prior"] != snap_l["rank_prior"]):
            delta_rank = 0.0
        else:
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i, "date": asof.strftime("%Y-%m-%d"), "round": ROUND_CODE,
            "player_a": A, "player_b": B, "pred_winner": winner, "correct_prediction": np.nan,
            "confidence": conf, "prob_player_a_win": prob_a, "prob_player_b_win": 1.0 - prob_a,
            "delta_elo": delta("pre_elo"), "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"), "delta_wr_Hard": delta("wr_Hard"), "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"), "delta_current_elo": delta("current_elo"), "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"), "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"), "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"), "delta_h2h_wr_prior": delta("h2h_wr_prior"), "delta_rank": delta_rank,
        })

    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    # Align to our standard header
    header = [
        "match_no","date","round","player_a","player_b","pred_winner","correct_prediction","confidence",
        "prob_player_a_win","prob_player_b_win",
        "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
        "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
        "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
    ]
    for c in header:
        if c not in out_df.columns: out_df[c] = pd.NA
    out_df = out_df[header].copy()

    os.makedirs(os.path.dirname(OUT_CSV) or ".", exist_ok=True)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] US Open R128 predictions → {OUT_CSV}")
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(out_df.head(12).to_string(index=False))

if __name__ == "__main__":
    main()


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] US Open R128 predictions → ./reports/usopen2025_R128_predictions.csv
 match_no       date round          player_a                   player_b      pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-24  R128     Jannik Sinner                Vit Kopriva    Jannik Sinner                 NaN    0.819774           0.819774           0.180226 646.102151         493.735388        0.300725       0.364262       0.261569        0.404640         646.102151      565.752726                5.0           -21.699966                  0.800000                 0.800000           6.0            0.000000         0.0
        2 2025-08-24  R128    Alexei Popyrin            Emil Ruusuvuori  Emil Ru

In [89]:
# filepath: ./predict_usopen_2025_r64.py
"""
US Open 2025 – Round of 64 predictions (deltas-only CSV).

Inputs:
  - ./reports/usopen2025_R128_predictions_complete.csv  (R128 with correct_prediction filled)
  - ./models/rf_model.joblib
  - Base dataset: prefers ./data/match_dataset_post_cincinnati_2025.csv

Outputs:
  - ./data/match_dataset_post_uso_r128_2025.csv
  - ./reports/usopen2025_R64_predictions.csv
"""

from __future__ import annotations

import os, re, unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ---------- Config ----------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]
PREF_DATASETS = [
    "./data/match_dataset_post_cincinnati_2025.csv",
    "./data/match_dataset_post_wimbledon_2025_canada.csv",
    "./data/match_dataset_post_wimbledon_2025.csv",
    "./atp_matches_2021_2024.csv",
    "/mnt/data/atp_matches_2021_2024.csv",
]

R128_COMPLETE = "./reports/usopen2025_R128_predictions_complete.csv"
OUT_DATASET   = "./data/match_dataset_post_uso_r128_2025.csv"
OUT_CSV       = "./reports/usopen2025_R64_predictions.csv"

AS_OF_DATE_R64 = "2025-08-29"   # change if your schedule differs
TOURNEY_NAME   = "US Open 2025"
SURFACE        = "Hard"
TOURNEY_LEVEL  = "G"
ROUND_R128     = "R128"
ROUND_R64      = "R64"
BEST_OF        = 5

RANK_CLIP      = 500
SMOOTH_ALPHA   = 5.0
SMOOTH_BETA    = 5.0

# ---------- Small alias map (why: harmonize common variants with history) ----------
ALIAS = {
    "Felix Auger-Aliassime": "Felix Auger Aliassime",
    "Alex de Minaur": "Alex De Minaur",
    "Botic Van De Zandschulp": "Botic van de Zandschulp",
    "Giovanni Mpetshi-Perricard": "Giovanni Mpetshi Perricard",
}
def alias(s: str) -> str:
    return ALIAS.get(s, s)

# ---------- Utils ----------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

# ---------- Elo/log ----------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    need = ["tourney_date","tourney_name","surface","tourney_level","match_num",
            "winner_name","loser_name","round","best_of"]
    for c in need:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    BASE, K, KS = params.base, params.k_overall, params.k_surface
    elo_overall: Dict[str,float] = {}
    elo_surface: Dict[Tuple[str,str],float] = {}

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = alias(r["winner_name"]); l = alias(r["loser_name"])
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0/(1.0+10**((ew_l-ew_w)/400))
        exp_w_s = 1.0/(1.0+10**((es_l-es_w)/400))

        new_ew_w = ew_w + K  * (1-exp_w)
        new_ew_l = ew_l - K  * (1-exp_w)
        new_es_w = es_w + KS * (1-exp_w_s)
        new_es_l = es_l - KS * (1-exp_w_s)

        rows.append({"player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
                     "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[w_rank] if w_rank else np.nan})
        rows.append({"player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
                     "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[l_rank] if l_rank else np.nan})

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon); log["opp_key"] = log["opp"].map(canon)

    # Rest/workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0,60)

    # Rolling counts
    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out=[]
        for dt in dates:
            while dq and (dt-dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0)); dq.append(dt)
        return out
    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, True))

    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5, min_periods=3).mean()).fillna(0.5).clip(0,1)

    # Streaks (why: we must capture true momentum post R128)
    def streak_prior(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.shift(1).fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    def streak_post(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # Averages & smoothed WRs
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["wr_overall_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["wr_overall_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # Per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)
    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values; tmp["wr_post_sm"] = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_prior.columns: surf_prior[srf]=np.nan
        if srf not in surf_post.columns:  surf_post[srf]=np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)
    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # Peaks & rank
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)
    return log

# ---------- R128 complete → dataset snapshot ----------
def load_r128_complete(path: str) -> pd.DataFrame:
    df = pd.read_csv(path).sort_values("match_no").reset_index(drop=True)
    need = ["match_no","player_a","player_b","pred_winner","correct_prediction","date"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {path}")
    if df["correct_prediction"].isna().any():
        bad = df[df["correct_prediction"].isna()]["match_no"].tolist()
        raise ValueError(f"'correct_prediction' has NaN for match_no={bad[:10]}… Fill 0/1 first.")
    # aliases for safety
    for col in ["player_a","player_b","pred_winner"]:
        df[col] = df[col].astype(str).map(alias)
    return df

def actual_winner_row(r: pd.Series) -> str:
    return r["pred_winner"] if int(r["correct_prediction"])==1 else (r["player_b"] if r["pred_winner"]==r["player_a"] else r["player_a"])

def append_usopen_r128_results(base: pd.DataFrame, r128: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for _, r in r128.iterrows():
        w = actual_winner_row(r); l = r["player_b"] if w==r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": TOURNEY_LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": alias(str(w)),
            "loser_name": alias(str(l)),
            "round": ROUND_R128,
            "best_of": BEST_OF,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    extra["surface"] = extra["surface"].map(normalize_surface)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common+add]], ignore_index=True, sort=False)

def winners_in_bracket_order(r128: pd.DataFrame) -> List[str]:
    r128 = r128.sort_values("match_no").reset_index(drop=True)
    return [actual_winner_row(r) for _, r in r128.iterrows()]

def r64_pairs_from_winners(winners: List[str]) -> List[Tuple[str,str]]:
    if len(winners) != 64:
        raise ValueError(f"Expected 64 winners from R128, got {len(winners)}")
    pairs=[]; 
    for i in range(0, 64, 2):
        pairs.append((alias(winners[i]), alias(winners[i+1])))
    return pairs

# ---------- Snapshot & predict ----------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str,float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"]==pk) & (log["date"]<=asof)].sort_values("date")
    snap = {"pre_elo":1500.0,"pre_selo":1500.0,"rest_days":30.0,"matches_28d":0.0,
            "rolling_10_winrate":0.5,"rolling_5_winrate":0.5,"streak":0.0,
            "avg_rest_days":20.0,"win_rate":0.5,"peak_elo":1500.0,"current_elo":1500.0,
            "wr_Clay":0.5,"wr_Grass":0.5,"wr_Hard":0.5,"rank_prior":np.nan,"h2h_wr_prior":0.5}
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]
        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        snap.update({
            "pre_elo": float(pre_elo), "current_elo": float(pre_elo),
            "peak_elo": float(last["peak_elo_post"] if after_last else last["peak_elo_prior"]),
            "win_rate": float(last["wr_overall_post_sm"] if after_last else last["wr_overall_prior_sm"]),
            "avg_rest_days": float(last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]),
            "matches_28d": float(last.get("matches_28d_post" if after_last else "matches_28d_prior", 0.0)),
            "rolling_10_winrate": float(last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]),
            "rolling_5_winrate": float(last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]),
            "streak": float(last["streak_post"] if after_last else last["streak_prior"]),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
            "wr_Clay": float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"]),
            "wr_Grass": float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"]),
            "wr_Hard": float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"]),
        })
        sub_s = sub[sub["surface"]==target_surface]
        if not sub_s.empty:
            ls = sub_s.iloc[-1]
            snap["pre_selo"] = float(ls["post_selo"] if asof > ls["date"] else ls["pre_selo"])
    h2h = log[(log["player_key"]==pk) & (log["opp_key"]==ok) & (log["date"]<=asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1+wins)/(2+cnt)
    return snap

# ---------- Main ----------
def main() -> None:
    # model
    model_path = _first_existing(MODEL_PATHS)
    if not model_path: raise FileNotFoundError("rf_model.joblib not found")
    bundle = load(model_path)
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # base dataset
    data_path = _first_existing(PREF_DATASETS)
    if not data_path: raise FileNotFoundError("No base dataset found.")
    base = pd.read_csv(data_path)

    # R128 actuals
    r128 = load_r128_complete(R128_COMPLETE)

    # Append R128 → dataset snapshot
    matches = append_usopen_r128_results(base, r128)
    _ensure_dir(OUT_DATASET)
    matches.to_csv(OUT_DATASET, index=False, encoding="utf-8-sig")
    print(f"[OK] Dataset with USO R128 appended → {OUT_DATASET} (rows: {len(matches)})")

    # Build log up to after R128
    log = build_player_log(matches, EloParams())

    # Build R64 pairs
    winners = winners_in_bracket_order(r128)
    pairs = r64_pairs_from_winners(winners)

    asof = _safe_date(AS_OF_DATE_R64)

    rows=[]
    for i, (A,B) in enumerate(pairs, start=1):
        sa = snapshot_asof(log, A, B, asof, SURFACE)
        sb = snapshot_asof(log, B, A, asof, SURFACE)

        # pipeline expects alphabetical orientation; preserve bracket order with match_no
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)
        def d(k): return float(s1[k]-s2[k])

        feats = {
            "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_R64, "best_of": BEST_OF,
            "elo_diff": d("pre_elo"), "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"), "diff_wr_Hard": d("wr_Hard"), "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"), "diff_current_elo": d("current_elo"), "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"), "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"), "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"), "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # rank handling (why: avoid huge artificial diffs when ranks missing)
        if pd.isna(sa.get("rank_prior")) or pd.isna(sb.get("rank_prior")):
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip((s1["rank_prior"] - s2["rank_prior"]), -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:,1][0])
        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        winner = A if prob_a >= 0.5 else B
        conf = max(prob_a, 1.0 - prob_a)

        snap_w, snap_l = (sa, sb) if winner == A else (sb, sa)
        def delta(k): return float(snap_w[k]-snap_l[k])
        if pd.isna(snap_w.get("rank_prior")) or pd.isna(snap_l.get("rank_prior")):
            delta_rank = 0.0
        else:
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i, "date": asof.strftime("%Y-%m-%d"), "round": ROUND_R64,
            "player_a": A, "player_b": B, "pred_winner": winner, "correct_prediction": np.nan,
            "confidence": conf, "prob_player_a_win": prob_a, "prob_player_b_win": 1.0 - prob_a,
            "delta_elo": delta("pre_elo"), "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"), "delta_wr_Hard": delta("wr_Hard"), "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"), "delta_current_elo": delta("current_elo"), "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"), "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"), "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"), "delta_h2h_wr_prior": delta("h2h_wr_prior"), "delta_rank": delta_rank,
        })

    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    header = [
        "match_no","date","round","player_a","player_b","pred_winner","correct_prediction", "confidence",
        "prob_player_a_win","prob_player_b_win",
        "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
        "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
        "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",   
    ]
    for c in header:
        if c not in out_df.columns: out_df[c] = pd.NA
    out_df = out_df[header]

    _ensure_dir(OUT_CSV)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] US Open R64 predictions → {OUT_CSV}")
    with pd.option_context("display.max_columns", None, "display.width", 170):
        print(out_df.head(10).to_string(index=False))

if __name__ == "__main__":
    main()


[OK] Dataset with USO R128 appended → ./data/match_dataset_post_uso_r128_2025.csv (rows: 12029)


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] US Open R64 predictions → ./reports/usopen2025_R64_predictions.csv
 match_no       date round         player_a              player_b           pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-29   R64    Jannik Sinner        Alexei Popyrin         Jannik Sinner                 NaN    0.835467           0.835467           0.164533 404.736742         340.591159        0.360248       0.278438       0.200345        0.292903         404.736742      417.190657                0.0            -3.759064                  0.300000                      0.2           0.0           -0.333333         0.0
        2 2025-08-29   R64   Valentin Royer      Denis Shapovalov      Denis Shapova

In [92]:
# filepath: ./predict_usopen_2025_r32.py
"""
US Open 2025 – Round of 32 predictions (deltas-only CSV).

Inputs:
  - ./reports/usopen2025_R64_predictions_complete.csv  (R64 with correct_prediction filled)
  - ./models/rf_model.joblib
  - Base dataset snapshot:
      prefers ./data/match_dataset_post_uso_r128_2025.csv
      then   ./data/match_dataset_post_cincinnati_2025.csv
      then   ./data/match_dataset_post_wimbledon_2025_canada.csv
      then   ./data/match_dataset_post_wimbledon_2025.csv
      then   ./atp_matches_2021_2024.csv

Outputs:
  - ./data/match_dataset_post_uso_r64_2025.csv
  - ./reports/usopen2025_R32_predictions.csv
"""

from __future__ import annotations

import os, re, unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ---------------- Config ----------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]

PREF_DATASETS = [
    "./data/match_dataset_post_uso_r128_2025.csv",           # preferred (after your R128 step)
    "./data/match_dataset_post_cincinnati_2025.csv",
    "./data/match_dataset_post_wimbledon_2025_canada.csv",
    "./data/match_dataset_post_wimbledon_2025.csv",
    "./atp_matches_2021_2024.csv",
    "/mnt/data/atp_matches_2021_2024.csv",
]

R64_COMPLETE = "./reports/usopen2025_R64_predictions_complete.csv"
OUT_DATASET   = "./data/match_dataset_post_uso_r64_2025.csv"
OUT_CSV       = "./reports/usopen2025_R32_predictions.csv"

# If None, we will auto-set to (max R64 date + 1 day)
AS_OF_DATE_R32: Optional[str] = None

TOURNEY_NAME   = "US Open 2025"
SURFACE        = "Hard"
TOURNEY_LEVEL  = "G"
ROUND_R64      = "R64"
ROUND_R32      = "R32"
BEST_OF        = 5

RANK_CLIP      = 500
SMOOTH_ALPHA   = 5.0
SMOOTH_BETA    = 5.0

# ---------------- Name alias map (harmonize with historical strings) ----------------
ALIAS = {
    "Felix Auger-Aliassime": "Felix Auger Aliassime",
    "Alex de Minaur": "Alex De Minaur",
    "Botic Van De Zandschulp": "Botic van de Zandschulp",
    "Giovanni Mpetshi-Perricard": "Giovanni Mpetshi Perricard",
}
def alias(s: str) -> str:
    return ALIAS.get(s, s)

# ---------------- Utils ----------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

# ---------------- Elo/log ----------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    need = ["tourney_date","tourney_name","surface","tourney_level","match_num",
            "winner_name","loser_name","round","best_of"]
    for c in need:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    BASE, K, KS = params.base, params.k_overall, params.k_surface
    elo_overall: Dict[str,float] = {}
    elo_surface: Dict[Tuple[str,str],float] = {}

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = alias(r["winner_name"]); l = alias(r["loser_name"])
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0/(1.0+10**((ew_l-ew_w)/400))
        exp_w_s = 1.0/(1.0+10**((es_l-es_w)/400))

        new_ew_w = ew_w + K  * (1-exp_w)
        new_ew_l = ew_l - K  * (1-exp_w)
        new_es_w = es_w + KS * (1-exp_w_s)
        new_es_l = es_l - KS * (1-exp_w_s)

        rows.append({"player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
                     "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[w_rank] if w_rank else np.nan})
        rows.append({"player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
                     "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[l_rank] if l_rank else np.nan})

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon); log["opp_key"] = log["opp"].map(canon)

    # Rest/workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0,60)

    # Rolling counts
    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out=[]
        for dt in dates:
            while dq and (dt-dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0)); dq.append(dt)
        return out
    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, True))

    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)

    # Streaks
    def streak_prior(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.shift(1).fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    def streak_post(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # Averages & smoothed WRs
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["wr_overall_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["wr_overall_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # Per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)
    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values; tmp["wr_post_sm"] = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_prior.columns: surf_prior[srf]=np.nan
        if srf not in surf_post.columns:  surf_post[srf]=np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)
    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # Peaks & rank
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)
    return log

# ---------------- R64 complete → dataset snapshot ----------------
def load_r64_complete(path: str) -> pd.DataFrame:
    df = pd.read_csv(path).sort_values("match_no").reset_index(drop=True)
    need = ["match_no","player_a","player_b","pred_winner","correct_prediction","date"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {path}")
    if df["correct_prediction"].isna().any():
        bad = df[df["correct_prediction"].isna()]["match_no"].tolist()
        raise ValueError(f"'correct_prediction' has NaN for match_no={bad[:10]}… Fill 0/1 first.")
    # normalize
    for col in ["player_a","player_b","pred_winner"]:
        df[col] = df[col].astype(str).map(alias)
    return df

def actual_winner_row(r: pd.Series) -> str:
    return r["pred_winner"] if int(r["correct_prediction"])==1 else (r["player_b"] if r["pred_winner"]==r["player_a"] else r["player_a"])

def append_usopen_r64_results(base: pd.DataFrame, r64: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for _, r in r64.iterrows():
        w = actual_winner_row(r); l = r["player_b"] if w==r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": TOURNEY_LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": alias(str(w)),
            "loser_name": alias(str(l)),
            "round": ROUND_R64,
            "best_of": BEST_OF,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    extra["surface"] = extra["surface"].map(normalize_surface)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common+add]], ignore_index=True, sort=False)

def winners_in_bracket_order(r64: pd.DataFrame) -> List[str]:
    r64 = r64.sort_values("match_no").reset_index(drop=True)
    return [actual_winner_row(r) for _, r in r64.iterrows()]

def r32_pairs_from_winners(winners: List[str]) -> List[Tuple[str,str]]:
    if len(winners) != 32:
        raise ValueError(f"Expected 32 winners from R64, got {len(winners)}")
    pairs=[]
    for i in range(0, 32, 2):
        pairs.append((alias(winners[i]), alias(winners[i+1])))
    return pairs

# ---------------- Snapshot & predict ----------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str,float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"]==pk) & (log["date"]<=asof)].sort_values("date")
    snap = {"pre_elo":1500.0,"pre_selo":1500.0,"rest_days":30.0,"matches_28d":0.0,
            "rolling_10_winrate":0.5,"rolling_5_winrate":0.5,"streak":0.0,
            "avg_rest_days":20.0,"win_rate":0.5,"peak_elo":1500.0,"current_elo":1500.0,
            "wr_Clay":0.5,"wr_Grass":0.5,"wr_Hard":0.5,"rank_prior":np.nan,"h2h_wr_prior":0.5}
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]
        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        snap.update({
            "pre_elo": float(pre_elo), "current_elo": float(pre_elo),
            "peak_elo": float(last["peak_elo_post"] if after_last else last["peak_elo_prior"]),
            "win_rate": float(last["wr_overall_post_sm"] if after_last else last["wr_overall_prior_sm"]),
            "avg_rest_days": float(last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]),
            "matches_28d": float(last.get("matches_28d_post" if after_last else "matches_28d_prior", 0.0)),
            "rolling_10_winrate": float(last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]),
            "rolling_5_winrate": float(last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]),
            "streak": float(last["streak_post"] if after_last else last["streak_prior"]),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
            "wr_Clay": float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"]),
            "wr_Grass": float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"]),
            "wr_Hard": float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"]),
        })
        sub_s = sub[sub["surface"]==target_surface]
        if not sub_s.empty:
            ls = sub_s.iloc[-1]
            snap["pre_selo"] = float(ls["post_selo"] if asof > ls["date"] else ls["pre_selo"])
    h2h = log[(log["player_key"]==pk) & (log["opp_key"]==ok) & (log["date"]<=asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1+wins)/(2+cnt)
    return snap

# ---------------- Main ----------------
def main() -> None:
    # model
    model_path = _first_existing(MODEL_PATHS)
    if not model_path: raise FileNotFoundError("rf_model.joblib not found")
    bundle = load(model_path)
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # base dataset
    data_path = _first_existing(PREF_DATASETS)
    if not data_path: raise FileNotFoundError("No base dataset found.")
    base = pd.read_csv(data_path)

    # R64 complete
    r64 = load_r64_complete(R64_COMPLETE)

    # Append R64 → dataset snapshot
    matches = append_usopen_r64_results(base, r64)
    _ensure_dir(OUT_DATASET)
    matches.to_csv(OUT_DATASET, index=False, encoding="utf-8-sig")
    print(f"[OK] Dataset with USO R64 appended → {OUT_DATASET} (rows: {len(matches)})")

    # Build log up to after R64
    log = build_player_log(matches, EloParams())

    # Build R32 pairs
    winners = winners_in_bracket_order(r64)
    pairs = r32_pairs_from_winners(winners)

    # as-of date
    if AS_OF_DATE_R32:
        asof = _safe_date(AS_OF_DATE_R32)
    else:
        # auto: (max R64 date + 1 day)
        max_r64 = pd.to_datetime(r64["date"]).max()
        asof = (max_r64 + pd.Timedelta(days=1)) if pd.notna(max_r64) else _safe_date("2025-09-01")

    rows=[]
    for i, (A,B) in enumerate(pairs, start=1):
        sa = snapshot_asof(log, A, B, asof, SURFACE)
        sb = snapshot_asof(log, B, A, asof, SURFACE)

        # pipeline orientation alphabetical; bracket order via match_no
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)
        def d(k): return float(s1[k]-s2[k])

        feats = {
            "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_R32, "best_of": BEST_OF,
            "elo_diff": d("pre_elo"), "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"), "diff_wr_Hard": d("wr_Hard"), "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"), "diff_current_elo": d("current_elo"), "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"), "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"), "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"), "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # rank handling
        if pd.isna(sa.get("rank_prior")) or pd.isna(sb.get("rank_prior")):
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip((s1["rank_prior"] - s2["rank_prior"]), -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:,1][0])
        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        winner = A if prob_a >= 0.5 else B
        conf = max(prob_a, 1.0 - prob_a)

        snap_w, snap_l = (sa, sb) if winner == A else (sb, sa)
        def delta(k): return float(snap_w[k]-snap_l[k])
        if pd.isna(snap_w.get("rank_prior")) or pd.isna(snap_l.get("rank_prior")):
            delta_rank = 0.0
        else:
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i, "date": asof.strftime("%Y-%m-%d"), "round": ROUND_R32,
            "player_a": A, "player_b": B, "pred_winner": winner, "correct_prediction": np.nan,
            "confidence": conf, "prob_player_a_win": prob_a, "prob_player_b_win": 1.0 - prob_a,
            "delta_elo": delta("pre_elo"), "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"), "delta_wr_Hard": delta("wr_Hard"), "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"), "delta_current_elo": delta("current_elo"), "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"), "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"), "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"), "delta_h2h_wr_prior": delta("h2h_wr_prior"), "delta_rank": delta_rank,
        })

    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    header = [
        "match_no","date","round","player_a","player_b","pred_winner", "correct_prediction", "confidence",
        "prob_player_a_win","prob_player_b_win",
        "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
        "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
        "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
    ]
    for c in header:
        if c not in out_df.columns: out_df[c] = pd.NA
    out_df = out_df[header]

    _ensure_dir(OUT_CSV)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] US Open R32 predictions → {OUT_CSV}")
    with pd.option_context("display.max_columns", None, "display.width", 170):
        print(out_df.head(10).to_string(index=False))

if __name__ == "__main__":
    main()


[OK] Dataset with USO R64 appended → ./data/match_dataset_post_uso_r64_2025.csv (rows: 12061)


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] US Open R32 predictions → ./reports/usopen2025_R32_predictions.csv
 match_no       date round         player_a              player_b      pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-30   R32    Jannik Sinner      Denis Shapovalov    Jannik Sinner                 NaN    0.857291           0.857291           0.142709 479.135656         401.596477        0.167391       0.252023       0.190141        0.241158         479.135656      448.216778                5.0            -2.385489                       0.3                      0.4           0.0           -0.333333         0.0
        2 2025-08-30   R32 Alexander Bublik            Tommy Paul       Tommy Paul            

In [94]:
# filepath: ./predict_usopen_2025_r16.py
"""
US Open 2025 – Round of 16 predictions (deltas-only CSV).

Inputs:
  - ./reports/usopen2025_R32_predictions_complete.csv (R32 with correct_prediction filled 0/1)
  - ./models/rf_model.joblib
  - Base dataset snapshot (prefer later):
      ./data/match_dataset_post_uso_r64_2025.csv
      ./data/match_dataset_post_uso_r128_2025.csv
      ./data/match_dataset_post_cincinnati_2025.csv
      ./data/match_dataset_post_wimbledon_2025_canada.csv
      ./data/match_dataset_post_wimbledon_2025.csv
      ./atp_matches_2021_2024.csv

Outputs:
  - ./data/match_dataset_post_uso_r32_2025.csv
  - ./reports/usopen2025_R16_predictions.csv
"""

from __future__ import annotations

import os, re, unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ---------------- Config ----------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]

PREF_DATASETS = [
    "./data/match_dataset_post_uso_r64_2025.csv",
    "./data/match_dataset_post_uso_r128_2025.csv",
    "./data/match_dataset_post_cincinnati_2025.csv",
    "./data/match_dataset_post_wimbledon_2025_canada.csv",
    "./data/match_dataset_post_wimbledon_2025.csv",
    "./atp_matches_2021_2024.csv",
    "/mnt/data/atp_matches_2021_2024.csv",
]

R32_COMPLETE = "./reports/usopen2025_R32_predictions_complete.csv"
OUT_DATASET   = "./data/match_dataset_post_uso_r32_2025.csv"
OUT_CSV       = "./reports/usopen2025_R16_predictions.csv"

# If None, auto = (max R32 date + 1 day)
AS_OF_DATE_R16: Optional[str] = None

TOURNEY_NAME   = "US Open 2025"
SURFACE        = "Hard"
TOURNEY_LEVEL  = "G"
ROUND_R32      = "R32"
ROUND_R16      = "R16"
BEST_OF        = 5

RANK_CLIP      = 500
SMOOTH_ALPHA   = 5.0
SMOOTH_BETA    = 5.0

# ---------------- Name aliases (harmonize with historical strings) ----------------
ALIAS = {
    "Felix Auger-Aliassime": "Felix Auger Aliassime",
    "Alex de Minaur": "Alex De Minaur",
    "Botic Van De Zandschulp": "Botic van de Zandschulp",
    "Giovanni Mpetshi-Perricard": "Giovanni Mpetshi Perricard",
}
def alias(s: str) -> str:
    return ALIAS.get(s, s)

# ---------------- Utils ----------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

# ---------------- Elo/log ----------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    need = ["tourney_date","tourney_name","surface","tourney_level","match_num",
            "winner_name","loser_name","round","best_of"]
    for c in need:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    BASE, K, KS = params.base, params.k_overall, params.k_surface
    elo_overall: Dict[str,float] = {}
    elo_surface: Dict[Tuple[str,str],float] = {}

    rows: List[Dict] = []
    for _, r in m.iterrows():
      d = r["tourney_date"]; s = r["surface"]
      w = alias(r["winner_name"]); l = alias(r["loser_name"])
      ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
      es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

      exp_w   = 1.0/(1.0+10**((ew_l-ew_w)/400))
      exp_w_s = 1.0/(1.0+10**((es_l-es_w)/400))

      new_ew_w = ew_w + K  * (1-exp_w)
      new_ew_l = ew_l - K  * (1-exp_w)
      new_es_w = es_w + KS * (1-exp_w_s)
      new_es_l = es_l - KS * (1-exp_w_s)

      rows.append({"player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
                   "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
                   "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                   "rank": r[w_rank] if w_rank else np.nan})
      rows.append({"player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
                   "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
                   "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                   "rank": r[l_rank] if l_rank else np.nan})

      elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
      elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon); log["opp_key"] = log["opp"].map(canon)

    # Rest/workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0,60)

    # Rolling counts (prior/post)
    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out=[]
        for dt in dates:
            while dq and (dt-dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0)); dq.append(dt)
        return out
    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, True))

    grp = log.groupby("player_key")
    # Form (prior/post)
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)

    # Streaks (post captures current momentum)
    def streak_prior(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.shift(1).fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    def streak_post(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # Averages & smoothed WRs
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["wr_overall_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["wr_overall_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # Per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)
    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values; tmp["wr_post_sm"] = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_prior.columns: surf_prior[srf]=np.nan
        if srf not in surf_post.columns:  surf_post[srf]=np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)
    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # Peaks & rank
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)
    return log

# ---------------- R32 complete → dataset snapshot ----------------
def load_r32_complete(path: str) -> pd.DataFrame:
    df = pd.read_csv(path).sort_values("match_no").reset_index(drop=True)
    need = ["match_no","player_a","player_b","pred_winner","correct_prediction","date"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {path}")
    if df["correct_prediction"].isna().any():
        bad = df[df["correct_prediction"].isna()]["match_no"].tolist()
        raise ValueError(f"'correct_prediction' has NaN for match_no={bad[:10]}… Fill 0/1 first.")
    for col in ["player_a","player_b","pred_winner"]:
        df[col] = df[col].astype(str).map(alias)
    return df

def actual_winner_row(r: pd.Series) -> str:
    return r["pred_winner"] if int(r["correct_prediction"])==1 else (r["player_b"] if r["pred_winner"]==r["player_a"] else r["player_a"])

def append_usopen_r32_results(base: pd.DataFrame, r32: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for _, r in r32.iterrows():
        w = actual_winner_row(r); l = r["player_b"] if w==r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": TOURNEY_LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": alias(str(w)),
            "loser_name": alias(str(l)),
            "round": ROUND_R32,
            "best_of": BEST_OF,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    extra["surface"] = extra["surface"].map(normalize_surface)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common+add]], ignore_index=True, sort=False)

def winners_in_bracket_order(r32: pd.DataFrame) -> List[str]:
    r32 = r32.sort_values("match_no").reset_index(drop=True)
    return [actual_winner_row(r) for _, r in r32.iterrows()]

def r16_pairs_from_winners(winners: List[str]) -> List[Tuple[str,str]]:
    if len(winners) != 16:
        raise ValueError(f"Expected 16 winners from R32, got {len(winners)}")
    pairs=[]
    for i in range(0, 16, 2):
        pairs.append((alias(winners[i]), alias(winners[i+1])))
    return pairs

# ---------------- Snapshot & predict ----------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str,float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"]==pk) & (log["date"]<=asof)].sort_values("date")
    snap = {"pre_elo":1500.0,"pre_selo":1500.0,"rest_days":30.0,"matches_28d":0.0,
            "rolling_10_winrate":0.5,"rolling_5_winrate":0.5,"streak":0.0,
            "avg_rest_days":20.0,"win_rate":0.5,"peak_elo":1500.0,"current_elo":1500.0,
            "wr_Clay":0.5,"wr_Grass":0.5,"wr_Hard":0.5,"rank_prior":np.nan,"h2h_wr_prior":0.5}
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]
        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        snap.update({
            "pre_elo": float(pre_elo), "current_elo": float(pre_elo),
            "peak_elo": float(last["peak_elo_post"] if after_last else last["peak_elo_prior"]),
            "win_rate": float(last["wr_overall_post_sm"] if after_last else last["wr_overall_prior_sm"]),
            "avg_rest_days": float(last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]),
            "matches_28d": float(last.get("matches_28d_post" if after_last else "matches_28d_prior", 0.0)),
            "rolling_10_winrate": float(last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]),
            "rolling_5_winrate": float(last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]),
            "streak": float(last["streak_post"] if after_last else last["streak_prior"]),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
            "wr_Clay": float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"]),
            "wr_Grass": float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"]),
            "wr_Hard": float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"]),
        })
        sub_s = sub[sub["surface"]==target_surface]
        if not sub_s.empty:
            ls = sub_s.iloc[-1]
            snap["pre_selo"] = float(ls["post_selo"] if asof > ls["date"] else ls["pre_selo"])
    h2h = log[(log["player_key"]==pk) & (log["opp_key"]==ok) & (log["date"]<=asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1+wins)/(2+cnt)
    return snap

# ---------------- Main ----------------
def main() -> None:
    # model
    model_path = _first_existing(MODEL_PATHS)
    if not model_path: raise FileNotFoundError("rf_model.joblib not found")
    bundle = load(model_path)
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # base dataset
    data_path = _first_existing(PREF_DATASETS)
    if not data_path: raise FileNotFoundError("No base dataset found.")
    base = pd.read_csv(data_path)

    # R32 complete
    r32 = load_r32_complete(R32_COMPLETE)

    # Append R32 → dataset snapshot
    matches = append_usopen_r32_results(base, r32)
    _ensure_dir(OUT_DATASET)
    matches.to_csv(OUT_DATASET, index=False, encoding="utf-8-sig")
    print(f"[OK] Dataset with USO R32 appended → {OUT_DATASET} (rows: {len(matches)})")

    # Build log up to after R32
    log = build_player_log(matches, EloParams())

    # Build R16 pairs from winners (bracket order)
    winners = winners_in_bracket_order(r32)
    pairs = r16_pairs_from_winners(winners)

    # as-of date
    if AS_OF_DATE_R16:
        asof = _safe_date(AS_OF_DATE_R16)
    else:
        max_r32 = pd.to_datetime(r32["date"]).max()
        asof = (max_r32 + pd.Timedelta(days=1)) if pd.notna(max_r32) else _safe_date("2025-09-02")

    rows=[]
    for i, (A,B) in enumerate(pairs, start=1):
        sa = snapshot_asof(log, A, B, asof, SURFACE)
        sb = snapshot_asof(log, B, A, asof, SURFACE)

        # model orientation alphabetical; bracket order via match_no
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)
        def d(k): return float(s1[k]-s2[k])

        feats = {
            "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_R16, "best_of": BEST_OF,
            "elo_diff": d("pre_elo"), "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"), "diff_wr_Hard": d("wr_Hard"), "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"), "diff_current_elo": d("current_elo"), "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"), "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"), "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"), "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # rank robust
        if pd.isna(sa.get("rank_prior")) or pd.isna(sb.get("rank_prior")):
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip((s1["rank_prior"] - s2["rank_prior"]), -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:,1][0])
        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        winner = A if prob_a >= 0.5 else B
        conf = max(prob_a, 1.0 - prob_a)

        snap_w, snap_l = (sa, sb) if winner == A else (sb, sa)
        def delta(k): return float(snap_w[k]-snap_l[k])
        if pd.isna(snap_w.get("rank_prior")) or pd.isna(snap_l.get("rank_prior")):
            delta_rank = 0.0
        else:
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i, "date": asof.strftime("%Y-%m-%d"), "round": ROUND_R16,
            "player_a": A, "player_b": B, "pred_winner": winner, "correct_prediction": np.nan,
            "confidence": conf, "prob_player_a_win": prob_a, "prob_player_b_win": 1.0 - prob_a,
            "delta_elo": delta("pre_elo"), "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"), "delta_wr_Hard": delta("wr_Hard"), "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"), "delta_current_elo": delta("current_elo"), "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"), "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"), "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"), "delta_h2h_wr_prior": delta("h2h_wr_prior"), "delta_rank": delta_rank,
        })

    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    header = [
        "match_no","date","round","player_a","player_b","pred_winner", "correct_prediction", "confidence",
        "prob_player_a_win","prob_player_b_win",
        "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
        "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
        "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
    ]
    for c in header:
        if c not in out_df.columns: out_df[c] = pd.NA
    out_df = out_df[header]

    _ensure_dir(OUT_CSV)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] US Open R16 predictions → {OUT_CSV}")
    with pd.option_context("display.max_columns", None, "display.width", 170):
        print(out_df.head(10).to_string(index=False))

if __name__ == "__main__":
    main()


[OK] Dataset with USO R32 appended → ./data/match_dataset_post_uso_r32_2025.csv (rows: 12077)


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] US Open R16 predictions → ./reports/usopen2025_R16_predictions.csv
 match_no       date round              player_a           player_b     pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-08-31   R16         Jannik Sinner   Alexander Bublik   Jannik Sinner                 NaN    0.852873           0.852873           0.147127 521.980704         425.622977        0.099209       0.284867       0.286295        0.267261         521.980704      414.701791                6.0            -1.794461                       0.5                      0.2           0.0            0.333333         0.0
        2 2025-08-31   R16       Lorenzo Musetti        Jaume Munar Lorenzo Musetti         

In [97]:
# filepath: ./predict_usopen_2025_qf.py
"""
US Open 2025 – Quarterfinal predictions (deltas-only CSV).

Inputs:
  - ./reports/usopen2025_R16_predictions_complete.csv  (R16 with correct_prediction filled 0/1)
  - ./models/rf_model.joblib
  - Base dataset snapshot (prefer later):
      ./data/match_dataset_post_uso_r32_2025.csv
      ./data/match_dataset_post_uso_r64_2025.csv
      ./data/match_dataset_post_uso_r128_2025.csv
      ./data/match_dataset_post_cincinnati_2025.csv
      ./data/match_dataset_post_wimbledon_2025_canada.csv
      ./data/match_dataset_post_wimbledon_2025.csv
      ./atp_matches_2021_2024.csv

Outputs:
  - ./data/match_dataset_post_uso_r16_2025.csv
  - ./reports/usopen2025_QF_predictions.csv
"""

from __future__ import annotations

import os, re, unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ---------------- Config ----------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]

PREF_DATASETS = [
    "./data/match_dataset_post_uso_r32_2025.csv",
    "./data/match_dataset_post_uso_r64_2025.csv",
    "./data/match_dataset_post_uso_r128_2025.csv",
    "./data/match_dataset_post_cincinnati_2025.csv",
    "./data/match_dataset_post_wimbledon_2025_canada.csv",
    "./data/match_dataset_post_wimbledon_2025.csv",
    "./atp_matches_2021_2024.csv",
    "/mnt/data/atp_matches_2021_2024.csv",
]

R16_COMPLETE = "./reports/usopen2025_R16_predictions_complete.csv"
OUT_DATASET   = "./data/match_dataset_post_uso_r16_2025.csv"
OUT_CSV       = "./reports/usopen2025_QF_predictions.csv"

# If None, auto = (max R16 date + 1 day)
AS_OF_DATE_QF: Optional[str] = "2025-09-02"

TOURNEY_NAME   = "US Open 2025"
SURFACE        = "Hard"
TOURNEY_LEVEL  = "G"
ROUND_R16      = "R16"
ROUND_QF       = "QF"
BEST_OF        = 5

RANK_CLIP      = 500
SMOOTH_ALPHA   = 5.0
SMOOTH_BETA    = 5.0

# ---------------- Name aliases (harmonize with historical strings) ----------------
ALIAS = {
    "Felix Auger-Aliassime": "Felix Auger Aliassime",
    "Alex de Minaur": "Alex De Minaur",
    "Botic Van De Zandschulp": "Botic van de Zandschulp",
    "Giovanni Mpetshi-Perricard": "Giovanni Mpetshi Perricard",
}
def alias(s: str) -> str:
    return ALIAS.get(s, s)

# ---------------- Utils ----------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii","ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

# ---------------- Elo/log ----------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    need = ["tourney_date","tourney_name","surface","tourney_level","match_num",
            "winner_name","loser_name","round","best_of"]
    for c in need:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    BASE, K, KS = params.base, params.k_overall, params.k_surface
    elo_overall: Dict[str,float] = {}
    elo_surface: Dict[Tuple[str,str],float] = {}

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = alias(r["winner_name"]); l = alias(r["loser_name"])
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0/(1.0+10**((ew_l-ew_w)/400))
        exp_w_s = 1.0/(1.0+10**((es_l-es_w)/400))

        new_ew_w = ew_w + K  * (1-exp_w)
        new_ew_l = ew_l - K  * (1-exp_w)
        new_es_w = es_w + KS * (1-exp_w_s)
        new_es_l = es_l - KS * (1-exp_w_s)

        rows.append({"player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
                     "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[w_rank] if w_rank else np.nan})
        rows.append({"player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
                     "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[l_rank] if l_rank else np.nan})

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon); log["opp_key"] = log["opp"].map(canon)

    # Rest/workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0,60)

    # Rolling counts (prior/post)
    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out=[]
        for dt in dates:
            while dq and (dt-dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0)); dq.append(dt)
        return out
    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, True))

    grp = log.groupby("player_key")
    # Form (prior/post)
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)

    # Streaks
    def streak_prior(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.shift(1).fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    def streak_post(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # Averages & smoothed WRs
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["wr_overall_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["wr_overall_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # Per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)
    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values; tmp["wr_post_sm"] = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_prior.columns: surf_prior[srf]=np.nan
        if srf not in surf_post.columns:  surf_post[srf]=np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)
    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # Peaks & rank
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)
    return log

# ---------------- R16 complete → dataset snapshot ----------------
def load_r16_complete(path: str) -> pd.DataFrame:
    df = pd.read_csv(path).sort_values("match_no").reset_index(drop=True)
    need = ["match_no","player_a","player_b","pred_winner","correct_prediction","date"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {path}")
    if df["correct_prediction"].isna().any():
        bad = df[df["correct_prediction"].isna()]["match_no"].tolist()
        raise ValueError(f"'correct_prediction' has NaN for match_no={bad[:10]}… Fill 0/1 first.")
    for col in ["player_a","player_b","pred_winner"]:
        df[col] = df[col].astype(str).map(alias)
    return df

def actual_winner_row(r: pd.Series) -> str:
    return r["pred_winner"] if int(r["correct_prediction"])==1 else (r["player_b"] if r["pred_winner"]==r["player_a"] else r["player_a"])

def append_usopen_r16_results(base: pd.DataFrame, r16: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for _, r in r16.iterrows():
        w = actual_winner_row(r); l = r["player_b"] if w==r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": TOURNEY_LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": alias(str(w)),
            "loser_name": alias(str(l)),
            "round": ROUND_R16,
            "best_of": BEST_OF,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    extra["surface"] = extra["surface"].map(normalize_surface)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common+add]], ignore_index=True, sort=False)

def winners_in_bracket_order(r16: pd.DataFrame) -> List[str]:
    r16 = r16.sort_values("match_no").reset_index(drop=True)
    return [actual_winner_row(r) for _, r in r16.iterrows()]

def qf_pairs_from_winners(winners: List[str]) -> List[Tuple[str,str]]:
    if len(winners) != 8:
        raise ValueError(f"Expected 8 winners from R16, got {len(winners)}")
    pairs=[]
    for i in range(0, 8, 2):
        pairs.append((alias(winners[i]), alias(winners[i+1])))
    return pairs

# ---------------- Snapshot & predict ----------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str,float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"]==pk) & (log["date"]<=asof)].sort_values("date")
    snap = {"pre_elo":1500.0,"pre_selo":1500.0,"rest_days":30.0,"matches_28d":0.0,
            "rolling_10_winrate":0.5,"rolling_5_winrate":0.5,"streak":0.0,
            "avg_rest_days":20.0,"win_rate":0.5,"peak_elo":1500.0,"current_elo":1500.0,
            "wr_Clay":0.5,"wr_Grass":0.5,"wr_Hard":0.5,"rank_prior":np.nan,"h2h_wr_prior":0.5}
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]
        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        snap.update({
            "pre_elo": float(pre_elo), "current_elo": float(pre_elo),
            "peak_elo": float(last["peak_elo_post"] if after_last else last["peak_elo_prior"]),
            "win_rate": float(last["wr_overall_post_sm"] if after_last else last["wr_overall_prior_sm"]),
            "avg_rest_days": float(last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]),
            "matches_28d": float(last.get("matches_28d_post" if after_last else "matches_28d_prior", 0.0)),
            "rolling_10_winrate": float(last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]),
            "rolling_5_winrate": float(last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]),
            "streak": float(last["streak_post"] if after_last else last["streak_prior"]),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
            "wr_Clay": float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"]),
            "wr_Grass": float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"]),
            "wr_Hard": float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"]),
        })
        sub_s = sub[sub["surface"]==target_surface]
        if not sub_s.empty:
            ls = sub_s.iloc[-1]
            snap["pre_selo"] = float(ls["post_selo"] if asof > ls["date"] else ls["pre_selo"])
    h2h = log[(log["player_key"]==pk) & (log["opp_key"]==ok) & (log["date"]<=asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1+wins)/(2+cnt)
    return snap

# ---------------- Main ----------------
def main() -> None:
    # model
    model_path = _first_existing(MODEL_PATHS)
    if not model_path: raise FileNotFoundError("rf_model.joblib not found")
    bundle = load(model_path)
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # base dataset
    data_path = _first_existing(PREF_DATASETS)
    if not data_path: raise FileNotFoundError("No base dataset found.")
    base = pd.read_csv(data_path)

    # R16 complete
    r16 = load_r16_complete(R16_COMPLETE)

    # Append R16 → dataset snapshot
    matches = append_usopen_r16_results(base, r16)
    _ensure_dir(OUT_DATASET)
    matches.to_csv(OUT_DATASET, index=False, encoding="utf-8-sig")
    print(f"[OK] Dataset with USO R16 appended → {OUT_DATASET} (rows: {len(matches)})")

    # Build log up to after R16
    log = build_player_log(matches, EloParams())

    # Build QF pairs from winners (bracket order)
    winners = winners_in_bracket_order(r16)
    pairs = qf_pairs_from_winners(winners)

    # as-of date
    if AS_OF_DATE_QF:
        asof = _safe_date(AS_OF_DATE_QF)
    else:
        max_r16 = pd.to_datetime(r16["date"]).max()
        asof = (max_r16 + pd.Timedelta(days=1)) if pd.notna(max_r16) else _safe_date("2025-09-03")

    rows=[]
    for i, (A,B) in enumerate(pairs, start=1):
        sa = snapshot_asof(log, A, B, asof, SURFACE)
        sb = snapshot_asof(log, B, A, asof, SURFACE)

        # pipeline orientation alphabetical; bracket order via match_no
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)
        def d(k): return float(s1[k]-s2[k])

        feats = {
            "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_QF, "best_of": BEST_OF,
            "elo_diff": d("pre_elo"), "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"), "diff_wr_Hard": d("wr_Hard"), "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"), "diff_current_elo": d("current_elo"), "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"), "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"), "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"), "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # rank robust
        if pd.isna(sa.get("rank_prior")) or pd.isna(sb.get("rank_prior")):
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip((s1["rank_prior"] - s2["rank_prior"]), -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:,1][0])
        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        winner = A if prob_a >= 0.5 else B
        conf = max(prob_a, 1.0 - prob_a)

        snap_w, snap_l = (sa, sb) if winner == A else (sb, sa)
        def delta(k): return float(snap_w[k]-snap_l[k])
        if pd.isna(snap_w.get("rank_prior")) or pd.isna(snap_l.get("rank_prior")):
            delta_rank = 0.0
        else:
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i, "date": asof.strftime("%Y-%m-%d"), "round": ROUND_QF,
            "player_a": A, "player_b": B, "pred_winner": winner, "correct_prediction": np.nan,
            "confidence": conf, "prob_player_a_win": prob_a, "prob_player_b_win": 1.0 - prob_a,
            "delta_elo": delta("pre_elo"), "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"), "delta_wr_Hard": delta("wr_Hard"), "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"), "delta_current_elo": delta("current_elo"), "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"), "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"), "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"), "delta_h2h_wr_prior": delta("h2h_wr_prior"), "delta_rank": delta_rank,
        })

    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    header = [
        "match_no","date","round","player_a","player_b","pred_winner", "correct_prediction", "confidence",
        "prob_player_a_win","prob_player_b_win",
        "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
        "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
        "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
    ]
    for c in header:
        if c not in out_df.columns: out_df[c] = pd.NA
    out_df = out_df[header]

    _ensure_dir(OUT_CSV)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] US Open QF predictions → {OUT_CSV}")
    with pd.option_context("display.max_columns", None, "display.width", 170):
        print(out_df.head(10).to_string(index=False))

if __name__ == "__main__":
    main()


[OK] Dataset with USO R16 appended → ./data/match_dataset_post_uso_r16_2025.csv (rows: 12085)


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] US Open QF predictions → ./reports/usopen2025_QF_predictions.csv
 match_no       date round              player_a        player_b           pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-09-02    QF         Jannik Sinner Lorenzo Musetti         Jannik Sinner                 NaN    0.852555           0.852555           0.147445 408.245585         427.349429        0.127648       0.319688       0.088100        0.241487         408.245585      356.100244                5.0            -1.271518                       0.4                      0.0           0.0            0.200000         0.0
        2 2025-09-02    QF Felix Auger Aliassime  Alex De Minaur Felix Auger Aliassime  

In [98]:
# filepath: ./predict_usopen_2025_sf.py
"""
US Open 2025 – Semifinal predictions (deltas-only CSV).

Inputs:
  - ./reports/usopen2025_QF_predictions_complete.csv  (QF with correct_prediction filled 0/1)
  - ./models/rf_model.joblib
  - Base dataset snapshot (prefer later):
      ./data/match_dataset_post_uso_r16_2025.csv
      ./data/match_dataset_post_uso_r32_2025.csv
      ./data/match_dataset_post_uso_r64_2025.csv
      ./data/match_dataset_post_uso_r128_2025.csv
      ./data/match_dataset_post_cincinnati_2025.csv
      ./data/match_dataset_post_wimbledon_2025_canada.csv
      ./data/match_dataset_post_wimbledon_2025.csv
      ./atp_matches_2021_2024.csv

Outputs:
  - ./data/match_dataset_post_uso_qf_2025.csv
  - ./reports/usopen2025_SF_predictions.csv
"""

from __future__ import annotations

import os, re, unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ---------------- Config ----------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]

PREF_DATASETS = [
    "./data/match_dataset_post_uso_r16_2025.csv",
    "./data/match_dataset_post_uso_r32_2025.csv",
    "./data/match_dataset_post_uso_r64_2025.csv",
    "./data/match_dataset_post_uso_r128_2025.csv",
    "./data/match_dataset_post_cincinnati_2025.csv",
    "./data/match_dataset_post_wimbledon_2025_canada.csv",
    "./data/match_dataset_post_wimbledon_2025.csv",
    "./atp_matches_2021_2024.csv",
    "/mnt/data/atp_matches_2021_2024.csv",
]

QF_COMPLETE = "./reports/usopen2025_QF_predictions_complete.csv"
OUT_DATASET  = "./data/match_dataset_post_uso_qf_2025.csv"
OUT_CSV      = "./reports/usopen2025_SF_predictions.csv"

# If None, auto = (max QF date + 1 day)
AS_OF_DATE_SF: Optional[str] = "2025-09-05"

TOURNEY_NAME   = "US Open 2025"
SURFACE        = "Hard"
TOURNEY_LEVEL  = "G"
ROUND_QF       = "QF"
ROUND_SF       = "SF"
BEST_OF        = 5

RANK_CLIP      = 500
SMOOTH_ALPHA   = 5.0
SMOOTH_BETA    = 5.0

# ---------------- Name aliases (why: harmonize with historical strings) ----------------
ALIAS = {
    "Felix Auger-Aliassime": "Felix Auger Aliassime",
    "Alex de Minaur": "Alex De Minaur",
    "Botic Van De Zandschulp": "Botic van de Zandschulp",
    "Giovanni Mpetshi-Perricard": "Giovanni Mpetshi Perricard",
}
def alias(s: str) -> str:
    return ALIAS.get(s, s)

# ---------------- Utils ----------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

# ---------------- Elo/log ----------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    need = ["tourney_date","tourney_name","surface","tourney_level","match_num",
            "winner_name","loser_name","round","best_of"]
    for c in need:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    BASE, K, KS = params.base, params.k_overall, params.k_surface
    elo_overall: Dict[str,float] = {}
    elo_surface: Dict[Tuple[str,str],float] = {}

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = alias(r["winner_name"]); l = alias(r["loser_name"])
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0/(1.0+10**((ew_l-ew_w)/400))
        exp_w_s = 1.0/(1.0+10**((es_l-es_w)/400))

        new_ew_w = ew_w + K  * (1-exp_w)
        new_ew_l = ew_l - K  * (1-exp_w)
        new_es_w = es_w + KS * (1-exp_w_s)
        new_es_l = es_l - KS * (1-exp_w_s)

        rows.append({"player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
                     "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[w_rank] if w_rank else np.nan})
        rows.append({"player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
                     "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[l_rank] if l_rank else np.nan})

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon); log["opp_key"] = log["opp"].map(canon)

    # Rest/workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0,60)

    # Rolling counts (prior/post)
    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out=[]
        for dt in dates:
            while dq and (dt-dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0)); dq.append(dt)
        return out
    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, True))

    grp = log.groupby("player_key")
    # Form (prior/post)
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)

    # Streaks
    def streak_prior(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.shift(1).fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    def streak_post(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    # Averages & smoothed WRs
    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["wr_overall_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["wr_overall_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    # Per-surface smoothed WRs
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)
    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values; tmp["wr_post_sm"] = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_prior.columns: surf_prior[srf]=np.nan
        if srf not in surf_post.columns:  surf_post[srf]=np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)
    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    # Peaks & rank
    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)
    return log

# ---------------- QF complete → dataset snapshot ----------------
def load_qf_complete(path: str) -> pd.DataFrame:
    df = pd.read_csv(path).sort_values("match_no").reset_index(drop=True)
    need = ["match_no","player_a","player_b","pred_winner","correct_prediction","date"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {path}")
    if df["correct_prediction"].isna().any():
        bad = df[df["correct_prediction"].isna()]["match_no"].tolist()
        raise ValueError(f"'correct_prediction' has NaN for match_no={bad[:10]}… Fill 0/1 first.")
    for col in ["player_a","player_b","pred_winner"]:
        df[col] = df[col].astype(str).map(alias)
    return df

def actual_winner_row(r: pd.Series) -> str:
    return r["pred_winner"] if int(r["correct_prediction"])==1 else (r["player_b"] if r["pred_winner"]==r["player_a"] else r["player_a"])

def append_usopen_qf_results(base: pd.DataFrame, qf: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for _, r in qf.iterrows():
        w = actual_winner_row(r); l = r["player_b"] if w==r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": TOURNEY_LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": alias(str(w)),
            "loser_name": alias(str(l)),
            "round": ROUND_QF,
            "best_of": BEST_OF,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    extra["surface"] = extra["surface"].map(normalize_surface)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common+add]], ignore_index=True, sort=False)

def winners_in_bracket_order(qf: pd.DataFrame) -> List[str]:
    qf = qf.sort_values("match_no").reset_index(drop=True)
    return [actual_winner_row(r) for _, r in qf.iterrows()]

def sf_pairs_from_winners(winners: List[str]) -> List[Tuple[str,str]]:
    if len(winners) != 4:
        raise ValueError(f"Expected 4 winners from QF, got {len(winners)}")
    # Bracket order: (1 vs 2), (3 vs 4)
    return [(alias(winners[0]), alias(winners[1])),
            (alias(winners[2]), alias(winners[3]))]

# ---------------- Snapshot & predict ----------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str,float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"]==pk) & (log["date"]<=asof)].sort_values("date")
    snap = {"pre_elo":1500.0,"pre_selo":1500.0,"rest_days":30.0,"matches_28d":0.0,
            "rolling_10_winrate":0.5,"rolling_5_winrate":0.5,"streak":0.0,
            "avg_rest_days":20.0,"win_rate":0.5,"peak_elo":1500.0,"current_elo":1500.0,
            "wr_Clay":0.5,"wr_Grass":0.5,"wr_Hard":0.5,"rank_prior":np.nan,"h2h_wr_prior":0.5}
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]
        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        snap.update({
            "pre_elo": float(pre_elo), "current_elo": float(pre_elo),
            "peak_elo": float(last["peak_elo_post"] if after_last else last["peak_elo_prior"]),
            "win_rate": float(last["wr_overall_post_sm"] if after_last else last["wr_overall_prior_sm"]),
            "avg_rest_days": float(last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]),
            "matches_28d": float(last.get("matches_28d_post" if after_last else "matches_28d_prior", 0.0)),
            "rolling_10_winrate": float(last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]),
            "rolling_5_winrate": float(last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]),
            "streak": float(last["streak_post"] if after_last else last["streak_prior"]),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
            "wr_Clay": float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"]),
            "wr_Grass": float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"]),
            "wr_Hard": float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"]),
        })
        sub_s = sub[sub["surface"]==target_surface]
        if not sub_s.empty:
            ls = sub_s.iloc[-1]
            snap["pre_selo"] = float(ls["post_selo"] if asof > ls["date"] else ls["pre_selo"])
    h2h = log[(log["player_key"]==pk) & (log["opp_key"]==ok) & (log["date"]<=asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1+wins)/(2+cnt)
    return snap

# ---------------- Main ----------------
def main() -> None:
    # model
    model_path = _first_existing(MODEL_PATHS)
    if not model_path: raise FileNotFoundError("rf_model.joblib not found")
    bundle = load(model_path)
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    # base dataset
    data_path = _first_existing(PREF_DATASETS)
    if not data_path: raise FileNotFoundError("No base dataset found.")
    base = pd.read_csv(data_path)

    # QF complete
    qf = load_qf_complete(QF_COMPLETE)

    # Append QF → dataset snapshot
    matches = append_usopen_qf_results(base, qf)
    _ensure_dir(OUT_DATASET)
    matches.to_csv(OUT_DATASET, index=False, encoding="utf-8-sig")
    print(f"[OK] Dataset with USO QF appended → {OUT_DATASET} (rows: {len(matches)})")

    # Build log up to after QF
    log = build_player_log(matches, EloParams())

    # Build SF pairs from winners (bracket order)
    winners = winners_in_bracket_order(qf)
    pairs = sf_pairs_from_winners(winners)

    # as-of date
    if AS_OF_DATE_SF:
        asof = _safe_date(AS_OF_DATE_SF)
    else:
        max_qf = pd.to_datetime(qf["date"]).max()
        asof = (max_qf + pd.Timedelta(days=1)) if pd.notna(max_qf) else _safe_date("2025-09-05")

    rows=[]
    for i, (A,B) in enumerate(pairs, start=1):
        sa = snapshot_asof(log, A, B, asof, SURFACE)
        sb = snapshot_asof(log, B, A, asof, SURFACE)

        # Pipeline expects alphabetical orientation; we preserve bracket order via match_no.
        p1, p2 = (A, B) if A <= B else (B, A)
        s1, s2 = (sa, sb) if p1 == A else (sb, sa)
        def d(k): return float(s1[k]-s2[k])

        feats = {
            "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_SF, "best_of": BEST_OF,
            "elo_diff": d("pre_elo"), "selo_diff": d("pre_selo"),
            "diff_wr_Grass": d("wr_Grass"), "diff_wr_Hard": d("wr_Hard"), "diff_wr_Clay": d("wr_Clay"),
            "diff_win_rate": d("win_rate"), "diff_current_elo": d("current_elo"), "diff_peak_elo": d("peak_elo"),
            "diff_matches_28d": d("matches_28d"), "diff_avg_rest_days": d("avg_rest_days"),
            "diff_rolling_10_winrate": d("rolling_10_winrate"), "diff_rolling_5_winrate": d("rolling_5_winrate"),
            "diff_streak": d("streak"), "diff_h2h_wr_prior": d("h2h_wr_prior"),
        }
        # rank robust (why: missing/legacy ranks can bias diffs)
        if pd.isna(sa.get("rank_prior")) or pd.isna(sb.get("rank_prior")):
            feats["rank_diff"] = 0.0
        else:
            feats["rank_diff"] = float(np.clip((s1["rank_prior"] - s2["rank_prior"]), -RANK_CLIP, RANK_CLIP))

        X = {c: feats.get(c, 0.0) for c in feature_cols}
        proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:,1][0])
        prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
        winner = A if prob_a >= 0.5 else B
        conf = max(prob_a, 1.0 - prob_a)

        snap_w, snap_l = (sa, sb) if winner == A else (sb, sa)
        def delta(k): return float(snap_w[k]-snap_l[k])
        if pd.isna(snap_w.get("rank_prior")) or pd.isna(snap_l.get("rank_prior")):
            delta_rank = 0.0
        else:
            delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

        rows.append({
            "match_no": i, "date": asof.strftime("%Y-%m-%d"), "round": ROUND_SF,
            "player_a": A, "player_b": B, "pred_winner": winner, "correct_prediction": np.nan,
            "confidence": conf, "prob_player_a_win": prob_a, "prob_player_b_win": 1.0 - prob_a,
            "delta_elo": delta("pre_elo"), "delta_surface_elo": delta("pre_selo"),
            "delta_wr_Grass": delta("wr_Grass"), "delta_wr_Hard": delta("wr_Hard"), "delta_wr_Clay": delta("wr_Clay"),
            "delta_win_rate": delta("win_rate"), "delta_current_elo": delta("current_elo"), "delta_peak_elo": delta("peak_elo"),
            "delta_matches_28d": delta("matches_28d"), "delta_avg_rest_days": delta("avg_rest_days"),
            "delta_rolling_10_winrate": delta("rolling_10_winrate"), "delta_rolling_5_winrate": delta("rolling_5_winrate"),
            "delta_streak": delta("streak"), "delta_h2h_wr_prior": delta("h2h_wr_prior"), "delta_rank": delta_rank,
        })

    out_df = pd.DataFrame(rows).sort_values("match_no").reset_index(drop=True)

    header = [
        "match_no","date","round","player_a","player_b","pred_winner", "correct_prediction", "confidence",
        "prob_player_a_win","prob_player_b_win",
        "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
        "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
        "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
    ]
    for c in header:
        if c not in out_df.columns: out_df[c] = pd.NA
    out_df = out_df[header]

    _ensure_dir(OUT_CSV)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] US Open SF predictions → {OUT_CSV}")
    with pd.option_context("display.max_columns", None, "display.width", 170):
        print(out_df.to_string(index=False))

if __name__ == "__main__":
    main()


[OK] Dataset with USO QF appended → ./data/match_dataset_post_uso_qf_2025.csv (rows: 12089)


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] US Open SF predictions → ./reports/usopen2025_SF_predictions.csv
 match_no       date round       player_a              player_b    pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-09-05    SF  Jannik Sinner Felix Auger Aliassime  Jannik Sinner                 NaN    0.792169           0.792169           0.207831 373.456770         329.806226        0.161836       0.171626       0.124923        0.170998         373.456770      300.066104                2.0            -0.917102                       0.1                      0.0           0.0           -0.333333         0.0
        2 2025-09-05    SF Novak Djokovic        Carlos Alcaraz Novak Djokovic                 NaN    0.

In [100]:
# filepath: ./predict_usopen_2025_f.py
"""
US Open 2025 – Final prediction (deltas-only CSV).

Inputs:
  - ./reports/usopen2025_SF_predictions_complete.csv  (SF with correct_prediction filled 0/1)
  - ./models/rf_model.joblib
  - Base dataset snapshot (prefer later):
      ./data/match_dataset_post_uso_qf_2025.csv
      ./data/match_dataset_post_uso_r16_2025.csv
      ./data/match_dataset_post_uso_r32_2025.csv
      ./data/match_dataset_post_uso_r64_2025.csv
      ./data/match_dataset_post_uso_r128_2025.csv
      ./data/match_dataset_post_cincinnati_2025.csv
      ./data/match_dataset_post_wimbledon_2025_canada.csv
      ./data/match_dataset_post_wimbledon_2025.csv
      ./atp_matches_2021_2024.csv

Outputs:
  - ./data/match_dataset_post_uso_sf_2025.csv
  - ./reports/usopen2025_F_predictions.csv
"""

from __future__ import annotations

import os, re, unicodedata
from collections import deque
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from joblib import load
from sklearn.pipeline import Pipeline

# ---------------- Config ----------------
MODEL_PATHS = ["./models/rf_model.joblib", "/mnt/data/models/rf_model.joblib"]

PREF_DATASETS = [
    "./data/match_dataset_post_uso_qf_2025.csv",
    "./data/match_dataset_post_uso_r16_2025.csv",
    "./data/match_dataset_post_uso_r32_2025.csv",
    "./data/match_dataset_post_uso_r64_2025.csv",
    "./data/match_dataset_post_uso_r128_2025.csv",
    "./data/match_dataset_post_cincinnati_2025.csv",
    "./data/match_dataset_post_wimbledon_2025_canada.csv",
    "./data/match_dataset_post_wimbledon_2025.csv",
    "./atp_matches_2021_2024.csv",
    "/mnt/data/atp_matches_2021_2024.csv",
]

SF_COMPLETE = "./reports/usopen2025_SF_predictions_complete.csv"
OUT_DATASET  = "./data/match_dataset_post_uso_sf_2025.csv"
OUT_CSV      = "./reports/usopen2025_F_predictions.csv"

# If None, auto = (max SF date + 1 day)
AS_OF_DATE_F: Optional[str] = "2025-09-07"

TOURNEY_NAME   = "US Open 2025"
SURFACE        = "Hard"
TOURNEY_LEVEL  = "G"
ROUND_SF       = "SF"
ROUND_F        = "F"
BEST_OF        = 5

RANK_CLIP      = 500
SMOOTH_ALPHA   = 5.0
SMOOTH_BETA    = 5.0

# ---------------- Name aliases ----------------
ALIAS = {
    "Felix Auger-Aliassime": "Felix Auger Aliassime",
    "Alex de Minaur": "Alex De Minaur",
    "Botic Van De Zandschulp": "Botic van de Zandschulp",
    "Giovanni Mpetshi-Perricard": "Giovanni Mpetshi Perricard",
}
def alias(s: str) -> str:
    return ALIAS.get(s, s)

# ---------------- Utils ----------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

# ---------------- Elo/log ----------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    need = ["tourney_date","tourney_name","surface","tourney_level","match_num",
            "winner_name","loser_name","round","best_of"]
    for c in need:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    BASE, K, KS = params.base, params.k_overall, params.k_surface
    elo_overall: Dict[str,float] = {}
    elo_surface: Dict[Tuple[str,str],float] = {}

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = alias(r["winner_name"]); l = alias(r["loser_name"])
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0/(1.0+10**((ew_l-ew_w)/400))
        exp_w_s = 1.0/(1.0+10**((es_l-es_w)/400))

        new_ew_w = ew_w + K  * (1-exp_w)
        new_ew_l = ew_l - K  * (1-exp_w)
        new_es_w = es_w + KS * (1-exp_w_s)
        new_es_l = es_l - KS * (1-exp_w_s)

        rows.append({"player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
                     "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[w_rank] if w_rank else np.nan})
        rows.append({"player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
                     "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[l_rank] if l_rank else np.nan})

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon); log["opp_key"] = log["opp"].map(canon)

    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0,60)

    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out=[]
        for dt in dates:
            while dq and (dt-dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0)); dq.append(dt)
        return out
    for w in (7,14,28):
        log[f"matches_{w}d_prior"] = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, False))
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, True))

    grp = log.groupby("player_key")
    log["rolling_10_winrate_prior"] = grp["is_win"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_10_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_prior"]  = grp["is_win"].transform(lambda s: s.shift(1).rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]   = grp["is_win"].transform(lambda s: s.rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)

    def streak_prior(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.shift(1).fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    def streak_post(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    log["streak_prior"] = grp["is_win"].transform(streak_prior)
    log["streak_post"]  = grp["is_win"].transform(streak_post)

    log["avg_rest_days_prior"] = grp["rest_days"].transform(lambda s: s.shift(1).expanding().mean()).fillna(20.0).clip(0,60)
    log["avg_rest_days_post"]  = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    def smooth(w: pd.Series, n: pd.Series) -> pd.Series:
        return (SMOOTH_ALPHA + w) / (SMOOTH_ALPHA + SMOOTH_BETA + n)
    wins_prior_ov = grp["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior_ov    = grp["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["wr_overall_prior_sm"] = smooth(wins_prior_ov, n_prior_ov).clip(0,1)
    log["wr_overall_post_sm"]  = smooth(wins_post_ov,  n_post_ov).clip(0,1)

    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_prior = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().sum()).fillna(0.0)
    wins_post  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_prior    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.shift(1).expanding().count()).fillna(0.0)
    n_post     = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    wr_prior_sm = smooth(wins_prior, n_prior).clip(0,1)
    wr_post_sm  = smooth(wins_post,  n_post).clip(0,1)
    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_prior_sm"] = wr_prior_sm.values; tmp["wr_post_sm"] = wr_post_sm.values
    surf_prior = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_prior_sm", aggfunc="last")
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm",  aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_prior.columns: surf_prior[srf]=np.nan
        if srf not in surf_post.columns:  surf_post[srf]=np.nan
    surf_prior = surf_prior[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post  = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()
    for df in (surf_prior, surf_post):
        df[["Clay","Grass","Hard"]] = df.groupby("player_key")[["Clay","Grass","Hard"]].ffill()
        df[["Clay","Grass","Hard"]] = df[["Clay","Grass","Hard"]].fillna(0.5)
    log = log.merge(surf_prior.rename(columns={"Clay":"wr_Clay_prior_sm","Grass":"wr_Grass_prior_sm","Hard":"wr_Hard_prior_sm"}),
                    on=["player_key","date"], how="left")
    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")

    log["peak_elo_prior"] = grp["pre_elo"].transform(lambda s: s.shift(1).cummax()).fillna(1500.0)
    log["peak_elo_post"]  = grp["post_elo"].transform(lambda s: s.cummax()).fillna(1500.0)
    log["rank_prior"]     = log.groupby("player_key")["rank"].shift(1)
    return log

# ---------------- SF complete → dataset snapshot ----------------
def load_sf_complete(path: str) -> pd.DataFrame:
    df = pd.read_csv(path).sort_values("match_no").reset_index(drop=True)
    need = ["match_no","player_a","player_b","pred_winner","correct_prediction","date"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {path}")
    if df["correct_prediction"].isna().any():
        bad = df[df["correct_prediction"].isna()]["match_no"].tolist()
        raise ValueError(f"'correct_prediction' has NaN for match_no={bad[:10]}… Fill 0/1 first.")
    for col in ["player_a","player_b","pred_winner"]:
        df[col] = df[col].astype(str).map(alias)
    return df

def actual_winner_row(r: pd.Series) -> str:
    return r["pred_winner"] if int(r["correct_prediction"])==1 else (r["player_b"] if r["pred_winner"]==r["player_a"] else r["player_a"])

def append_usopen_sf_results(base: pd.DataFrame, sf: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for _, r in sf.iterrows():
        w = actual_winner_row(r); l = r["player_b"] if w==r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": TOURNEY_NAME,
            "surface": SURFACE,
            "tourney_level": TOURNEY_LEVEL,
            "match_num": int(r["match_no"]),
            "winner_name": alias(str(w)),
            "loser_name": alias(str(l)),
            "round": ROUND_SF,
            "best_of": BEST_OF,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    extra["surface"] = extra["surface"].map(normalize_surface)
    common = [c for c in extra.columns if c in base.columns]
    add = [c for c in ["tourney_date","tourney_name","surface","tourney_level","match_num",
                       "winner_name","loser_name","round","best_of"] if c not in common]
    return pd.concat([base, extra[common+add]], ignore_index=True, sort=False)

def winners_in_bracket_order(sf: pd.DataFrame) -> List[str]:
    sf = sf.sort_values("match_no").reset_index(drop=True)
    return [actual_winner_row(r) for _, r in sf.iterrows()]

def f_pair_from_winners(winners: List[str]) -> Tuple[str,str]:
    if len(winners) != 2:
        raise ValueError(f"Expected 2 winners from SF, got {len(winners)}")
    return (alias(winners[0]), alias(winners[1]))

# ---------------- Snapshot & predict ----------------
def snapshot_asof(log: pd.DataFrame, player: str, opp: str, asof: pd.Timestamp, target_surface: str) -> Dict[str,float]:
    pk = canon(player); ok = canon(opp)
    sub = log[(log["player_key"]==pk) & (log["date"]<=asof)].sort_values("date")
    snap = {"pre_elo":1500.0,"pre_selo":1500.0,"rest_days":30.0,"matches_28d":0.0,
            "rolling_10_winrate":0.5,"rolling_5_winrate":0.5,"streak":0.0,
            "avg_rest_days":20.0,"win_rate":0.5,"peak_elo":1500.0,"current_elo":1500.0,
            "wr_Clay":0.5,"wr_Grass":0.5,"wr_Hard":0.5,"rank_prior":np.nan,"h2h_wr_prior":0.5}
    if not sub.empty:
        last = sub.iloc[-1]
        after_last = asof > last["date"]
        pre_elo = last["post_elo"] if after_last else last["pre_elo"]
        snap.update({
            "pre_elo": float(pre_elo), "current_elo": float(pre_elo),
            "peak_elo": float(last["peak_elo_post"] if after_last else last["peak_elo_prior"]),
            "win_rate": float(last["wr_overall_post_sm"] if after_last else last["wr_overall_prior_sm"]),
            "avg_rest_days": float(last["avg_rest_days_post"] if after_last else last["avg_rest_days_prior"]),
            "matches_28d": float(last.get("matches_28d_post" if after_last else "matches_28d_prior", 0.0)),
            "rolling_10_winrate": float(last["rolling_10_winrate_post"] if after_last else last["rolling_10_winrate_prior"]),
            "rolling_5_winrate": float(last["rolling_5_winrate_post"] if after_last else last["rolling_5_winrate_prior"]),
            "streak": float(last["streak_post"] if after_last else last["streak_prior"]),
            "rest_days": float(np.clip((asof - last["date"]).days, 0, 60)),
            "wr_Clay": float(last["wr_Clay_post_sm"] if after_last else last["wr_Clay_prior_sm"]),
            "wr_Grass": float(last["wr_Grass_post_sm"] if after_last else last["wr_Grass_prior_sm"]),
            "wr_Hard": float(last["wr_Hard_post_sm"] if after_last else last["wr_Hard_prior_sm"]),
        })
        sub_s = sub[sub["surface"]==target_surface]
        if not sub_s.empty:
            ls = sub_s.iloc[-1]
            snap["pre_selo"] = float(ls["post_selo"] if asof > ls["date"] else ls["pre_selo"])
    h2h = log[(log["player_key"]==pk) & (log["opp_key"]==ok) & (log["date"]<=asof)]
    if not h2h.empty:
        wins = int(h2h["is_win"].sum()); cnt = int(len(h2h))
        snap["h2h_wr_prior"] = (1+wins)/(2+cnt)
    return snap

# ---------------- Main ----------------
def main() -> None:
    model_path = _first_existing(MODEL_PATHS)
    if not model_path: raise FileNotFoundError("rf_model.joblib not found")
    bundle = load(model_path)
    pipe: Pipeline = bundle["pipeline"]
    feature_cols: List[str] = bundle["feature_cols"]

    data_path = _first_existing(PREF_DATASETS)
    if not data_path: raise FileNotFoundError("No base dataset found.")
    base = pd.read_csv(data_path)

    sf = load_sf_complete(SF_COMPLETE)

    matches = append_usopen_sf_results(base, sf)
    _ensure_dir(OUT_DATASET)
    matches.to_csv(OUT_DATASET, index=False, encoding="utf-8-sig")
    print(f"[OK] Dataset with USO SF appended → {OUT_DATASET} (rows: {len(matches)})")

    log = build_player_log(matches, EloParams())

    winners = winners_in_bracket_order(sf)
    A, B = f_pair_from_winners(winners)

    if AS_OF_DATE_F:
        asof = _safe_date(AS_OF_DATE_F)
    else:
        max_sf = pd.to_datetime(sf["date"]).max()
        asof = (max_sf + pd.Timedelta(days=1)) if pd.notna(max_sf) else _safe_date("2025-09-07")

    sa = snapshot_asof(log, A, B, asof, SURFACE)
    sb = snapshot_asof(log, B, A, asof, SURFACE)

    p1, p2 = (A, B) if A <= B else (B, A)
    s1, s2 = (sa, sb) if p1 == A else (sb, sa)
    def d(k): return float(s1[k]-s2[k])

    feats = {
        "surface": SURFACE, "tourney_level": TOURNEY_LEVEL, "round": ROUND_F, "best_of": BEST_OF,
        "elo_diff": d("pre_elo"), "selo_diff": d("pre_selo"),
        "diff_wr_Grass": d("wr_Grass"), "diff_wr_Hard": d("wr_Hard"), "diff_wr_Clay": d("wr_Clay"),
        "diff_win_rate": d("win_rate"), "diff_current_elo": d("current_elo"), "diff_peak_elo": d("peak_elo"),
        "diff_matches_28d": d("matches_28d"), "diff_avg_rest_days": d("avg_rest_days"),
        "diff_rolling_10_winrate": d("rolling_10_winrate"), "diff_rolling_5_winrate": d("rolling_5_winrate"),
        "diff_streak": d("streak"), "diff_h2h_wr_prior": d("h2h_wr_prior"),
    }
    if pd.isna(sa.get("rank_prior")) or pd.isna(sb.get("rank_prior")):
        feats["rank_diff"] = 0.0
    else:
        feats["rank_diff"] = float(np.clip((s1["rank_prior"] - s2["rank_prior"]), -RANK_CLIP, RANK_CLIP))

    X = {c: feats.get(c, 0.0) for c in feature_cols}
    proba_p1 = float(pipe.predict_proba(pd.DataFrame([X]))[:,1][0])
    prob_a = proba_p1 if p1 == A else 1.0 - proba_p1
    winner = A if prob_a >= 0.5 else B
    conf = max(prob_a, 1.0 - prob_a)

    snap_w, snap_l = (sa, sb) if winner == A else (sb, sa)
    def delta(k): return float(snap_w[k]-snap_l[k])
    if pd.isna(snap_w.get("rank_prior")) or pd.isna(snap_l.get("rank_prior")):
        delta_rank = 0.0
    else:
        delta_rank = float(np.clip(snap_w["rank_prior"] - snap_l["rank_prior"], -RANK_CLIP, RANK_CLIP))

    out_df = pd.DataFrame([{
        "match_no": 1, "date": asof.strftime("%Y-%m-%d"), "round": ROUND_F,
        "player_a": A, "player_b": B, "pred_winner": winner, "correct_prediction": np.nan,
        "confidence": conf, "prob_player_a_win": prob_a, "prob_player_b_win": 1.0 - prob_a,
        "delta_elo": delta("pre_elo"), "delta_surface_elo": delta("pre_selo"),
        "delta_wr_Grass": delta("wr_Grass"), "delta_wr_Hard": delta("wr_Hard"), "delta_wr_Clay": delta("wr_Clay"),
        "delta_win_rate": delta("win_rate"), "delta_current_elo": delta("current_elo"), "delta_peak_elo": delta("peak_elo"),
        "delta_matches_28d": delta("matches_28d"), "delta_avg_rest_days": delta("avg_rest_days"),
        "delta_rolling_10_winrate": delta("rolling_10_winrate"), "delta_rolling_5_winrate": delta("rolling_5_winrate"),
        "delta_streak": delta("streak"), "delta_h2h_wr_prior": delta("h2h_wr_prior"), "delta_rank": delta_rank,
    }])

    header = [
        "match_no","date","round","player_a","player_b","pred_winner", "correct_prediction", "confidence",
        "prob_player_a_win","prob_player_b_win",
        "delta_elo","delta_surface_elo","delta_wr_Grass","delta_wr_Hard","delta_wr_Clay",
        "delta_win_rate","delta_current_elo","delta_peak_elo","delta_matches_28d","delta_avg_rest_days",
        "delta_rolling_10_winrate","delta_rolling_5_winrate","delta_streak","delta_h2h_wr_prior","delta_rank",
    ]
    for c in header:
        if c not in out_df.columns: out_df[c] = pd.NA
    out_df = out_df[header]

    _ensure_dir(OUT_CSV)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] US Open Final prediction → {OUT_CSV}")
    with pd.option_context("display.max_columns", None, "display.width", 170):
        print(out_df.to_string(index=False))

if __name__ == "__main__":
    main()


[OK] Dataset with USO SF appended → ./data/match_dataset_post_uso_sf_2025.csv (rows: 12091)


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

[OK] US Open Final prediction → ./reports/usopen2025_F_predictions.csv
 match_no       date round      player_a       player_b   pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 2025-09-07     F Jannik Sinner Carlos Alcaraz Jannik Sinner                 NaN    0.683296           0.683296           0.316704  92.130257         120.806942       -0.078063       0.055314      -0.104252       -0.003217          92.130257       98.823645                0.0            -0.437832                      -0.1                      0.0          -6.0           -0.142857         0.0


In [102]:
# filepath: ./combine_usopen_2025_predictions.py
"""
Combine all completed US Open 2025 prediction CSVs into a single file.

- Preserves bracket order (R128→R64→R32→R16→QF→SF→F, then by match_no).
- Prefers '*_predictions_complete.csv' for each round; can optionally include non-complete files.
- Column layout follows the R128 header when available; any missing columns are added as NaN.

CLI usage examples:
  python combine_usopen_2025_predictions.py
  python combine_usopen_2025_predictions.py --only-complete all
  python combine_usopen_2025_predictions.py --out ./reports/usopen2025_all_rounds_predictions_combined.csv
  python combine_usopen_2025_predictions.py --include-noncomplete

Programmatic:
  from combine_usopen_2025_predictions import combine_usopen_predictions
  combine_usopen_predictions(out="...", only_complete="all", include_noncomplete=False)
"""

from __future__ import annotations

import argparse
import os
from typing import List, Dict, Optional

import pandas as pd

ROUNDS: List[str] = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]
ROUND_ORDER: Dict[str, int] = {r: i for i, r in enumerate(ROUNDS, start=1)}

SEARCH_DIRS = ["./reports", "/mnt/data"]
NAME_PATTERNS = [
    "usopen2025_{round}_predictions_complete.csv",  # preferred
    "usopen2025_{round}_predictions.csv",           # fallback
]

DEFAULT_OUT = "./reports/usopen2025_all_rounds_predictions_combined.csv"


def _find_round_file(round_code: str, allow_fallback: bool) -> Optional[str]:
    for d in SEARCH_DIRS:
        preferred = os.path.join(d, NAME_PATTERNS[0].format(round=round_code))
        if os.path.exists(preferred):
            return preferred
    if allow_fallback:
        for d in SEARCH_DIRS:
            fallback = os.path.join(d, NAME_PATTERNS[1].format(round=round_code))
            if os.path.exists(fallback):
                return fallback
    return None


def _load_round_df(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "round" not in df.columns:
        raise ValueError(f"'round' column missing in {path}")
    if "match_no" not in df.columns:
        raise ValueError(f"'match_no' column missing in {path}")
    return df


def _qualifies_by_completeness(df: pd.DataFrame, mode: str) -> bool:
    """
    mode:
      - 'off': do not enforce completeness
      - 'any': require 'correct_prediction' exists and at least one non-null
      - 'all': require 'correct_prediction' exists and all rows non-null
    """
    if mode == "off":
        return True
    if "correct_prediction" not in df.columns:
        return False
    if mode == "any":
        return df["correct_prediction"].notna().any()
    if mode == "all":
        return df["correct_prediction"].notna().all()
    return True


def combine_usopen_predictions(
    out: str = DEFAULT_OUT,
    only_complete: str = "all",            # 'off' | 'any' | 'all'
    include_noncomplete: bool = False,      # allow fallback to non-complete files
) -> pd.DataFrame:
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)

    # Discover files
    allow_fallback = include_noncomplete or (only_complete == "off")
    files: List[str] = []
    for r in ROUNDS:
        p = _find_round_file(r, allow_fallback=allow_fallback)
        if p:
            files.append(p)
        else:
            print(f"[WARN] No file found for round {r} (preferred or fallback).")

    if not files:
        raise FileNotFoundError("No round prediction CSVs found in ./reports or /mnt/data.")

    # Determine reference header from R128 if available
    r128_path = _find_round_file("R128", allow_fallback=allow_fallback)
    ref_header: Optional[List[str]] = None
    if r128_path:
        ref_header = list(pd.read_csv(r128_path, nrows=0).columns)

    # Load, filter by completeness, collect union
    round_frames: List[pd.DataFrame] = []
    union_cols: set[str] = set(ref_header or [])
    used_files: List[str] = []

    for path in files:
        try:
            df = _load_round_df(path)
        except Exception as e:
            print(f"[SKIP] {path}: {e}")
            continue

        if not _qualifies_by_completeness(df, only_complete):
            print(f"[SKIP] {path}: does not meet --only-complete={only_complete}")
            continue

        union_cols.update(df.columns)
        round_frames.append(df)
        used_files.append(path)

    if not round_frames:
        raise RuntimeError("No qualifying round files after filtering.")

    # Build ref header if not from R128
    if ref_header is None:
        preferred = [
            "match_no", "date", "round", "player_a", "player_b",
            "pred_winner", "confidence", "prob_player_a_win", "prob_player_b_win",
        ]
        deltas = sorted([c for c in union_cols if c.startswith("delta_")])
        rest = [c for c in union_cols if c not in set(preferred + deltas)]
        ref_header = preferred + deltas + sorted(rest)

    # Align frames
    aligned: List[pd.DataFrame] = []
    for df in round_frames:
        for col in ref_header:
            if col not in df.columns:
                df[col] = pd.NA
        aligned.append(df[ref_header].copy())

    # Combine & sort
    big = pd.concat(aligned, ignore_index=True)

    big["__round_order__"] = big["round"].map(ROUND_ORDER).fillna(9999).astype(int)
    big["__match_no__"] = pd.to_numeric(big["match_no"], errors="coerce")

    big = big.sort_values(
        ["__round_order__", "__match_no__"],
        kind="mergesort"
    ).drop(columns=["__round_order__", "__match_no__"]).reset_index(drop=True)

    big.to_csv(out, index=False, encoding="utf-8-sig")

    print(f"[OK] Combined predictions saved → {out}")
    print(f"[INFO] Included files ({len(used_files)}):")
    for p in used_files:
        print(f"  - {p}")
    with pd.option_context("display.max_columns", None, "display.width", 200):
        print(big.head(14).to_string(index=False))

    return big


def _parse_args_safe() -> argparse.Namespace:
    parser = argparse.ArgumentParser(add_help=True)
    parser.add_argument("--out", default=DEFAULT_OUT, help="Output CSV path")
    parser.add_argument(
        "--only-complete",
        choices=["off", "any", "all"],
        default="all",
        help=("Completeness filter per file: "
              "'off' = include any file found; "
              "'any' = require at least one non-null 'correct_prediction'; "
              "'all' = require all rows non-null 'correct_prediction' (default).")
    )
    parser.add_argument(
        "--include-noncomplete",
        action="store_true",
        help="Allow fallback to non-complete round files if no complete file is found."
    )
    # parse_known_args avoids SystemExit in notebooks (unknown argv)
    args, _unknown = parser.parse_known_args()
    return args


def main() -> None:
    args = _parse_args_safe()
    combine_usopen_predictions(
        out=args.out,
        only_complete=args.only_complete,
        include_noncomplete=args.include_noncomplete,
    )


if __name__ == "__main__":
    main()


[OK] Combined predictions saved → ./reports/usopen2025_all_rounds_predictions_combined.csv
[INFO] Included files (7):
  - ./reports\usopen2025_R128_predictions_complete.csv
  - ./reports\usopen2025_R64_predictions_complete.csv
  - ./reports\usopen2025_R32_predictions_complete.csv
  - ./reports\usopen2025_R16_predictions_complete.csv
  - ./reports\usopen2025_QF_predictions_complete.csv
  - ./reports\usopen2025_SF_predictions_complete.csv
  - ./reports\usopen2025_F_predictions_complete.csv
 match_no      date round          player_a                   player_b      pred_winner  correct_prediction  confidence  prob_player_a_win  prob_player_b_win  delta_elo  delta_surface_elo  delta_wr_Grass  delta_wr_Hard  delta_wr_Clay  delta_win_rate  delta_current_elo  delta_peak_elo  delta_matches_28d  delta_avg_rest_days  delta_rolling_10_winrate  delta_rolling_5_winrate  delta_streak  delta_h2h_wr_prior  delta_rank
        1 8/24/2025  R128     Jannik Sinner                Vit Kopriva    Jannik Sinn

In [104]:
# filepath: ./update_player_profiles_post_usopen.py
"""
Update dataset with US Open 2025 results and rebuild player profiles.

Outputs:
  - ./data/match_dataset_post_usopen_2025.csv
  - ./reports/player_profiles_post_usopen_2025.csv
  - ./reports/player_profiles_latest.csv  (same as above, for convenience)
"""

from __future__ import annotations

import os, re, unicodedata
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

# ---------------- Paths ----------------
PREF_BASESETS = [
    "./data/match_dataset_post_cincinnati_2025.csv",
    "./data/match_dataset_post_wimbledon_2025_canada.csv",
    "./data/match_dataset_post_wimbledon_2025.csv",
    "./atp_matches_2021_2024.csv",
    "/mnt/data/atp_matches_2021_2024.csv",
]
USO_REPORTS_DIRS = ["./reports", "/mnt/data"]
USO_COMPLETE_PAT = "usopen2025_{round}_predictions_complete.csv"
USO_ROUNDS = ["R128","R64","R32","R16","QF","SF","F"]

OUT_DATASET   = "./data/match_dataset_post_usopen_2025.csv"
OUT_PROFILES  = "./reports/player_profiles_post_usopen_2025.csv"
OUT_LATEST    = "./reports/player_profiles_latest.csv"

# ---------------- Small utils ----------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii","ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+"," ", s.lower()).strip()

# Harmonize frequent name variants
ALIAS = {
    "Felix Auger-Aliassime": "Felix Auger Aliassime",
    "Alex de Minaur": "Alex De Minaur",
    "Botic Van De Zandschulp": "Botic van de Zandschulp",
    "Giovanni Mpetshi-Perricard": "Giovanni Mpetshi Perricard",
}
def alias(s: str) -> str:
    return ALIAS.get(s, s)

# ---------------- Elo/log ----------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = 24.0
    k_surface: float = 18.0

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    need = ["tourney_date","tourney_name","surface","tourney_level","match_num",
            "winner_name","loser_name","round","best_of"]
    for c in need:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m["winner_name"] = m["winner_name"].astype(str).map(alias)
    m["loser_name"]  = m["loser_name"].astype(str).map(alias)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    w_rank = "winner_rank" if "winner_rank" in m.columns else None
    l_rank = "loser_rank" if "loser_rank" in m.columns else None

    BASE, K, KS = params.base, params.k_overall, params.k_surface
    elo_overall: Dict[str,float] = {}
    elo_surface: Dict[Tuple[str,str],float] = {}

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0/(1.0 + 10**((ew_l - ew_w)/400))
        exp_w_s = 1.0/(1.0 + 10**((es_l - es_w)/400))

        new_ew_w = ew_w + K  * (1 - exp_w)
        new_ew_l = ew_l - K  * (1 - exp_w)
        new_es_w = es_w + KS * (1 - exp_w_s)
        new_es_l = es_l - KS * (1 - exp_w_s)

        rows.append({"player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
                     "pre_elo": ew_w, "pre_selo": es_w, "post_elo": new_ew_w, "post_selo": new_es_w,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[w_rank] if w_rank else np.nan})
        rows.append({"player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
                     "pre_elo": ew_l, "pre_selo": es_l, "post_elo": new_ew_l, "post_selo": new_es_l,
                     "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"],
                     "rank": r[l_rank] if l_rank else np.nan})

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w,s)] = new_es_w; elo_surface[(l,s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon)
    log["opp_key"]    = log["opp"].map(canon)

    # Rest/workload
    log["prev_date"] = log.groupby("player_key")["date"].shift(1)
    log["rest_days"] = (log["date"] - log["prev_date"]).dt.days
    log["rest_days"] = log["rest_days"].fillna(30).clip(0,60)

    # Rolling counts (inclusive post)
    from collections import deque
    def rolling_counts(dates: List[pd.Timestamp], window: int, inclusive: bool) -> List[int]:
        dq = deque(); out=[]
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq) + (1 if inclusive else 0)); dq.append(dt)
        return out
    for w in (7,14,28):
        log[f"matches_{w}d_post"]  = log.groupby("player_key")["date"].transform(lambda s: rolling_counts(list(s), w, True))

    grp = log.groupby("player_key")

    # Smoothed WR overall (post)
    SMOOTH_A, SMOOTH_B = 5.0, 5.0
    wins_post_ov  = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_post_ov     = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["wr_overall_post_sm"] = ((SMOOTH_A + wins_post_ov) / (SMOOTH_A + SMOOTH_B + n_post_ov)).clip(0,1)

    # Rolling form (post)
    log["rolling_10_winrate_post"] = grp["is_win"].transform(lambda s: s.rolling(10, min_periods=3).mean()).fillna(0.5).clip(0,1)
    log["rolling_5_winrate_post"]  = grp["is_win"].transform(lambda s: s.rolling(5,  min_periods=3).mean()).fillna(0.5).clip(0,1)

    # Streak (post)
    def streak_post(series: pd.Series) -> pd.Series:
        out=[]; cur=0
        for v in series.fillna(-1).astype(int):
            if v==1: cur = cur+1 if cur>=0 else 1
            elif v==0: cur = cur-1 if cur<=0 else -1
            else: cur = 0
            out.append(cur)
        return pd.Series(out, index=series.index)
    log["streak_post"] = grp["is_win"].transform(streak_post)

    # Avg rest (post)
    log["avg_rest_days_post"] = grp["rest_days"].transform(lambda s: s.expanding().mean()).fillna(20.0).clip(0,60)

    # Per-surface smoothed WRs (post)
    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_post = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    n_post    = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    wr_post_sm = ((SMOOTH_A + wins_post) / (SMOOTH_A + SMOOTH_B + n_post)).clip(0,1)
    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_post_sm"] = wr_post_sm.values
    surf_post  = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_post_sm", aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_post.columns: surf_post[srf] = np.nan
    surf_post = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()
    # forward-fill per player across dates
    surf_post[["Clay","Grass","Hard"]] = surf_post.groupby("player_key")[["Clay","Grass","Hard"]].ffill().fillna(0.5)
    log = log.merge(
        surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
        on=["player_key","date"], how="left"
    )

    return log

# ---------------- USO appenders ----------------
def _find_round_file(round_code: str) -> Optional[str]:
    for d in USO_REPORTS_DIRS:
        p = os.path.join(d, USO_COMPLETE_PAT.format(round=round_code))
        if os.path.exists(p):
            return p
    return None

def _load_complete_round(path: str) -> pd.DataFrame:
    df = pd.read_csv(path).sort_values("match_no").reset_index(drop=True)
    for c in ["match_no","player_a","player_b","pred_winner","correct_prediction","date","round"]:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {path}")
    if df["correct_prediction"].isna().any():
        raise ValueError(f"{path}: 'correct_prediction' has NaN; fill 0/1 before updating.")
    for col in ["player_a","player_b","pred_winner"]:
        df[col] = df[col].astype(str).map(alias)
    return df

def _actual_winner_row(r: pd.Series) -> str:
    return r["pred_winner"] if int(r["correct_prediction"])==1 else (r["player_b"] if r["pred_winner"]==r["player_a"] else r["player_a"])

def _append_usopen_round(base: pd.DataFrame, rnd_df: pd.DataFrame, round_code: str) -> pd.DataFrame:
    rows=[]
    for _, r in rnd_df.iterrows():
        w = _actual_winner_row(r); l = r["player_b"] if w==r["player_a"] else r["player_a"]
        rows.append({
            "tourney_date": _safe_date(r["date"]),
            "tourney_name": "US Open 2025",
            "surface": "Hard",
            "tourney_level": "G",
            "match_num": int(r["match_no"]),
            "winner_name": alias(str(w)),
            "loser_name":  alias(str(l)),
            "round": round_code,
            "best_of": 5,
            "winner_rank": np.nan,
            "loser_rank": np.nan,
        })
    extra = pd.DataFrame(rows)
    extra["surface"] = extra["surface"].map(normalize_surface)

    # Align to base columns
    cols = list(base.columns)
    for c in extra.columns:
        if c not in cols:
            cols.append(c)
    base = base.reindex(columns=cols)
    extra = extra.reindex(columns=cols)

    out = pd.concat([base, extra], ignore_index=True, sort=False)

    # Defensive de-dup (rare, but protects repeated runs)
    dedupe_keys = ["tourney_name","round","match_num","winner_name","loser_name"]
    if all(k in out.columns for k in dedupe_keys):
        out = out.drop_duplicates(subset=dedupe_keys, keep="last").reset_index(drop=True)

    return out

# ---------------- Build profiles table ----------------
def build_player_profiles(log: pd.DataFrame) -> pd.DataFrame:
    # Last match row per player
    idx = log.groupby("player_key")["date"].idxmax()
    last = log.loc[idx, ["player","player_key","date","post_elo","wr_overall_post_sm",
                         "rolling_10_winrate_post","rolling_5_winrate_post",
                         "streak_post","avg_rest_days_post","matches_28d_post"]].copy()
    last = last.rename(columns={"player":"name","date":"last_match_date"})

    # Peak overall elo
    peak = log.groupby("player_key")["post_elo"].max().rename("peak_elo")

    # Latest surface Elo per surface (as-of last known, not necessarily last match surface)
    last_selo = (
        log.sort_values(["player_key","surface","date"])
           .groupby(["player_key","surface"])["post_selo"].last()
           .unstack("surface")
           .reindex(columns=["Clay","Grass","Hard"])
    )
    for srf in ["Clay","Grass","Hard"]:
        if srf not in last_selo.columns: last_selo[srf] = np.nan
    last_selo = last_selo.rename(columns={"Clay":"selo_Clay","Grass":"selo_Grass","Hard":"selo_Hard"})

    # Per-surface WRs (already forward-filled to each date)
    wr_cols = ["wr_Clay_post_sm","wr_Grass_post_sm","wr_Hard_post_sm"]
    wr_last = log.loc[idx, ["player_key"] + wr_cols].set_index("player_key")
    wr_last = wr_last.rename(columns={
        "wr_Clay_post_sm":"wr_Clay",
        "wr_Grass_post_sm":"wr_Grass",
        "wr_Hard_post_sm":"wr_Hard",
    })

    prof = (
        last.set_index("player_key")
            .join(peak).join(last_selo).join(wr_last)
            .reset_index(drop=True)
    )

    # Final columns & formatting
    prof = prof.rename(columns={
        "post_elo":"current_elo",
        "wr_overall_post_sm":"overall_wr",
        "rolling_10_winrate_post":"form10_wr",
        "rolling_5_winrate_post":"form5_wr",
        "streak_post":"streak",
        "avg_rest_days_post":"avg_rest_days",
        "matches_28d_post":"matches_28d",
    })

    cols = [
        "name","last_match_date","current_elo","peak_elo",
        "selo_Clay","selo_Grass","selo_Hard",
        "wr_Clay","wr_Grass","wr_Hard",
        "overall_wr","form10_wr","form5_wr","streak","avg_rest_days","matches_28d",
    ]
    prof = prof[cols].copy()

    # Types & fills
    prof["last_match_date"] = pd.to_datetime(prof["last_match_date"]).dt.date
    for c in ["selo_Clay","selo_Grass","selo_Hard"]:
        prof[c] = prof[c].fillna(1500.0)
    for c in ["wr_Clay","wr_Grass","wr_Hard","overall_wr","form10_wr","form5_wr"]:
        prof[c] = prof[c].clip(0,1)

    # Sort by current_elo desc for convenience
    prof = prof.sort_values("current_elo", ascending=False).reset_index(drop=True)
    return prof

# ---------------- Main ----------------
def main() -> None:
    # Base dataset
    base_path = _first_existing(PREF_BASESETS)
    if not base_path:
        raise FileNotFoundError("No base dataset found. Put a pre-USO snapshot under ./data/ or the raw CSV at project root.")
    base = pd.read_csv(base_path)

    # Append US Open rounds that exist (in order)
    appended = base.copy()
    for rnd in USO_ROUNDS:
        p = _find_round_file(rnd)
        if not p:
            print(f"[INFO] Missing {rnd} complete file; skipping.")
            continue
        rnd_df = _load_complete_round(p)
        appended = _append_usopen_round(appended, rnd_df, round_code=rnd)
        print(f"[OK] Appended {rnd}: {p}")

    # Persist dataset
    _ensure_dir(OUT_DATASET)
    appended.to_csv(OUT_DATASET, index=False, encoding="utf-8-sig")
    print(f"[OK] Saved dataset → {OUT_DATASET} (rows: {len(appended)})")

    # Rebuild log & profiles
    log = build_player_log(appended, EloParams())
    profiles = build_player_profiles(log)

    # Emit profiles
    _ensure_dir(OUT_PROFILES)
    profiles.to_csv(OUT_PROFILES, index=False, encoding="utf-8-sig")
    _ensure_dir(OUT_LATEST)
    profiles.to_csv(OUT_LATEST, index=False, encoding="utf-8-sig")
    print(f"[OK] Saved player profiles → {OUT_PROFILES} and {OUT_LATEST}")

    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(profiles.head(12).to_string(index=False))

if __name__ == "__main__":
    main()


[OK] Appended R128: ./reports\usopen2025_R128_predictions_complete.csv
[OK] Appended R64: ./reports\usopen2025_R64_predictions_complete.csv
[OK] Appended R32: ./reports\usopen2025_R32_predictions_complete.csv
[OK] Appended R16: ./reports\usopen2025_R16_predictions_complete.csv
[OK] Appended QF: ./reports\usopen2025_QF_predictions_complete.csv
[OK] Appended SF: ./reports\usopen2025_SF_predictions_complete.csv
[OK] Appended F: ./reports\usopen2025_F_predictions_complete.csv
[OK] Saved dataset → ./data/match_dataset_post_usopen_2025.csv (rows: 12084)
[OK] Saved player profiles → ./reports/player_profiles_post_usopen_2025.csv and ./reports/player_profiles_latest.csv
            name last_match_date  current_elo    peak_elo   selo_Clay  selo_Grass   selo_Hard  wr_Clay  wr_Grass  wr_Hard  overall_wr  form10_wr  form5_wr  streak  avg_rest_days  matches_28d
   Jannik Sinner      2025-09-07  2084.608666 2106.582562 1671.841552 1666.048857 1977.390177 0.690141  0.717391 0.802575    0.784848     